In [ ]:
!pip -q install chromadb sentence-transformers pandas tqdm
!pip install --upgrade langchain langchain-community sentence-transformers chromadb tqdm
!pip install -U llama-index transformers accelerate sentence-transformers chromadb
!pip install rank_bm25
!pip install langchain-community
!pip install pytorch-tabnet

In [ ]:
!pip install tavily-python


In [ ]:
# CELL 1: FOUNDATION / GLOBAL CONFIG

import os
import re
import json
import math
import time
import random
import hashlib
import zipfile
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import requests
import chromadb
import joblib

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# Optional model imports
from pytorch_tabnet.tab_model import TabNetClassifier

try:
    import tensorflow as tf
except Exception:
    tf = None


# GLOBAL RANDOMNESS
SEED = 7
random.seed(SEED)
np.random.seed(SEED)



# OPENAI CONFIG

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # change if you want: gpt-4o, gpt-4.1, gpt-4
TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.0"))
MAX_OUTPUT_TOKENS = int(os.getenv("OPENAI_MAX_OUTPUT_TOKENS", "250"))

openai_api_key_value = os.getenv("OPENAI_API_KEY", "").strip()
client = OpenAI(api_key=openai_api_key_value)


# PATHS
CHROMA_ZIP_PATH = "data/vector_database_updated (1).zip"
CHROMA_DIR = "data/chroma_uja1"
PERSIST_DIR = CHROMA_DIR

PROTO_FILE = "data/collection_prototypes_e5 (1).json"

RISK_MODEL_ZIP = "data/Proposal_Algorithm12.zip"
RISK_SCALER_PATH = "data/scaler.pkl"

SUPPORT_MODEL_PATH = "data/autoencoder_model.h5"
SUPPORT_SCALER_PATH = "data/scaler gh.pkl"
SUPPORT_PPINFO_PATH = "data/preprocess_info.json"
SUPPORT_AECOLS_PATH = "data/ae_columns.json"


DB_PATH = "data/student_db_updated_no_training (1) (1) (1).db"


# RETRIEVAL CONFIG
EMBED_MODEL = "intfloat/multilingual-e5-large"
USE_E5_PREFIX = True

ALL_COLLECTIONS = [
    "regulations",
    "transfer_rules",
    "degree_plans",
    "electives",
    "student_helpers",
    "faculty_offices",
    "clubs",
    "specialization",
    "coop_replies",
    "academic_calendar",
    "coop_rules",
    "activities",
    "course_bundles",
]


VECTOR_COLLECTION_NAMES = [
    "students_v2",
    "activities",
    "course_bundles",
]

ROUTER = {
    "w_proto": 0.55,
    "w_evidence": 0.45,
    "peek_k": 4,
    "strong_score": 0.64,
    "strong_margin": 0.08,
    "weak_score": 0.52,
    "close_margin_global": 0.02,
    "max_candidates": 3,
}

RETR = {
    "dense_k_each": 35,
    "bm25_k_each": 40,
    "final_k": 80,
    "rrf_k": 60,
    "top_parents": 10,
    "expand_parts_default": 6,
    "max_context_chars": 14000,
    "max_total_parts": 60,
    "max_chunk_chars": 2600,
    "max_parts_per_parent": 6,
    "prefer_parent_diversity": True,
}

COL_OVERRIDES = {
    "transfer_rules": {"dense_k_each": 70, "expand_parts": 3},
    "regulations": {"dense_k_each": 55, "expand_parts": 6},
    "degree_plans": {"dense_k_each": 55, "expand_parts": 3},
    "student_helpers": {"dense_k_each": 45, "expand_parts": 2},
    "faculty_offices": {"dense_k_each": 45, "expand_parts": 2},
    "academic_calendar": {"dense_k_each": 45, "expand_parts": 2},
}


# WEB FALLBACK CONFIG
WEB_FALLBACK_ENABLED = True
ALLOWED_UJ_DOMAINS = ["uj.edu.sa"]
UJ_MAX_RESULTS = 5
FULL_WEB_MAX_RESULTS = 5


# MODEL THRESHOLDS
MIN_RISK_FEATURES = 3
MIN_SUPPORT_FEATURES = 4
DEFAULT_SUPPORT_THRESHOLD = 0.65


# SHARED LABELS
AR_FEATURE_NAMES = {
    "attendance": "الحضور",
    "participation": "المشاركة",
    "quizzes": "متوسط الكويزات",
    "assignments": "متوسط الواجبات",
    "midterm": "درجة الاختبار النصفي",
    "projects": "درجة المشاريع",

    "Attendance (%)": "الحضور",
    "Participation_Score": "المشاركة",
    "Quizzes_Avg": "متوسط الكويزات",
    "Assignments_Avg": "متوسط الواجبات",
    "Midterm_Score": "درجة الاختبار النصفي",
    "Projects_Score": "درجة المشاريع",

    "Study Hours Per Week": "ساعات المذاكرة الأسبوعية",
    "Social Media Usage (Hours per day)": "استخدام وسائل التواصل يوميًا",
    "Sleep Duration (Hours per night)": "ساعات النوم الليلية",
    "Physical Exercise (Hours per week)": "ساعات النشاط البدني أسبوعيًا",
    "Family Support  ": "الدعم الأسري",
    "Financial Stress": "الضغط المالي",
    "Peer Pressure": "ضغط الأقران",
    "Mental Stress Level": "مستوى الضغط النفسي",
    "Diet Quality": "جودة التغذية",
    "Cognitive Distortions": "التشوهات المعرفية",
}


# SHARED HELPERS
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    t = str(s)
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    t = t.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
    t = t.replace("٪", "%")
    t = re.sub(r"\s+", " ", t).strip()
    return t

def to_ar_feature_name(name: str) -> str:
    return AR_FEATURE_NAMES.get(name, name)

def to_ar_feature_dict(features: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not features:
        return {}
    return {to_ar_feature_name(k): v for k, v in features.items()}

def to_ar_feature_list(feature_names: Optional[List[str]]) -> List[str]:
    if not feature_names:
        return []
    return [to_ar_feature_name(x) for x in feature_names]

def safe_round(x, ndigits: int = 4):
    try:
        if x is None:
            return None
        return round(float(x), ndigits)
    except Exception:
        return x

def safe_block_fingerprint(text: str) -> str:
    return hashlib.sha1(normalize_text(text).encode("utf-8")).hexdigest()

def cap_chars(text: str, max_chars: int) -> str:
    t = text or ""
    if len(t) <= max_chars:
        return t
    return t[:max_chars].rstrip() + " …"

def make_tool_response(
    *,
    ok: bool,
    tool_name: str,
    task_type: str,
    summary_ar: str,
    structured_output: Optional[Dict[str, Any]] = None,
    features_used: Optional[Dict[str, Any]] = None,
    missing_features: Optional[List[str]] = None,
    warnings: Optional[List[str]] = None,
    error: Optional[str] = None,
    model_type: Optional[str] = None,
    provenance: Optional[Dict[str, Any]] = None,
    raw_output: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    return {
        "ok": ok,
        "tool_name": tool_name,
        "task_type": task_type,
        "summary_ar": summary_ar,
        "structured_output": structured_output or {},
        "features_used": to_ar_feature_dict(features_used),
        "missing_features": to_ar_feature_list(missing_features),
        "warnings": warnings or [],
        "error": error,
        "model_type": model_type,
        "provenance": provenance or {},
        "raw_output": raw_output,
    }


# =========================
# INIT CHROMA + EMBEDDER
# =========================
if not os.path.exists(CHROMA_DIR):
    if not os.path.exists(CHROMA_ZIP_PATH):
        raise FileNotFoundError(f"ملف قاعدة Chroma غير موجود: {CHROMA_ZIP_PATH}")
    with zipfile.ZipFile(CHROMA_ZIP_PATH, "r") as z:
        z.extractall(CHROMA_DIR)

chroma = chromadb.PersistentClient(path=PERSIST_DIR)
embedder = SentenceTransformer(EMBED_MODEL)



def extract_chroma_zip(
    zip_path: str = "chroma_students_db.zip",
    extract_to: str = "./chroma_students_db",
) -> str:
    if os.path.exists(extract_to):
        return extract_to

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Chroma zip not found: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)

    return extract_to

#######################################
## basic_autocorrect+ ai_autocorrect ##
#######################################
COMMON_CORRECTIONS = {
    "خطه": "خطة",
    "الماده": "المادة",
    "ماده": "مادة",
    "اصطناعى": "اصطناعي",
    "الاصطناعى": "الاصطناعي",
    "كمم": "كم",
    "متبقى": "متبقي",
    "المتطلبات السابقه": "المتطلبات السابقة",
}
def basic_autocorrect(text: str) -> str:
    text = normalize_text(text)

    for wrong, correct in COMMON_CORRECTIONS.items():
        text = text.replace(wrong, correct)

    text = re.sub(r"([\u0600-\u06FF])\1{2,}", r"\1", text)
    return text


def remove_repeated_phrases(text: str, max_phrase_len: int = 6) -> str:
    words = text.split()

    for n in range(max_phrase_len, 0, -1):
        i = 0
        new_words = []

        while i < len(words):
            phrase = words[i:i+n]
            next_phrase = words[i+n:i+2*n]

            if phrase and phrase == next_phrase:
                new_words.extend(phrase)
                i += 2 * n
            else:
                new_words.append(words[i])
                i += 1

        words = new_words

    return " ".join(words)


def ai_autocorrect(question: str) -> str:
    question = normalize_text(question)

    prompt = f"""
صحح الأخطاء الإملائية والكتابية فقط بدون تغيير المعنى.

قواعد صارمة:
- لا تغيّر معنى السؤال.
- لا تستبدل كلمة بكلمة من مقرر.
- لا تضف معلومات.
- لا تحذف كلمات مهمة مثل: كم، عدد، ساعات، متطلبات، خطة.
- لا تجب على السؤال.
- أرجع السؤال المصحح فقط.

السؤال:
{question}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "أنت مصحح إملائي عربي. صحح النص فقط بدون تغيير المعنى."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=80,
        )

        corrected = resp.choices[0].message.content.strip()
        return normalize_text(corrected)

    except Exception:
        return question


print("✅ Cell 1 loaded successfully.")
print("Collections:", [c.name for c in chroma.list_collections()])


print("✅ Cell 1 loaded successfully.")
print("Collections:", [c.name for c in chroma.list_collections()])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Cell 1 loaded successfully.
Collections: ['transfer_rules', 'degree_plans', 'electives', 'coop_rules', 'activities', 'regulations', 'academic_calendar', 'coop_replies', 'clubs', 'student_helpers', 'faculty_offices', 'specialization', 'course_bundles']
✅ Cell 1 loaded successfully.
Collections: ['transfer_rules', 'degree_plans', 'electives', 'coop_rules', 'activities', 'regulations', 'academic_calendar', 'coop_replies', 'clubs', 'student_helpers', 'faculty_offices', 'specialization', 'course_bundles']


In [ ]:
# CELL 2: RETRIEVAL LAYER

# DATA CONTRACTS
@dataclass
class RouteResult:
    mode: str
    candidates: List[str]
    scores: List[Tuple[str, float]]
    meta: Dict[str, Any]


@dataclass
class RetrievalConfidence:
    confidence: float
    answerable: bool
    fallback_recommended: bool
    reason: str
    features: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RetrievalResult:
    query: str
    mode: str
    variant: str
    structured_score: float
    structured: bool

    route_mode: str
    route_candidates: List[str]
    route_meta: Dict[str, Any]
    route_top_scores: List[Tuple[str, float]]

    chunk_hits: List[Dict[str, Any]]
    parents: List[Dict[str, Any]]

    context_blocks: List[Dict[str, Any]]
    context_text: str
    sources: List[str]
    context_debug: Dict[str, Any]

    confidence: RetrievalConfidence

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# TOKENIZATION
_AR_PUNCT = re.compile(r"[^\w\s\u0600-\u06FF%/.-]+", re.UNICODE)

def ar_tokenize(text: str) -> List[str]:
    t = normalize_text(text)
    t = _AR_PUNCT.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return [w for w in t.split(" ") if w]


# EMBEDDING
def embed_query(text: str) -> np.ndarray:
    q = normalize_text(text)
    if USE_E5_PREFIX and not q.startswith("query:"):
        q = "query: " + q
    v = embedder.encode([q], normalize_embeddings=True)[0]
    return np.asarray(v, dtype=np.float32)


def assert_collections_present(expected: List[str]):
    existing = set(c.name for c in chroma.list_collections())
    missing = [c for c in expected if c not in existing]
    if missing:
        raise RuntimeError(f"Missing collections: {missing}. Existing: {sorted(existing)}")


# STRUCTURED QUERY SCORING
def structured_query_score(q: str) -> float:
    qn = normalize_text(q)
    score = 0.0

    if re.search(r"\bBS-[A-Z0-9-]{2,}\b", qn):
        score += 0.40
    if re.search(r"\b[A-Z]{2,6}\s?\d{2,4}\b", qn):
        score += 0.35
    if re.search(r"\b\d{4}-\d{2}-\d{2}\b", qn):
        score += 0.25
    if re.search(r"\u0627\u0644\u0645\u0627\u062F\u0629\s*[:\uFF1A]?\s*\d+(\.\d+)?", qn):
        score += 0.40
    if "%" in qn:
        score += 0.15

    nums = re.findall(r"\d+", qn)
    if len(nums) >= 2:
        score += 0.15

    return float(min(score, 1.0))


def is_structured_query(q: str, threshold: float = 0.45) -> bool:
    return structured_query_score(q) >= threshold


# DENSE RETRIEVAL

def dense_topk(collection: str, qvec: np.ndarray, k: int) -> List[dict]:
    col = chroma.get_collection(collection)
    res = col.query(
        query_embeddings=[qvec.tolist()],
        n_results=int(k),
        include=["documents", "metadatas", "distances"]
    )

    out = []
    ids = res["ids"][0]
    dists = res["distances"][0]
    metas = res["metadatas"][0]
    docs = res["documents"][0]

    for _id, dist, md, doc in zip(ids, dists, metas, docs):
        md = md or {}
        out.append({
            "collection": collection,
            "id": _id,
            "distance": float(dist),
            "sim": float(max(0.0, 1.0 - float(dist))),
            "parent_id": md.get("parent_id"),
            "source_ref": md.get("source_ref"),
            "chunk_idx": md.get("chunk_idx"),
            "n_parts": md.get("n_parts"),
            "doc_type": md.get("doc_type"),
            "text": doc or "",
        })

    out.sort(key=lambda x: x["distance"])
    return out


# PROTOTYPE ROUTER
def build_or_load_prototypes() -> Dict[str, List[float]]:
    if not os.path.exists(PROTO_FILE):
        raise FileNotFoundError(f"Prototype file not found: {PROTO_FILE}")
    with open(PROTO_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))


def proto_scores(qvec: np.ndarray, protos: Dict[str, List[float]]) -> Dict[str, float]:
    scores = {}
    for cname in ALL_COLLECTIONS:
        p = np.asarray(protos[cname], dtype=np.float32)
        scores[cname] = 0.0 if np.allclose(p, 0.0) else cosine_sim(qvec, p)
    return scores


def evidence_scores(qvec: np.ndarray, peek_k: int) -> Dict[str, float]:
    scores = {}
    for cname in ALL_COLLECTIONS:
        hits = dense_topk(cname, qvec, k=peek_k)
        if not hits:
            scores[cname] = 0.0
            continue

        d1 = float(hits[0]["distance"])
        s = float(max(0.0, min(1.0, 1.0 - d1)))

        parent_counts = Counter((h.get("parent_id") or "NO_PARENT") for h in hits)
        best_parent_hits = max(parent_counts.values()) if parent_counts else 1
        s = min(1.0, s + 0.03 * max(0, best_parent_hits - 1))
        scores[cname] = s

    return scores


def route_query(query: str, protos: Dict[str, List[float]]) -> RouteResult:
    qvec = embed_query(query)
    ps = proto_scores(qvec, protos)
    es = evidence_scores(qvec, peek_k=ROUTER["peek_k"])

    comb = {
        c: (ROUTER["w_proto"] * ps[c]) + (ROUTER["w_evidence"] * es[c])
        for c in ALL_COLLECTIONS
    }

    ranked = sorted(comb.items(), key=lambda x: (-x[1], x[0]))
    (c1, s1) = ranked[0]
    (c2, s2) = ranked[1] if len(ranked) > 1 else (None, 0.0)
    margin = float(s1 - s2)

    strong = (s1 >= ROUTER["strong_score"]) and (margin >= ROUTER["strong_margin"])
    weak = (s1 < ROUTER["weak_score"])

    if weak or margin < ROUTER["close_margin_global"]:
        return RouteResult(
            mode="GLOBAL",
            candidates=[],
            scores=ranked[:10],
            meta={"s1": float(s1), "s2": float(s2), "margin": margin, "reason": "uncertain_or_weak"}
        )

    if strong:
        return RouteResult(
            mode="ROUTED",
            candidates=[c1],
            scores=ranked[:10],
            meta={"s1": float(s1), "s2": float(s2), "margin": margin, "reason": "strong_top1"}
        )

    cands = [c1]
    if c2:
        cands.append(c2)
    if len(ranked) >= 3 and margin < 0.06:
        cands.append(ranked[2][0])
    cands = cands[:ROUTER["max_candidates"]]

    return RouteResult(
        mode="ROUTED",
        candidates=cands,
        scores=ranked[:10],
        meta={"s1": float(s1), "s2": float(s2), "margin": margin, "reason": "medium_topk"}
    )


# BM25
# =========================
_BM25_CACHE: Dict[str, Tuple[Optional[BM25Okapi], List[dict]]] = {}

def bm25_build(collection: str):
    if collection in _BM25_CACHE:
        return _BM25_CACHE[collection]

    col = chroma.get_collection(collection)
    res = col.get(include=["documents", "metadatas"])

    ids = res["ids"]
    docs = res["documents"]
    metas = res["metadatas"]

    corpus, meta = [], []
    for _id, doc, md in zip(ids, docs, metas):
        doc = doc or ""
        toks = ar_tokenize(doc)
        if not toks:
            continue

        corpus.append(toks)
        md = md or {}
        meta.append({
            "collection": collection,
            "id": _id,
            "parent_id": md.get("parent_id"),
            "source_ref": md.get("source_ref"),
            "chunk_idx": md.get("chunk_idx"),
            "doc_type": md.get("doc_type"),
            "text": doc,
        })

    bm25 = BM25Okapi(corpus) if corpus else None
    _BM25_CACHE[collection] = (bm25, meta)
    return bm25, meta


def bm25_topk(collection: str, query: str, k: int) -> List[dict]:
    bm25, meta = bm25_build(collection)
    if bm25 is None or not meta:
        return []

    qtok = ar_tokenize(query)
    scores = bm25.get_scores(qtok)
    idx = np.argsort(-scores)[:int(k)]

    out = []
    for j in idx:
        m = meta[int(j)]
        out.append({**m, "bm25": float(scores[int(j)])})

    return out


# =========================
# RRF
# =========================
def rrf_fuse(dense_hits: List[dict], bm25_hits: List[dict], final_k: int) -> List[dict]:
    rrf_k = RETR["rrf_k"]
    score = defaultdict(float)
    item = {}

    d_sorted = sorted(dense_hits, key=lambda x: x.get("distance", 9.9))
    for rank, h in enumerate(d_sorted, start=1):
        key = (h["collection"], h["id"])
        score[key] += 1.0 / (rrf_k + rank)
        item[key] = h

    b_sorted = sorted(bm25_hits, key=lambda x: -x.get("bm25", -1e9))
    for rank, h in enumerate(b_sorted, start=1):
        key = (h["collection"], h["id"])
        score[key] += 1.0 / (rrf_k + rank)
        item.setdefault(key, h)

    fused = sorted(score.items(), key=lambda x: -x[1])[:int(final_k)]
    out = []
    for (col, _id), sc in fused:
        h = dict(item[(col, _id)])
        h["rrf_score"] = float(sc)
        if h.get("distance") is None:
            h["distance"] = 0.999
        out.append(h)

    return out


# =========================
# PARENT AGGREGATION
# =========================
def aggregate_parents(chunk_hits: List[dict], top_parents: int) -> List[dict]:
    by = defaultdict(list)

    for h in chunk_hits:
        pid = h.get("parent_id") or "NO_PARENT"
        gpid = f"{h['collection']}::{pid}"
        by[gpid].append(h)

    parents = []
    for gpid, lst in by.items():
        lst = sorted(lst, key=lambda x: x.get("distance", 9.9))
        best_dist = float(lst[0].get("distance", 9.9))
        support = float(sum((1.0 - float(x.get("distance", 0.999))) for x in lst))

        parents.append({
            "gparent_id": gpid,
            "collection": lst[0]["collection"],
            "parent_id": lst[0].get("parent_id"),
            "best_distance": best_dist,
            "support_sum_sim": support,
            "n_hits": int(len(lst)),
            "top_source_refs": [x.get("source_ref") for x in lst[:5]],
        })

    parents.sort(key=lambda p: (p["best_distance"], -p["support_sum_sim"], p["gparent_id"]))
    for i, p in enumerate(parents, start=1):
        p["rank"] = i

    return parents[:int(top_parents)]


def expand_parent(collection: str, parent_id: str, max_parts: int) -> List[dict]:
    col = chroma.get_collection(collection)
    res = col.get(where={"parent_id": parent_id}, include=["documents", "metadatas"])
    rows = list(zip(res["ids"], res["documents"], res["metadatas"]))

    def key_fn(row):
        _id, doc, md = row
        md = md or {}
        return int(md.get("chunk_idx", 0))

    rows.sort(key=key_fn)
    if max_parts is not None:
        rows = rows[:int(max_parts)]

    out = []
    for _id, doc, md in rows:
        md = md or {}
        out.append({
            "collection": collection,
            "id": _id,
            "parent_id": md.get("parent_id"),
            "source_ref": md.get("source_ref"),
            "chunk_idx": md.get("chunk_idx"),
            "doc_type": md.get("doc_type"),
            "text": doc or "",
        })

    return out


# =========================
# CONTEXT BLOCKS
# =========================
def build_context_blocks_from_parents(
    parents: List[dict],
    max_context_chars: int,
    max_total_parts: int,
    max_chunk_chars: int,
    max_parts_per_parent: int,
    prefer_parent_diversity: bool = True
) -> Dict[str, Any]:

    context_blocks = []
    sources = []
    included_parts = []
    total_chars = 0
    total_parts = 0

    seen_block_hash = set()
    dropped = []

    expanded_by_parent = []
    for p in parents:
        col = p["collection"]
        pid = p["parent_id"]
        want = int(COL_OVERRIDES.get(col, {}).get("expand_parts", RETR["expand_parts_default"]))
        want = min(want, int(max_parts_per_parent))
        parts = expand_parent(col, pid, max_parts=want)
        expanded_by_parent.append(parts)

    parent_indices = list(range(len(parents)))
    part_cursor = [0] * len(parents)

    def take_next():
        if not parents:
            return None

        if not prefer_parent_diversity:
            for i in parent_indices:
                if part_cursor[i] < len(expanded_by_parent[i]):
                    j = part_cursor[i]
                    part_cursor[i] += 1
                    return i, expanded_by_parent[i][j]
            return None

        for _ in range(len(parent_indices) * 2):
            i = parent_indices.pop(0)
            parent_indices.append(i)
            if part_cursor[i] < len(expanded_by_parent[i]):
                j = part_cursor[i]
                part_cursor[i] += 1
                return i, expanded_by_parent[i][j]

        return None

    while total_parts < max_total_parts and total_chars < max_context_chars:
        nxt = take_next()
        if nxt is None:
            break

        p_i, part = nxt
        col = part.get("collection") or parents[p_i]["collection"]
        pid = part.get("parent_id") or parents[p_i]["parent_id"]

        raw = (part.get("text") or "").strip()
        if not raw:
            dropped.append({"reason": "empty_text", "collection": col, "parent_id": pid})
            continue

        raw = normalize_text(raw)
        raw = cap_chars(raw, max_chunk_chars)

        sid = part.get("source_ref") or f"{col}:{part.get('id')}"
        block_text = f"[{sid}] (collection={col})\n{raw}\n"

        fp = safe_block_fingerprint(block_text)
        if fp in seen_block_hash:
            dropped.append({"reason": "dedup", "source_ref": sid})
            continue
        seen_block_hash.add(fp)

        if total_chars + len(block_text) > max_context_chars:
            dropped.append({"reason": "max_context_chars_reached", "source_ref": sid})
            break

        context_blocks.append({
            "text": block_text,
            "source_ref": sid,
            "collection": col,
            "parent_id": pid,
            "chunk_idx": part.get("chunk_idx"),
            "doc_type": part.get("doc_type"),
        })

        included_parts.append(part)
        sources.append(sid)
        total_chars += len(block_text)
        total_parts += 1

    dedup_sources = []
    seen = set()
    for s in sources:
        if s in seen:
            continue
        seen.add(s)
        dedup_sources.append(s)

    context_text = "\n".join([b["text"] for b in context_blocks]).strip()

    return {
        "context_blocks": context_blocks,
        "context_text": context_text,
        "sources": dedup_sources,
        "included_parts": included_parts,
        "debug": {
            "total_parts": total_parts,
            "total_chars": total_chars,
            "dropped_count": len(dropped),
            "first_drop": dropped[0] if dropped else None,
        }
    }


# =========================
# PLANNER-AWARE RETRIEVAL HELPERS
# =========================

def _asdict_safe(x):
    try:
        if hasattr(x, "__dataclass_fields__"):
            return asdict(x)
        if isinstance(x, dict):
            return x
    except Exception:
        pass
    return {}


def _normalize_planner_hints(planner_hints=None, route_decision=None) -> Dict[str, Any]:
    """
    Accepts:
    - planner_hints dict
    - RouteDecision object from Cell 4
    - None

    Returns normalized planner config for retrieval.
    """
    d = {}

    if planner_hints is not None:
        d.update(_asdict_safe(planner_hints))

    if route_decision is not None:
        d.update(_asdict_safe(route_decision))

    valid_cols = set(ALL_COLLECTIONS)

    def clean_cols(xs):
        out, seen = [], set()
        for x in xs or []:
            x = str(x).strip()
            if x in valid_cols and x not in seen:
                seen.add(x)
                out.append(x)
        return out

    initial = clean_cols(
        d.get("initial_collections")
        or d.get("rag_collections")
        or d.get("route_candidates")
        or []
    )

    must = clean_cols(d.get("must_search_collections") or [])

    strategy = str(d.get("retrieval_strategy", "") or "").strip().lower()
    if strategy not in ["single", "multi", "global"]:
        strategy = "multi" if len(initial) >= 2 else ("single" if len(initial) == 1 else "global")

    return {
        "retrieval_strategy": strategy,
        "initial_collections": initial,
        "must_search_collections": must,
        "expand_if_weak": bool(d.get("expand_if_weak", True)),
        "allow_global_fallback": bool(d.get("allow_global_fallback", True)),
        "collection_reasoning": str(d.get("collection_reasoning", "")),
        "raw": d,
    }


def _select_search_collections(query: str, protos, mode: str, planner: Dict[str, Any]):
    """
    New selection logic:
    1) If forced global -> all collections
    2) If Cell 4 planner gives collections -> use them
    3) If planner says global -> all collections
    4) Else use old route_query
    """
    qvec = embed_query(query)

    if mode == "global":
        route = RouteResult(
            mode="GLOBAL",
            candidates=[],
            scores=[],
            meta={"reason": "forced_global"}
        )
        return ALL_COLLECTIONS[:], route

    strategy = planner.get("retrieval_strategy", "multi")
    initial = planner.get("initial_collections", []) or []
    must = planner.get("must_search_collections", []) or []

    if strategy == "global":
        route = RouteResult(
            mode="PLANNER_GLOBAL",
            candidates=ALL_COLLECTIONS[:],
            scores=[],
            meta={
                "reason": "planner_requested_global",
                "planner_strategy": strategy,
                "planner_initial": initial,
                "planner_must": must,
            }
        )
        return ALL_COLLECTIONS[:], route

    if initial:
        search_cols = []
        for c in must + initial:
            if c in ALL_COLLECTIONS and c not in search_cols:
                search_cols.append(c)

        # Paper-style safer behavior:
        # multi means do not restrict too aggressively
        if strategy == "multi" and len(search_cols) < 3:
            old_route = route_query(query, protos)
            for c, _ in old_route.scores[:3]:
                if c in ALL_COLLECTIONS and c not in search_cols:
                    search_cols.append(c)

        route = RouteResult(
            mode="PLANNER_ROUTED",
            candidates=search_cols,
            scores=[],
            meta={
                "reason": "planner_collections_used",
                "planner_strategy": strategy,
                "planner_initial": initial,
                "planner_must": must,
                "collection_reasoning": planner.get("collection_reasoning", ""),
            }
        )
        return search_cols, route

    # Old behavior fallback
    route = route_query(query, protos)
    search_cols = ALL_COLLECTIONS[:] if route.mode == "GLOBAL" else route.candidates[:]
    return search_cols, route


def _retrieval_looks_weak(chunk_hits: List[dict], parents: List[dict], context_text: str) -> bool:
    """
    Lightweight non-LLM confidence check.
    If weak, retrieve_final can expand to global.
    """
    if not chunk_hits:
        return True

    if not parents:
        return True

    if not (context_text or "").strip():
        return True

    best_dist = None
    try:
        best_dist = min(float(h.get("distance", 999.0)) for h in chunk_hits)
    except Exception:
        best_dist = None

    # Chroma distance depends on metric, but in your code sim = 1 - distance.
    # distance >= 0.55 usually means weak semantic hit for normalized embeddings.
    if best_dist is not None and best_dist >= 0.55:
        return True

    if len(context_text) < 500:
        return True

    return False


# =========================
# MAIN API
# =========================

def retrieve_final(
    query: str,
    protos: Dict[str, List[float]],
    mode: str = "router",
    use_bm25: bool = True,
    planner_hints: Optional[Dict[str, Any]] = None,
    route_decision: Optional[Any] = None,
) -> dict:
    """
    Backward compatible:
      retrieve_final(query, protos)

    New planner-aware usage:
      decision = route_question(query)
      retrieve_final(query, protos, route_decision=decision)

    Supports multi-collection agentic RAG routing.
    """
    qvec = embed_query(query)
    structured = is_structured_query(query)
    do_bm25 = bool(use_bm25 and structured)

    planner = _normalize_planner_hints(
        planner_hints=planner_hints,
        route_decision=route_decision,
    )

    search_cols, route = _select_search_collections(
        query=query,
        protos=protos,
        mode=mode,
        planner=planner,
    )

    dense_hits = []
    bm25_hits = []

    for cname in search_cols:
        k_each = int(COL_OVERRIDES.get(cname, {}).get("dense_k_each", RETR["dense_k_each"]))
        dense_hits.extend(dense_topk(cname, qvec, k=k_each))

        if do_bm25:
            kbm = int(COL_OVERRIDES.get(cname, {}).get("bm25_k_each", RETR["bm25_k_each"]))
            bm25_hits.extend(bm25_topk(cname, query, k=kbm))

    final_k = int(RETR["final_k"])

    if do_bm25:
        chunk_hits = rrf_fuse(dense_hits, bm25_hits, final_k=final_k)
        variant = (
            "planner_hybrid_rrf"
            if route.mode.startswith("PLANNER")
            else (
                "hybrid_rrf_global_forced"
                if mode == "global"
                else ("hybrid_rrf" if route.mode == "GLOBAL" else "routed_hybrid_rrf")
            )
        )
    else:
        chunk_hits = sorted(dense_hits, key=lambda x: x["distance"])[:final_k]
        variant = (
            "planner_dense"
            if route.mode.startswith("PLANNER")
            else (
                "dense_global_forced"
                if mode == "global"
                else ("dense_global" if route.mode == "GLOBAL" else "routed_dense")
            )
        )

    parents = aggregate_parents(chunk_hits, top_parents=int(RETR["top_parents"]))

    ctx = build_context_blocks_from_parents(
        parents=parents,
        max_context_chars=int(RETR["max_context_chars"]),
        max_total_parts=int(RETR["max_total_parts"]),
        max_chunk_chars=int(RETR["max_chunk_chars"]),
        max_parts_per_parent=int(RETR["max_parts_per_parent"]),
        prefer_parent_diversity=bool(RETR["prefer_parent_diversity"]),
    )

    expanded_to_global = False

    # Agentic RAG behavior:
    # If planner retrieval is weak, expand to global and rebuild context.
    if (
        planner.get("expand_if_weak", True)
        and planner.get("allow_global_fallback", True)
        and route.mode != "GLOBAL"
        and route.mode != "PLANNER_GLOBAL"
        and len(search_cols) < len(ALL_COLLECTIONS)
        and _retrieval_looks_weak(chunk_hits, parents, ctx["context_text"])
    ):
        expanded_to_global = True

        global_dense_hits = []
        global_bm25_hits = []

        for cname in ALL_COLLECTIONS:
            k_each = int(COL_OVERRIDES.get(cname, {}).get("dense_k_each", RETR["dense_k_each"]))
            global_dense_hits.extend(dense_topk(cname, qvec, k=k_each))

            if do_bm25:
                kbm = int(COL_OVERRIDES.get(cname, {}).get("bm25_k_each", RETR["bm25_k_each"]))
                global_bm25_hits.extend(bm25_topk(cname, query, k=kbm))

        if do_bm25:
            chunk_hits = rrf_fuse(global_dense_hits, global_bm25_hits, final_k=final_k)
            variant = variant + "_expanded_global"
        else:
            chunk_hits = sorted(global_dense_hits, key=lambda x: x["distance"])[:final_k]
            variant = variant + "_expanded_global"

        parents = aggregate_parents(chunk_hits, top_parents=int(RETR["top_parents"]))

        ctx = build_context_blocks_from_parents(
            parents=parents,
            max_context_chars=int(RETR["max_context_chars"]),
            max_total_parts=int(RETR["max_total_parts"]),
            max_chunk_chars=int(RETR["max_chunk_chars"]),
            max_parts_per_parent=int(RETR["max_parts_per_parent"]),
            prefer_parent_diversity=bool(RETR["prefer_parent_diversity"]),
        )

        search_cols = ALL_COLLECTIONS[:]

    return {
        "query": query,
        "mode": mode,
        "variant": variant,

        "structured_score": structured_query_score(query),
        "structured": structured,

        "route_mode": route.mode,
        "route_candidates": search_cols,
        "route_meta": {
            **(route.meta or {}),
            "planner": planner,
            "expanded_to_global": expanded_to_global,
        },
        "route_top_scores": route.scores,

        "chunk_hits": chunk_hits,
        "parents": parents,

        "context_blocks": ctx["context_blocks"],
        "context_text": ctx["context_text"],
        "sources": ctx["sources"],
        "context_debug": ctx["debug"],
    }


protos = build_or_load_prototypes()

print("✅ Cell 2 loaded successfully with planner-aware multi-collection retrieval.")

✅ Cell 2 loaded successfully with planner-aware multi-collection retrieval.


In [ ]:
# ============================================================
# CELL 3: FAST RAG ANSWER LAYER - COMPATIBLE WITH CELL 3B
# ============================================================

import os
import re
import requests
from typing import List, Dict, Any, Optional, Tuple


# ============================================================
# SYSTEM PROMPT
# ============================================================

RAG_ANSWER_SYSTEM_PROMPT = """
أنت مساعد أكاديمي ذكي لجامعة جدة.

مهمتك هي تقديم أفضل إجابة ممكنة للمستخدم اعتمادًا على الأدلة المتاحة لك من:
1. قاعدة المعرفة الداخلية.
2. موقع جامعة جدة.
3. الويب العام عند الحاجة.

القواعد:
- ابدأ دائمًا بقاعدة المعرفة الداخلية.
- إذا لم تكن قاعدة المعرفة كافية، استخدم البحث العميق.
- إذا لم يكن البحث العميق كافيًا، استخدم موقع جامعة جدة.
- إذا لم يكن موقع جامعة جدة كافيًا، استخدم الويب العام.
- لا تقل للمستخدم: "لا أعرف"، أو "لا توجد معلومات كافية"، أو "لا أستطيع الإجابة" إذا كانت هناك أي معلومة مفيدة في المصادر.
- إذا كانت المعلومات جزئية، قدّم أفضل إجابة ممكنة بصيغة حذرة مثل: "بحسب المعلومات المتاحة..."
- إذا كان سؤال المستخدم غامضًا جدًا ويحتمل أكثر من معنى، اسأل سؤالًا توضيحيًا ذكيًا ومختصرًا.
- لا تسأل سؤالًا توضيحيًا إذا كان بإمكانك تقديم إجابة مفيدة من الأدلة المتاحة.
- لا تخترع أرقامًا أو مواعيد أو شروطًا غير موجودة في المصادر.
- لا تذكر تفاصيل تقنية مثل RAG أو KB أو Tavily أو router للمستخدم.
- اكتب بالعربية الفصحى الواضحة.
""".strip()


REPHRASE_PROMPT = """
بصفتك مساعدًا أكاديميًا لجامعة جدة، أعد صياغة سؤال المستخدم ليصبح سؤالًا مستقلاً وواضحًا ومفيدًا لاسترجاع المعلومات.

سجل المحادثة:
{chat_history}

السؤال الأصلي:
{question}

السؤال المعاد صياغته:
""".strip()


# ============================================================
# OPTIONAL WEB SETTINGS
# ============================================================

WEB_FALLBACK_ENABLED = True

UJ_MAX_RESULTS = globals().get("UJ_MAX_RESULTS", 3)
FULL_WEB_MAX_RESULTS = globals().get("FULL_WEB_MAX_RESULTS", 3)

ALLOWED_UJ_DOMAINS = globals().get(
    "ALLOWED_UJ_DOMAINS",
    ["uj.edu.sa", "www.uj.edu.sa"]
)

# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_KEY"
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "").strip()


# ============================================================
# WEB SEARCH HELPERS
# ============================================================

def _host_is_allowed(url: str, allowed_domains: List[str]) -> bool:
    try:
        from urllib.parse import urlparse
        host = urlparse(url).netloc.lower()
    except Exception:
        return False

    for d in allowed_domains:
        d = d.lower()
        if host == d or host.endswith("." + d):
            return True

    return False


def tavily_search(query: str, max_results: int = 5) -> List[Dict[str, str]]:
    key = os.getenv("TAVILY_API_KEY")

    if not key or not WEB_FALLBACK_ENABLED:
        return []

    try:
        r = requests.post(
            "https://api.tavily.com/search",
            headers={
                "Content-Type": "application/json",
            },
            json={
                "api_key": key,
                "query": query,
                "search_depth": "advanced",
                "max_results": max_results,
                "include_answer": False,
                "include_raw_content": False,
            },
            timeout=20,
        )
        r.raise_for_status()
        data = r.json().get("results", []) or []
    except Exception as e:
        print(f"Tavily search error: {e}")
        return []

    out = []

    for it in data:
        url = it.get("url") or ""
        if not url:
            continue

        out.append({
            "title": it.get("title", ""),
            "url": url,
            "snippet": it.get("content", ""),
        })

        if len(out) >= max_results:
            break

    return out


def web_search_uj(query: str, max_results: int = UJ_MAX_RESULTS) -> List[Dict[str, str]]:
    scoped_query = f"site:uj.edu.sa {query}".strip()
    raw = tavily_search(scoped_query, max_results=max_results)

    out = []
    for it in raw:
        if _host_is_allowed(it["url"], ALLOWED_UJ_DOMAINS):
            out.append(it)

    return out[:max_results]


def web_search_full(query: str, max_results: int = FULL_WEB_MAX_RESULTS) -> List[Dict[str, str]]:
    return tavily_search(query, max_results=max_results)


def build_web_context(web_results: List[Dict[str, str]], prefix: str) -> Tuple[str, List[str]]:
    if not web_results:
        return "", []

    chunks = []
    sources = []

    for i, r in enumerate(web_results, 1):
        rid = f"{prefix}{i}"

        chunks.append(
            f"[{rid}] (web)\n"
            f"العنوان: {r.get('title', '')}\n"
            f"الرابط: {r.get('url', '')}\n"
            f"الملخص: {r.get('snippet', '')}"
        )

        sources.append(f"{rid} - {r.get('url', '')}")

    return "\n\n".join(chunks), sources


# ============================================================
# QUERY REWRITE - FAST
# ============================================================

def _should_rephrase(question: str, chat_history_text: str) -> bool:
    if not (chat_history_text or "").strip():
        return False

    q = normalize_text(question)

    if len(q.split()) <= 3:
        return True

    pronouns = [
        "هذا", "هذه", "ذلك", "تلك", "هنا", "هناك",
        "it", "this", "that", "they", "them"
    ]

    return any(p in q.lower() for p in pronouns)


def _rephrase_for_retrieval(question: str, chat_history_text: str) -> str:
    if not _should_rephrase(question, chat_history_text):
        return question

    try:
        prompt = REPHRASE_PROMPT.format(
            chat_history=chat_history_text,
            question=question,
        )

        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "أعد صياغة السؤال فقط بدون شرح."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=120,
        )

        rewritten = (resp.choices[0].message.content or "").strip()
        return rewritten if rewritten else question

    except Exception:
        return question


# ============================================================
# FAST CONTEXT JUDGE - NO LLM CALL
# ============================================================

def context_answers_question(question: str, context_text: str) -> bool:
    if not context_text or not context_text.strip():
        return False

    text = context_text.strip()

    # لا نعتبر أي مقطع قصير جدًا إجابة كافية تلقائيًا
    if len(text) < 120:
        return False

    no_answer_markers = [
        "لا توجد معلومات",
        "غير كافية",
        "لم يتم ذكر",
        "لا تحتوي",
        "لا يمكن تقديم",
        "لا أملك",
    ]

    if any(m in text for m in no_answer_markers):
        return False

    q = normalize_query_for_tools(question).lower()

    important_terms = []
    for token in re.split(r"\s+|،|,|\?|؟|\.", q):
        token = token.strip()
        if len(token) >= 4:
            important_terms.append(token)

    if not important_terms:
        return False

    hits = sum(1 for t in important_terms if t in text.lower())

    return hits >= 2


# ============================================================
# ROUTER AS HINT, NOT CONDITION
# ============================================================

def retrieve_with_router_as_hint(
    query: str,
    protos,
    mode: str = "router",
    route_decision=None,
    use_bm25: bool = True,
) -> Dict[str, Any]:
    """
    The router is used as a hint, not as a hard condition.

    Strategy:
    1. Try the router-selected / suggested collections first.
    2. If weak, search all collections.
    3. Return the better result.
    """

    # --------------------------------------------------------
    # STEP 1: Try router-guided retrieval first
    # --------------------------------------------------------
    routed_result = retrieve_final(
        query=query,
        protos=protos,
        mode=mode,
        use_bm25=use_bm25,
    )

    routed_context = routed_result.get("context_text", "") or ""
    routed_sources = routed_result.get("sources", []) or []
    routed_answered = context_answers_question(query, routed_context)

    routed_result["retrieval_scope"] = "router_hint"
    routed_result["router_hint_answered"] = routed_answered
    routed_result["router_was_condition"] = False

    if routed_answered:
        routed_result["expanded_to_all_collections"] = False
        return routed_result

    # --------------------------------------------------------
    # STEP 2: Router result is weak, so search all collections
    # --------------------------------------------------------
    try:
        all_result = retrieve_final(
            query=query,
            protos=protos,
            mode="all",
            use_bm25=use_bm25,
        )

        all_context = all_result.get("context_text", "") or ""
        all_sources = all_result.get("sources", []) or []
        all_answered = context_answers_question(query, all_context)

        all_result["retrieval_scope"] = "all_collections"
        all_result["router_hint_answered"] = routed_answered
        all_result["all_collections_answered"] = all_answered
        all_result["expanded_to_all_collections"] = True
        all_result["router_was_condition"] = False
        all_result["router_hint_result"] = {
            "context_text": routed_context,
            "sources": routed_sources,
        }

        # If all-collections search answers, use it.
        if all_answered:
            return all_result

        # If neither clearly answers, keep the result with more context.
        if len(all_context) > len(routed_context):
            return all_result

    except Exception as e:
        routed_result["all_collections_error"] = str(e)

    routed_result["expanded_to_all_collections"] = False
    return routed_result


# ============================================================
# SMART CLARIFICATION LAYER
# ============================================================

def should_ask_clarification(
    question: str,
    kb_context: str = "",
    uj_context: str = "",
    web_context: str = "",
) -> bool:
    """
    Ask clarification only when:
    - the question is short or ambiguous,
    - retrieval/web did not produce a strong answer,
    - and asking clarification would be better than giving a weak answer.
    """

    q = (question or "").strip()
    if not q:
        return True

    all_context = " ".join([
        kb_context or "",
        uj_context or "",
        web_context or "",
    ]).strip()

    # إذا كان لدينا سياق جيد، لا نسأل المستخدم
    if len(all_context) >= 500:
        return False

    words = q.split()

    very_short = len(words) <= 4

    vague_reference_words = [
        "هذا", "هذه", "ذلك", "تلك", "هنا", "هناك",
        "هو", "هي", "هم", "همه", "عنه", "عنها",
        "it", "this", "that", "they", "them",
    ]

    broad_academic_terms = [
        "القبول",
        "التسجيل",
        "التحويل",
        "المعادلة",
        "الحذف",
        "الإضافة",
        "الاعتذار",
        "التأجيل",
        "المكافأة",
        "الاختبار",
        "الخطة",
        "النسبة",
        "المعدل",
        "المقرر",
        "الشروط",
        "الموعد",
        "الرسوم",
        "الكلية",
        "التخصص",
        "البرنامج",
        "التدريب",
    ]

    vague_action_words = [
        "كيف",
        "متى",
        "وين",
        "أين",
        "كم",
        "طريقة",
        "ابغى",
        "أريد",
        "اقدر",
        "أقدر",
        "ممكن",
    ]

    has_vague_reference = any(w in q for w in vague_reference_words)
    has_broad_term = any(w in q for w in broad_academic_terms)
    has_vague_action = any(w in q for w in vague_action_words)

    # نسأل فقط إذا كان السؤال قصيرًا أو يعتمد على مرجع غير واضح
    # أو يجمع بين كلمة عامة وفعل عام بدون تفاصيل كافية
    if has_vague_reference:
        return True

    if very_short and (has_broad_term or has_vague_action):
        return True

    if has_broad_term and has_vague_action and len(words) <= 6:
        return True

    return False


def build_clarification_answer(question: str) -> str:
    prompt = f"""
السؤال الأصلي:
{question}

اكتب سؤالًا توضيحيًا واحدًا فقط للمستخدم باللغة العربية.

القواعد:
- لا تقل إنك لا تعرف.
- لا تقل إن المعلومات غير كافية.
- اطلب توضيحًا يساعدك على الإجابة بدقة.
- إذا كان السؤال يحتمل أكثر من معنى، اذكر احتمالين أو ثلاثة فقط.
- اجعل السؤال قصيرًا ومفيدًا وغير مزعج.
- لا تذكر تفاصيل تقنية.
""".strip()

    try:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "أنت مساعد أكاديمي ذكي. اسأل سؤالًا توضيحيًا مختصرًا فقط عندما يكون سؤال المستخدم غامضًا."
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
            max_tokens=120,
        )

        out = (resp.choices[0].message.content or "").strip()
        if out:
            return out

    except Exception:
        pass

    return "هل تقصد الشروط، الموعد، طريقة التقديم، أم حالة خاصة بك؟ حدّد المقصود لأجيبك بدقة."


# ============================================================
# FINAL LLM ANSWER
# ============================================================

NO_ANSWER_PHRASES = [
    "لا توجد معلومات كافية",
    "لا توجد مقاطع كافية",
    "لم أجد معلومات",
    "لا أملك معلومات",
    "لا يمكنني الإجابة",
    "المعلومات غير كافية",
    "لا توجد إجابة",
    "لا يوجد جواب",
    "لا أستطيع الإجابة",
    "لا أعرف",
]


def llm_rag_answer_from_contexts(
    question: str,
    kb_context: str = "",
    uj_context: str = "",
    web_context: str = "",
    sources_kb: Optional[List[str]] = None,
    sources_uj: Optional[List[str]] = None,
    sources_web: Optional[List[str]] = None,
) -> str:
    sources_kb = sources_kb or []
    sources_uj = sources_uj or []
    sources_web = sources_web or []

    user_prompt = f"""
سؤال المستخدم:
{question}

(1) مقاطع قاعدة المعرفة الداخلية:
{kb_context.strip() if kb_context.strip() else "لا توجد مقاطع من قاعدة المعرفة الداخلية."}

(2) مقاطع موقع جامعة جدة:
{uj_context.strip() if uj_context.strip() else "لا توجد مقاطع من موقع جامعة جدة."}

(3) مقاطع الويب العام:
{web_context.strip() if web_context.strip() else "لا توجد مقاطع من الويب العام."}

التعليمات:
- استخدم فقط المعلومات الموجودة في المقاطع المعروضة.
- اتبع ترتيب الأولوية: قاعدة المعرفة الداخلية ثم موقع جامعة جدة ثم الويب العام.
- إذا كانت قاعدة المعرفة الداخلية كافية فلا تعتمد على الويب.
- إذا كانت قاعدة المعرفة الداخلية غير كافية وكان موقع الجامعة يحتوي الجواب، فاعتمد موقع الجامعة.
- لا تستخدم الويب العام إلا عند الحاجة.
- إذا كانت المعلومات جزئية، قدّم أفضل إجابة ممكنة بصيغة حذرة.
- لا تقل "لا توجد معلومات كافية" إذا كانت هناك أي معلومة مفيدة في المقاطع.
- لا تقل "لا أعرف" أو "لا أستطيع الإجابة" إذا كانت هناك أي معلومة يمكن استخدامها.
- إذا وجدت أي معلومة مفيدة، استخدمها لبناء إجابة عملية للمستخدم.
- لا تخترع معلومات غير موجودة في المقاطع.
- اجعل الإجابة مباشرة وواضحة.
""".strip()

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": RAG_ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=MAX_OUTPUT_TOKENS,
    )

    out = (resp.choices[0].message.content or "").strip()

    # --------------------------------------------------------
    # Anti "no answer" retry:
    # If the model says it cannot answer while there is any context,
    # force one more attempt using the available evidence.
    # --------------------------------------------------------
    has_any_context = any([
        kb_context.strip(),
        uj_context.strip(),
        web_context.strip(),
    ])

    if has_any_context and any(p in out for p in NO_ANSWER_PHRASES):
        retry_prompt = f"""
السؤال:
{question}

المعلومات المتاحة من المصادر:
{kb_context.strip()}

{uj_context.strip()}

{web_context.strip()}

أعد كتابة الإجابة للمستخدم.

القواعد:
- ممنوع أن تقول إنك لا تعرف إذا كانت هناك أي معلومة مفيدة في النص.
- ممنوع أن تقول إن المعلومات غير كافية إذا كانت هناك أي معلومة يمكن استخدامها.
- قدّم أفضل إجابة ممكنة بناءً على المعلومات المتاحة فقط.
- إذا كانت المعلومات جزئية، استخدم عبارة حذرة مثل: "بحسب المعلومات المتاحة..."
- لا تخترع معلومة غير موجودة.
- اجعل الإجابة مباشرة وواضحة.
""".strip()

        try:
            retry = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": RAG_ANSWER_SYSTEM_PROMPT},
                    {"role": "user", "content": retry_prompt},
                ],
                temperature=0.1,
                max_tokens=MAX_OUTPUT_TOKENS,
            )

            retry_out = (retry.choices[0].message.content or "").strip()
            if retry_out:
                out = retry_out

        except Exception:
            pass

    return out


# ============================================================
# TRACE
# ============================================================

def make_retrieval_trace(
    retrieval_query: str,
    kb_answered: bool,
    uj_answered: bool,
    full_web_answered: bool,
    used_deep: bool,
) -> Dict[str, Any]:
    return {
        "retrieval_query": retrieval_query,
        "kb_answered": kb_answered,
        "uj_web_answered": uj_answered,
        "full_web_answered": full_web_answered,
        "used_deep_retrieval": used_deep,
        "source_priority": ["KB", "UJ Web", "Full Web"],
    }


# ============================================================
# MAIN ANSWER
# ============================================================

def answer(
    query: str,
    protos,
    mode: str = "router",
    return_debug: bool = False,
    use_deep: bool = True,
    use_web: bool = True,
    always_fallback: bool = True,
    allow_clarification: bool = True,
):
    try:
        chat_history_text = build_chat_history_from_session()
    except Exception:
        chat_history_text = ""

    retrieval_query = _rephrase_for_retrieval(query, chat_history_text)

    # --------------------------------------------------------
    # STEP 1: Fast KB retrieval
    # This uses BOTH:
    # - mode=mode first       => router suggested collections
    # - mode="all" if weak    => all collections fallback
    # --------------------------------------------------------
    retrieval_result = retrieve_with_router_as_hint(
        query=retrieval_query,
        protos=protos,
        mode=mode,
        route_decision=None,
        use_bm25=True,
    )

    context_kb = retrieval_result.get("context_text", "") or ""
    sources_kb = retrieval_result.get("sources", []) or []

    kb_answered = context_answers_question(query, context_kb)

    used_deep = False
    deep_result = None

    # --------------------------------------------------------
    # STEP 2: Automatic deep KB retrieval if fast KB is weak
    # IMPORTANT:
    # Deep search uses mode="all" because fast retrieval already tried
    # router suggestion first, then all collections. If still weak,
    # deep search should be global across all collections.
    # --------------------------------------------------------
    if always_fallback and use_deep and not kb_answered and "kb_agent_search_deep" in globals():
        try:
            kb_deep = kb_agent_search_deep(
                query=retrieval_query,
                protos=protos,
                mode="all",
                use_bm25=True,
                route_decision=None,
            )

            deep_context = kb_deep.context_text or ""
            deep_sources = kb_deep.sources or []

            deep_answered = context_answers_question(query, deep_context)

            deep_result = {
                "context_text": deep_context,
                "sources": deep_sources,
                "deep_meta": getattr(kb_deep, "meta", {}),
            }

            used_deep = True

            # استخدم نتيجة البحث العميق إذا كانت أفضل أو أطول أو البحث السريع كان ضعيفًا
            if deep_context and (
                deep_answered
                or len(deep_context) > len(context_kb)
                or not context_kb.strip()
            ):
                context_kb = deep_context
                sources_kb = deep_sources
                retrieval_result = deep_result
                kb_answered = deep_answered

        except Exception as e:
            deep_result = {
                "error": str(e),
                "context_text": "",
                "sources": [],
            }

    # --------------------------------------------------------
    # STEP 3: UJ Web fallback - automatic
    # --------------------------------------------------------
    context_uj = ""
    sources_uj = []
    uj_answered = False

    if always_fallback and use_web and not kb_answered:
        try:
            uj_results = web_search_uj(
                retrieval_query,
                max_results=max(UJ_MAX_RESULTS, 5),
            )
            context_uj, sources_uj = build_web_context(uj_results, prefix="UJ")
            uj_answered = context_answers_question(query, context_uj)
        except Exception as e:
            context_uj = ""
            sources_uj = [f"UJ web search error: {e}"]

    # --------------------------------------------------------
    # STEP 4: Full Web fallback - automatic
    # --------------------------------------------------------
    context_web = ""
    sources_web = []
    full_web_answered = False

    if always_fallback and use_web and not kb_answered and not uj_answered:
        try:
            web_results = web_search_full(
                retrieval_query,
                max_results=max(FULL_WEB_MAX_RESULTS, 5),
            )
            context_web, sources_web = build_web_context(web_results, prefix="WEB")
            full_web_answered = context_answers_question(query, context_web)
        except Exception as e:
            context_web = ""
            sources_web = [f"Full web search error: {e}"]

    # --------------------------------------------------------
    # STEP 4.5: Smart clarification instead of weak answer
    # --------------------------------------------------------
    no_strong_answer = not kb_answered and not uj_answered and not full_web_answered

    if allow_clarification and no_strong_answer:
        ask_clarification = should_ask_clarification(
            question=query,
            kb_context=context_kb,
            uj_context=context_uj,
            web_context=context_web,
        )

        if ask_clarification:
            clarification_answer = build_clarification_answer(query)

            trace = make_retrieval_trace(
                retrieval_query=retrieval_query,
                kb_answered=kb_answered,
                uj_answered=uj_answered,
                full_web_answered=full_web_answered,
                used_deep=used_deep,
            )

            result = {
                "answer": clarification_answer,
                "trace": trace,
                "sources": {
                    "kb": sources_kb,
                    "uj_web": sources_uj,
                    "full_web": sources_web,
                },
                "contexts": {
                    "kb": context_kb,
                    "uj_web": context_uj,
                    "full_web": context_web,
                },
                "retrieval": retrieval_result,
                "deep_retrieval": deep_result,
                "clarification_requested": True,
            }

            if return_debug:
                return result

            return clarification_answer

    # --------------------------------------------------------
    # STEP 5: Final answer - LLM call
    # --------------------------------------------------------
    final_answer = llm_rag_answer_from_contexts(
        question=query,
        kb_context=context_kb,
        uj_context=context_uj,
        web_context=context_web,
        sources_kb=sources_kb,
        sources_uj=sources_uj,
        sources_web=sources_web,
    )

    trace = make_retrieval_trace(
        retrieval_query=retrieval_query,
        kb_answered=kb_answered,
        uj_answered=uj_answered,
        full_web_answered=full_web_answered,
        used_deep=used_deep,
    )

    result = {
        "answer": final_answer,
        "trace": trace,
        "sources": {
            "kb": sources_kb,
            "uj_web": sources_uj,
            "full_web": sources_web,
        },
        "contexts": {
            "kb": context_kb,
            "uj_web": context_uj,
            "full_web": context_web,
        },
        "retrieval": retrieval_result,
        "deep_retrieval": deep_result,
        "clarification_requested": False,
    }

    if return_debug:
        return result

    return final_answer


print("✅ Fast Cell 3 loaded successfully with router hint, all-collections fallback, deep global search, web fallback, and clarification handling.")

✅ Fast Cell 3 loaded successfully with router hint, all-collections fallback, deep global search, web fallback, and clarification handling.


In [ ]:
rag_result = answer(
    "عدد اقسام كلية الحاسب واسماءهم ",
    protos,

)

In [ ]:
rag_result

'تتكون كلية الحاسبات وتقنية المعلومات في جامعة جدة من قسمين رئيسيين:\n\n1. **كلية الحاسبات وتقنية المعلومات**، والتي تقدم تخصص **تقنية المعلومات**.\n2. **كلية علوم وهندسة الحاسب**، والتي تشمل التخصصات التالية:\n   - الأمن السيبراني\n   - الذكاء الاصطناعي\n   - علوم البيانات\n   - هندسة البرمجيات\n   - هندسة الحاسب والشبكات\n   - علوم الحاسب\n\nإذا كنت بحاجة إلى مزيد من المعلومات حول أي من هذه التخصصات، فلا تتردد في السؤال!'

In [ ]:
# ============================================================
# CELL 3B: KB DEEP RETRIEVAL PATCH
# ============================================================
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

@dataclass
class AgentEvidence:
    source_type: str
    source_id: str
    title: str = ""
    url: str = ""
    snippet: str = ""
    text: str = ""
    score: Optional[float] = None
    meta: Dict[str, Any] = field(default_factory=dict)

@dataclass
class AgentResult:
    ok: bool
    agent_name: str
    used: bool
    answerable: bool
    summary_ar: str
    evidences: List[AgentEvidence] = field(default_factory=list)
    context_text: str = ""
    sources: List[str] = field(default_factory=list)
    meta: Dict[str, Any] = field(default_factory=dict)
from collections import OrderedDict

def build_kb_query_variants(query: str) -> List[str]:
    q = normalize_text(query).strip()
    variants = [q]

    ql = q.lower()

    replacements = [
        ("مقرر حر", "مقرر اختياري"),
        ("المقررات الحرة", "المقررات الاختيارية"),
        ("المواد الحرة", "المواد الاختيارية"),
        ("مواد حرة", "مواد اختيارية"),
        ("elective", "اختياري"),
        ("free course", "مقرر اختياري"),
    ]

    expanded = q
    for a, b in replacements:
        if a in expanded:
            variants.append(expanded.replace(a, b))

    # إجمالي ساعات البرنامج
    if any(x in q for x in ["اجمالي الساعات", "إجمالي الساعات", "عدد ساعات البرنامج", "ساعات البرنامج", "بكالوريوس"]):
        variants.append("اجمالي ساعات برنامج بكالوريوس الذكاء الاصطناعي")
        variants.append("الخطة الدراسية الذكاء الاصطناعي اجمالي ساعات البرنامج")
        variants.append("degree plan total hours bachelor artificial intelligence")

    # المواد المطلوبة / الخطة الكاملة
    if any(x in q for x in ["المواد المتطلبة", "المواد المطلوبة", "المقررات المطلوبة", "الخطة الدراسية", "كل المواد"]):
        variants.append("الخطة الدراسية الكاملة لبرنامج الذكاء الاصطناعي")
        variants.append("المقررات المطلوبة لبرنامج الذكاء الاصطناعي")
        variants.append("all required courses bachelor artificial intelligence")
        variants.append("degree plans curriculum artificial intelligence")

    if "الذكاء الاصطناعي" in q or "ai" in q.lower():
        variants.append("المقررات الاختيارية في برنامج الذكاء الاصطناعي")
        variants.append("خطة الذكاء الاصطناعي elective ELEC I ELEC II")

    if "مقرر" in q or "مادة" in q or "خطة" in q:
        variants.append(q + " degree plan")
        variants.append(q + " electives")

    out = []
    seen = set()
    for v in variants:
        key = v.strip().lower()
        if key and key not in seen:
            seen.add(key)
            out.append(v.strip())

    return out[:6]
def dedup_chunk_hits(chunk_hits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    seen = set()

    for h in chunk_hits:
        key = (
            h.get("collection"),
            h.get("id"),
            h.get("parent_id"),
            h.get("source_ref"),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(h)

    return out


def rebuild_context_from_chunk_hits(chunk_hits: List[Dict[str, Any]]) -> Dict[str, Any]:
    parents = aggregate_parents(chunk_hits, top_parents=int(RETR["top_parents"]))

    ctx = build_context_blocks_from_parents(
        parents=parents,
        max_context_chars=int(RETR["max_context_chars"]),
        max_total_parts=int(RETR["max_total_parts"]),
        max_chunk_chars=int(RETR["max_chunk_chars"]),
        max_parts_per_parent=int(RETR["max_parts_per_parent"]),
        prefer_parent_diversity=bool(RETR["prefer_parent_diversity"]),
    )

    return {
        "parents": parents,
        "context_blocks": ctx["context_blocks"],
        "context_text": ctx["context_text"],
        "sources": ctx["sources"],
        "context_debug": ctx["debug"],
    }


def extract_counting_hints_from_context(context_text: str) -> List[str]:
    hints = []
    t = normalize_text(context_text)

    elec_matches = re.findall(r"\bELEC\s*(I|II|III|IV|V|\d+)\b", t, flags=re.IGNORECASE)
    if elec_matches:
        unique_elec = list(OrderedDict.fromkeys([m.upper() for m in elec_matches]))
        hints.append(f"تظهر في النتائج مقررات اختيارية مرمزة كالتالي: {', '.join(unique_elec)}.")

    free_course_matches = re.findall(r"مقرر\s*حر\s*\d+", t, flags=re.IGNORECASE)
    if free_course_matches:
        unique_free = list(OrderedDict.fromkeys(free_course_matches))
        hints.append(f"تظهر في النتائج مقررات حرة مسماة كالتالي: {', '.join(unique_free)}.")

    return hints


def kb_agent_search_deep(
    query: str,
    protos,
    mode: str = "router",
    use_bm25: bool = True,
    route_decision: Optional[Any] = None,
) -> AgentResult:
    variants = build_kb_query_variants(query)

    merged_chunk_hits = []
    retrieval_runs = []

    for qv in variants:
        rr = retrieve_final(
            query=qv,
            protos=protos,
            mode=mode,
            use_bm25=use_bm25,
            route_decision=route_decision,
        )

        retrieval_runs.append({
            "query_variant": qv,
            "route_mode": rr.get("route_mode"),
            "route_candidates": rr.get("route_candidates"),
            "variant": rr.get("variant"),
            "expanded_to_global": rr.get("route_meta", {}).get("expanded_to_global"),
        })

        merged_chunk_hits.extend(rr.get("chunk_hits", []) or [])

    merged_chunk_hits = dedup_chunk_hits(merged_chunk_hits)

    merged_chunk_hits = sorted(
        merged_chunk_hits,
        key=lambda x: (
            -(x.get("rrf_score", 0.0) or 0.0),
            x.get("distance", 999.0)
        )
    )[: int(RETR["final_k"])]

    rebuilt = rebuild_context_from_chunk_hits(merged_chunk_hits)
    context_text = rebuilt["context_text"]
    sources = rebuilt["sources"]

    counting_hints = extract_counting_hints_from_context(context_text)
    enriched_context = context_text

    if counting_hints:
        enriched_context += "\n\n[COUNTING_HINTS]\n" + "\n".join(f"- {h}" for h in counting_hints)

    evidences = []
    for i, ch in enumerate(merged_chunk_hits[:10], 1):
        evidences.append(
            AgentEvidence(
                source_type="kb",
                source_id=ch.get("id", f"kb_{i}"),
                title=ch.get("source_ref") or ch.get("collection") or "KB",
                url="",
                snippet="",
                text=ch.get("text", ""),
                score=ch.get("rrf_score", ch.get("sim")),
                meta={
                    "collection": ch.get("collection"),
                    "parent_id": ch.get("parent_id"),
                    "source_ref": ch.get("source_ref"),
                }
            )
        )

    answerable = context_answers_question(query, enriched_context)

    return AgentResult(
        ok=True,
        agent_name="kb_agent_deep",
        used=True,
        answerable=answerable,
        summary_ar="تم تنفيذ البحث العميق في قاعدة المعرفة.",
        evidences=evidences,
        context_text=enriched_context,
        sources=sources,
        meta={
            "query_variants": variants,
            "retrieval_runs": retrieval_runs,
            "counting_hints": counting_hints,
            "merged_hits": len(merged_chunk_hits),
            "route_decision_used": route_decision is not None,
        }
    )

In [ ]:
# ============================================================
# CELL 4: UNIFIED LLM ROUTER / PLANNER
# Semantic Path Router: Student Record / RAG / Hybrid
# Compatible with existing Cell 6 route_question(...)
# ============================================================

from dataclasses import dataclass, field, asdict
from typing import Dict, Any, Optional, List
import json
import re


# ============================================================
# DATA STRUCTURES
# ============================================================

@dataclass
class RouteDecision:
    primary_intent: str
    answer_mode: str

    needs_rag: bool
    needs_sql: bool
    needs_risk_model: bool
    needs_support_model: bool

    is_multi_intent: bool = False
    is_multi_source: bool = False

    sql_purpose: Optional[str] = None
    rag_purpose: Optional[str] = None
    model_purpose: Optional[str] = None

    final_reasoning_mode: str = "direct_answer"

    rag_collections: List[str] = field(default_factory=list)
    db_tables: List[str] = field(default_factory=list)

    sql_task_hint: Optional[str] = None
    student_id: Optional[str] = None

    can_answer_now: bool = True
    missing_information: List[str] = field(default_factory=list)

    confidence: float = 0.0
    reasoning_brief: str = ""

    original_question: Optional[str] = None
    routing_question: Optional[str] = None
    was_rewritten: bool = False

    retrieval_strategy: str = "multi"
    initial_collections: List[str] = field(default_factory=list)
    must_search_collections: List[str] = field(default_factory=list)
    expand_if_weak: bool = True
    allow_global_fallback: bool = True
    collection_reasoning: str = ""


# ============================================================
# CONSTANTS
# ============================================================

VALID_RAG_COLLECTIONS = [
    "regulations",
    "transfer_rules",
    "degree_plans",
    "electives",
    "student_helpers",
    "faculty_offices",
    "clubs",
    "specialization",
    "coop_replies",
    "academic_calendar",
    "coop_rules",
    "activities",
    "course_bundles",
]

VALID_DB_TABLES = [
    "students",
    "student_courses",
    "courses",
    "risk_features",
    "support_features",
]


# ============================================================
# ROUTER NORMALIZATION
# ============================================================

def normalize_query_for_tools(text: str) -> str:
    if text is None:
        return ""

    t = str(text)
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    t = t.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
    t = re.sub(r"\s+", " ", t).strip()
    return t


def extract_student_id(text: str):
    text = normalize_query_for_tools(text)
    patterns = [
        r"(?:student\s*id|student|id|رقم\s*الطالب|رقم\s*الطالبة|الطالب\s*رقم|الطالبة\s*رقم|الطالب|الطالبة)\s*[:#\-]?\s*(\d{3,12})",
        r"\b(\d{3,12})\b",
    ]

    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            return m.group(1)

    return None


def _dedup_valid(items: List[str], valid_items: List[str]) -> List[str]:
    out = []
    seen = set()
    valid_set = set(valid_items)

    for x in items or []:
        x = str(x).strip()
        if x in valid_set and x not in seen:
            seen.add(x)
            out.append(x)

    return out


# ============================================================
# OPTIONAL MEMORY REWRITE
# ============================================================

def prepare_question_for_routing(question: str) -> Dict[str, Any]:
    original = question
    cleaned = normalize_query_for_tools(question)

    try:
        if callable(globals().get("rewrite_query_with_memory")):
            rewritten = rewrite_query_with_memory(cleaned)
        else:
            rewritten = cleaned
    except Exception:
        rewritten = cleaned

    routing_question = normalize_query_for_tools(rewritten)

    return {
        "original_question": original,
        "cleaned_question": cleaned,
        "routing_question": routing_question,
        "was_rewritten": routing_question != cleaned,
    }


# ============================================================
# SEMANTIC PATH GUARDS
# ============================================================

def is_student_service_question(question: str) -> bool:
    """
    RAG/general university services, not student records.
    Example:
    - الطالبات المساعدات
    - دعم نفسي عام
    - خدمات الطالبات
    - أنشطة، أندية، إرشاد عام
    """
    q = normalize_query_for_tools(question).lower()

    service_markers = [
        "الطالبات المساعدات",
        "طالبات مساعدات",
        "student helpers",
        "peer tutor",
        "peer tutors",
        "مساعدة الطالبات",
        "مساعدات",
        "خدمات الطالبات",
        "خدمة الطالبات",
        "خدمات الطلاب",
        "دعم نفسي عام",
        "الدعم النفسي",
        "إرشاد عام",
        "ارشاد عام",
        "الإرشاد الطلابي",
        "الارشاد الطلابي",
        "أنشطة",
        "انشطة",
        "أندية",
        "اندية",
        "نادي",
        "club",
        "clubs",
    ]

    return any(x in q for x in service_markers)


def is_student_record_question(question: str) -> bool:
    """
    SQL / Student Record path:
    questions about actual student records, numbers, courses, advisors,
    health/accommodation flags, training status, backlog, risk/support features.
    """
    q = normalize_query_for_tools(question).lower()
    sid = extract_student_id(q)

    if sid:
        return True

    record_markers = [
        "فتح استثنائي",
"فتح استثنائي للطالبات",
"مادة تحتاج فتح",
"مقرر يحتاج فتح",
"مواد تحتاج فتح",
"المواد التي تحتاج فتح",
"استثنائي للطالبات",
        "كم عدد",
        "عدد الطالبات",
        "عدد الطلاب",
        "عدد المتوقع",
        "المتوقع تخرج",
        "متوقع تخرج",
        "المتوقع تدريب",
        "تدريبهم",
        "تدريبهن",
        "المرشدة",
        "المرشده",
        "المرشد",
        "عند كل مرشدة",
        "عند كل مرشده",
        "كل مرشدة",
        "كل مرشده",
        "حسب المرشدة",
        "حسب المرشده",
        "متعثر",
        "متعثرات",
        "متعثرين",
        "غير مجتاز",
        "لم تجتز",
        "لم يجتاز",
        "مواد الطالبة",
        "مواد الطالب",
        "المواد الباقية",
        "المواد الباقيه",
        "حالة صحية",
        "الحالة الصحية",
        "تسهيلات",
        "احتياج خاص",
        "دعم للطالبة",
        "دعم للطالب",
        "تحتاج دعم",
        "يحتاج دعم",
        "خطورة",
        "خطر",
        "risk",
        "support",
        "student record",
    ]

    return any(x in q for x in record_markers) and not is_student_service_question(q)


def is_policy_or_knowledge_question(question: str) -> bool:
    """
    RAG path:
    university policies, regulations, plans, clubs, activities, general services.
    """
    q = normalize_query_for_tools(question).lower()

    policy_markers = [
        "لائحة",
        "لوائح",
        "نظام",
        "أنظمة",
        "انظمة",
        "سياسة",
        "سياسات",
        "شروط",
        "شرط",
        "خطة",
        "خطط",
        "ساعات الخطة",
        "مقررات الخطة",
        "تحويل",
        "معادلة",
        "تخصص",
        "مسار",
        "تقويم",
        "مواعيد",
        "حذف وإضافة",
        "حذف واضافة",
        "أندية",
        "اندية",
        "أنشطة",
        "انشطة",
        "مكتب",
        "مكاتب",
        "عضو هيئة",
        "التدريب التعاوني",
        "جهات التدريب",
        "شركات التدريب",
        "الطالبات المساعدات",
        "خدمات الطالبات",
        "student helpers",
        "clubs",
        "calendar",
        "degree plan",
        "transfer",
        "regulations",
    ]

    return any(x in q for x in policy_markers) or is_student_service_question(q)


def is_hybrid_student_policy_question(question: str) -> bool:
    """
    Hybrid path:
    requires student record + university regulations/policies.
    Example:
    - حلل الطالب 2216908 واذكر التوصية وفق الأنظمة
    - هل الطالبة مؤهلة للتحويل؟
    """
    q = normalize_query_for_tools(question).lower()
    sid = extract_student_id(q)

    if not sid:
        return False

    hybrid_markers = [
        "وفق الأنظمة",
        "وفق الانظمة",
        "وفق الانظمه",
        "وفق النظام",
        "الأنظمة",
        "الانظمة",
        "الانظمه",
        "لائحة",
        "لوائح",
        "نظام",
        "شروط",
        "توصية",
        "التوصية",
        "التوصيه",
        "تحويل",
        "مؤهل",
        "اهلية",
        "أهلية",
        "حلل",
        "تحليل",
    ]

    return any(x in q for x in hybrid_markers)


# ============================================================
# LLM PLANNER PROMPT
# ============================================================

UNIFIED_ROUTER_SYSTEM_PROMPT = """
أنت مخطط Routing لنظام أكاديمي ذكي لجامعة جدة.

مهمتك بناء خطة تنفيذ، وليس الإجابة على السؤال.

المسارات المتاحة:

1) Student Record / SQL Path
هذا المسار يستخدم لبيانات الطلاب الفعلية من قاعدة البيانات، مثل:
- بيانات طالب أو طالبة محددة
- عدد الطلاب أو الطالبات
- المرشدات والطلاب التابعون لهن
- حالة التدريب لطالب أو مجموعة طلاب
- التعثر، المواد غير المجتازة، المواد المتبقية
- الحالات الصحية، التسهيلات، الاحتياج للدعم
- المتوقع تخرجهم
- التحليلات المبنية على سجلات الطلاب

2) Knowledge / RAG Path
هذا المسار يستخدم لمعلومات الجامعة العامة، مثل:
- اللوائح والأنظمة
- الخطط الدراسية
- التحويل والمعادلة
- الأندية والأنشطة
- الطالبات المساعدات والخدمات العامة
- التقويم الأكاديمي
- مكاتب أعضاء هيئة التدريس
- سياسات التدريب التعاوني وجهاته

VECTOR_KNOWLEDGE_BASE / RAG:
- regulations: أنظمة ولوائح الجامعة العامة
- transfer_rules: قواعد التحويل والمعادلة
- degree_plans: خطط الكلية والبرامج الدراسية
- electives: المقررات الاختيارية الخاصة بالذكاء الاصطناعي
- student_helpers: الطالبات المساعدات لمساعدة الطالبات في فهم المواد
- faculty_offices: مكاتب أعضاء هيئة التدريس والإيميلات
- clubs: أندية الجامعة
- specialization: التخصصات والمسارات ووصف كل تخصص ومتطلباته
- coop_replies: تجارب الطالبات مع جهات وشركات التدريب التعاوني
- academic_calendar: التقويم الأكاديمي والمواعيد
- coop_rules: قواعد وسياسات التدريب التعاوني
- activities: الأنشطة والفعاليات خارج الجامعة
- course_bundles: مقررات الذكاء الاصطناعي المطروحة/المدروسة في الفصل الحالي

3) Hybrid Path
يستخدم عندما يحتاج السؤال المصدرين معًا:
- بيانات طالب من SQL
- نظام أو سياسة أو توصية من RAG

مثال:
"حلل الطالب 2216908 واذكر التوصية وفق الأنظمة"
=> needs_sql=true و needs_rag=true

قواعد مهمة جدًا:
- لا تجعل كلمة "طالبات" وحدها سببًا لاختيار SQL.
- إذا السؤال عن خدمة عامة للطالبات مثل الطالبات المساعدات أو الإرشاد أو الأنشطة، فهذا RAG وليس SQL.
- إذا السؤال عن سجلات أو أعداد أو حالات أو مرشدات أو مواد لطالبات فعليات، فهذا SQL.
- إذا السؤال عن طالب محدد ومعه توصية أو نظام أو تحويل، فهذا Hybrid.
- لا تعتمد على الكلمات فقط؛ افهم نوع المعلومة المطلوبة.
- أعد JSON فقط.
""".strip()

UNIFIED_ROUTER_USER_PROMPT = """
حلل السؤال التالي:

السؤال:
{question}

أعد JSON فقط بهذا الشكل:
{{
  "primary_intent": "sql|rag|risk|support|hybrid|unknown",
  "answer_mode": "brief|analysis",

  "needs_rag": true,
  "needs_sql": false,
  "needs_risk_model": false,
  "needs_support_model": false,

  "is_multi_intent": false,
  "is_multi_source": false,

  "sql_purpose": null,
  "rag_purpose": null,
  "model_purpose": null,

  "final_reasoning_mode": "direct_answer|compare|explain|diagnose|recommend",

  "rag_collections": [],
  "db_tables": [],

  "sql_task_hint": null,

  "can_answer_now": true,
  "missing_information": [],

  "confidence": 0.0,
  "reasoning_brief": "",

  "retrieval_strategy": "single|multi|global",
  "initial_collections": [],
  "must_search_collections": [],
  "expand_if_weak": true,
  "allow_global_fallback": true,
  "collection_reasoning": ""
}}

اختيار المسار:
- Student Record / SQL:
  أسئلة سجلات الطلاب، الأعداد، التدريب، التعثر، المواد، المرشدات، الحالة الصحية، التسهيلات، الدعم لطالب محدد.

- Knowledge / RAG:
  أسئلة الأنظمة، اللوائح، الخطط، الطالبات المساعدات، الخدمات العامة، الأندية، الأنشطة، التقويم، مكاتب أعضاء هيئة التدريس.

- Hybrid:
  أسئلة تجمع طالبًا محددًا مع نظام أو توصية أو أهلية أو تحويل.

المجموعات المسموحة في RAG:
["regulations","transfer_rules","degree_plans","electives","student_helpers","faculty_offices","clubs","specialization","coop_replies","academic_calendar","coop_rules"]

الجداول المسموحة في SQL:
["students","student_courses","courses","risk_features","support_features"]

إرشادات اختيار المجموعات:
- سؤال لائحة/نظام/شروط: regulations
- سؤال تحويل/معادلة: transfer_rules + regulations
- سؤال خطة/ساعات/مواد برنامج: degree_plans + electives
- سؤال الطالبات المساعدات أو خدمات الطالبات: student_helpers
- سؤال أندية أو أنشطة: clubs + student_helpers
- سؤال تدريب تعاوني كجهات أو شركات: coop_replies + coop_rules + student_helpers
- سؤال تدريب تعاوني كسياسة/شروط: coop_rules + regulations
- سؤال تقويم/مواعيد: academic_calendar
- سؤال مكاتب دكاترة: faculty_offices
- سؤال تخصصات أو مسارات: specialization + degree_plans
""".strip()


# ============================================================
# JSON HELPERS
# ============================================================

def _safe_json_object(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except Exception:
        pass

    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
    except Exception:
        pass

    return {}


def _bool(x) -> bool:
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes", "y"]


def _safe_float(x, default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default


# ============================================================
# HEURISTIC COLLECTION HINTS
# ============================================================

def infer_collection_hints(question: str) -> List[str]:
    q = normalize_query_for_tools(question).lower()
    cols = []

    def add(*xs):
        for x in xs:
            if x not in cols:
                cols.append(x)

    if any(x in q for x in ["الطالبات المساعدات", "طالبات مساعدات", "student helpers", "خدمات الطالبات", "مساعدة الطالبات"]):
        add("student_helpers")

    if any(x in q for x in ["خطة", "ساعات", "ساعة", "مواد", "مقررات", "برنامج", "بكالوريوس", "degree plan"]):
        add("degree_plans")

    if any(x in q for x in ["اختياري", "حرة", "حر", "elective"]):
        add("electives", "degree_plans")

    if any(x in q for x in ["لائحة", "لوائح", "نظام", "أنظمة", "انظمة", "سياسة", "شرط", "شروط", "حرمان", "اعتذار", "تأجيل"]):
        add("regulations")

    if any(x in q for x in ["تحويل", "معادلة", "transfer"]):
        add("transfer_rules", "regulations")

    if any(x in q for x in ["تدريب", "تعاوني", "شركة", "شركات", "جهة", "جهات", "coop", "internship"]):
        add("coop_replies", "coop_rules", "student_helpers")

    if any(x in q for x in ["تقويم", "موعد", "مواعيد", "اختبارات", "حذف", "اضافة", "إضافة", "calendar"]):
        add("academic_calendar")

    if any(x in q for x in ["نادي", "أندية", "اندية", "club"]):
        add("clubs", "student_helpers")

    if any(x in q for x in ["مكتب", "عضو هيئة", "دكتور", "دكتورة", "faculty"]):
        add("faculty_offices")

    if any(x in q for x in ["تخصص", "مسار", "ذكاء اصطناعي", "علوم بيانات", "cyber", "ai", "specialization"]):
        add("specialization", "degree_plans")

    if any(x in q for x in ["نشاط", "أنشطة", "انشطة", "فعالية", "فعاليات", "خارج الجامعة", "خارج الجامعه", "activities"]):
        add("activities")

    if any(x in q for x in ["مواد هذا الفصل", "مقررات هذا الفصل", "المواد الحالية", "المقررات الحالية", "رزمه ", "رزمة"]):
        add("course_bundles")
    return _dedup_valid(cols, VALID_RAG_COLLECTIONS)


def infer_retrieval_strategy(question: str, cols: List[str]) -> str:
    q = normalize_query_for_tools(question).lower()

    broad_markers = [
        "كل", "جميع", "اشرح", "قارن", "أفضل", "افضل",
        "معلومات", "ايش الموجود", "وش عندكم", "هل توجد معلومات",
        "عام", "general",
    ]

    if any(x in q for x in broad_markers):
        return "multi"

    if len(cols) >= 3:
        return "multi"

    if len(cols) == 0:
        return "global"

    if len(cols) == 1:
        return "single"

    return "multi"


# ============================================================
# FALLBACK ROUTER
# ============================================================

def _fallback_route(question: str, reason: str = "llm_failed") -> RouteDecision:
    q = normalize_query_for_tools(question).lower()
    sid = extract_student_id(q)

    record = is_student_record_question(q)
    knowledge = is_policy_or_knowledge_question(q)
    hybrid = is_hybrid_student_policy_question(q)

    needs_sql = record or hybrid
    needs_rag = knowledge or hybrid

    needs_risk = any(x in q for x in ["خطورة", "risk", "خطر"])
    needs_support = any(x in q for x in ["دعم", "support", "stress", "توتر", "نفسي", "سلوكي"])

    if needs_risk or needs_support:
        needs_sql = True

    if not any([needs_sql, needs_rag, needs_risk, needs_support]):
        needs_rag = True

    rag_cols = infer_collection_hints(question)

    if needs_rag and not rag_cols:
        rag_cols = []

    strategy = infer_retrieval_strategy(question, rag_cols)

    if needs_rag and not rag_cols:
        strategy = "global"

    primary = "hybrid" if sum([needs_sql, needs_rag, needs_risk, needs_support]) > 1 else (
        "sql" if needs_sql else
        "risk" if needs_risk else
        "support" if needs_support else
        "rag"
    )

    return RouteDecision(
        primary_intent=primary,
        answer_mode="analysis" if any(x in q for x in ["اشرح", "حلل", "تحليل", "ليش", "why", "explain", "قارن"]) else "brief",

        needs_rag=needs_rag,
        needs_sql=needs_sql,
        needs_risk_model=needs_risk,
        needs_support_model=needs_support,

        is_multi_intent=sum([needs_sql, needs_rag, needs_risk, needs_support]) > 1,
        is_multi_source=sum([needs_sql, needs_rag, needs_risk, needs_support]) > 1,

        sql_purpose="student_record" if needs_sql else None,
        rag_purpose="university_knowledge" if needs_rag else None,
        model_purpose="risk_or_support" if (needs_risk or needs_support) else None,

        final_reasoning_mode="recommend" if hybrid else "direct_answer",

        rag_collections=rag_cols,
        initial_collections=rag_cols,
        must_search_collections=rag_cols[:2],

        db_tables=VALID_DB_TABLES if needs_sql else [],

        sql_task_hint="student_record_query" if needs_sql else None,
        student_id=sid,

        can_answer_now=True,
        missing_information=[],

        confidence=0.65,
        reasoning_brief=reason,
        collection_reasoning=f"fallback inferred: record={record}, knowledge={knowledge}, hybrid={hybrid}, collections={rag_cols}",

        original_question=question,
        routing_question=question,
        was_rewritten=False,

        retrieval_strategy=strategy,
        expand_if_weak=True,
        allow_global_fallback=True,
    )


# ============================================================
# NORMALIZE DECISION
# ============================================================

def normalize_route_decision(decision: RouteDecision, question: str) -> RouteDecision:
    q = normalize_query_for_tools(question).lower()

    decision.rag_collections = _dedup_valid(decision.rag_collections, VALID_RAG_COLLECTIONS)
    decision.initial_collections = _dedup_valid(decision.initial_collections, VALID_RAG_COLLECTIONS)
    decision.must_search_collections = _dedup_valid(decision.must_search_collections, VALID_RAG_COLLECTIONS)
    decision.db_tables = _dedup_valid(decision.db_tables, VALID_DB_TABLES)

    record = is_student_record_question(q)
    knowledge = is_policy_or_knowledge_question(q)
    hybrid = is_hybrid_student_policy_question(q)

    # Guardrails after LLM:
    # Student services must not become SQL only.
    if is_student_service_question(q) and not hybrid:
        decision.needs_sql = False
        decision.needs_rag = True
        decision.primary_intent = "rag"
        decision.rag_purpose = "student_services_or_general_university_knowledge"

    # Student record questions must use SQL.
    if record:
        decision.needs_sql = True
        decision.sql_purpose = decision.sql_purpose or "student_record"

    # Hybrid questions must use both.
    if hybrid:
        decision.needs_sql = True
        decision.needs_rag = True
        decision.primary_intent = "hybrid"
        decision.final_reasoning_mode = "recommend"

    inferred = infer_collection_hints(q)

    if decision.needs_rag:
        if not decision.rag_collections:
            decision.rag_collections = inferred[:]

        if not decision.initial_collections:
            decision.initial_collections = decision.rag_collections[:] or inferred[:]

        if not decision.must_search_collections:
            decision.must_search_collections = decision.initial_collections[:2]

    if decision.needs_sql and not decision.db_tables:
        decision.db_tables = VALID_DB_TABLES[:]

    if decision.retrieval_strategy not in ["single", "multi", "global"]:
        decision.retrieval_strategy = infer_retrieval_strategy(q, decision.initial_collections)

    if decision.needs_rag:
        if len(decision.initial_collections) >= 2:
            decision.retrieval_strategy = "multi"

        if not decision.initial_collections:
            decision.retrieval_strategy = "global"

        decision.expand_if_weak = True
        decision.allow_global_fallback = True

    if not any([
        decision.needs_rag,
        decision.needs_sql,
        decision.needs_risk_model,
        decision.needs_support_model,
    ]):
        decision.needs_rag = True
        decision.primary_intent = "rag"
        decision.retrieval_strategy = "global"
        decision.reasoning_brief += " | normalized_empty_plan_to_rag"

    decision.is_multi_source = sum([
        decision.needs_rag,
        decision.needs_sql,
        decision.needs_risk_model,
        decision.needs_support_model,
    ]) > 1

    decision.is_multi_intent = decision.is_multi_source

    if decision.is_multi_source:
        decision.primary_intent = "hybrid"

    return decision


# ============================================================
# MAIN PUBLIC ROUTER
# ============================================================

def route_question(question: str) -> RouteDecision:
    prep = prepare_question_for_routing(question)
    routing_question = prep["routing_question"]

    try:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": UNIFIED_ROUTER_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": UNIFIED_ROUTER_USER_PROMPT.format(question=routing_question),
                },
            ],
            temperature=0.0,
            max_tokens=700,
        )

        raw = (resp.choices[0].message.content or "").strip()
        data = _safe_json_object(raw)

        if not data:
            return normalize_route_decision(
                _fallback_route(routing_question, reason="empty_or_invalid_router_json"),
                routing_question,
            )

        decision = RouteDecision(
            primary_intent=str(data.get("primary_intent", "unknown")),
            answer_mode=str(data.get("answer_mode", "brief")),

            needs_rag=_bool(data.get("needs_rag", False)),
            needs_sql=_bool(data.get("needs_sql", False)),
            needs_risk_model=_bool(data.get("needs_risk_model", False)),
            needs_support_model=_bool(data.get("needs_support_model", False)),

            is_multi_intent=_bool(data.get("is_multi_intent", False)),
            is_multi_source=_bool(data.get("is_multi_source", False)),

            sql_purpose=data.get("sql_purpose"),
            rag_purpose=data.get("rag_purpose"),
            model_purpose=data.get("model_purpose"),

            final_reasoning_mode=str(data.get("final_reasoning_mode", "direct_answer")),

            rag_collections=list(data.get("rag_collections", []) or []),
            db_tables=list(data.get("db_tables", []) or []),

            sql_task_hint=data.get("sql_task_hint"),
            student_id=extract_student_id(routing_question),

            can_answer_now=_bool(data.get("can_answer_now", True)),
            missing_information=list(data.get("missing_information", []) or []),

            confidence=_safe_float(data.get("confidence", 0.0), 0.0),
            reasoning_brief=str(data.get("reasoning_brief", "")),

            original_question=prep["original_question"],
            routing_question=routing_question,
            was_rewritten=prep["was_rewritten"],

            retrieval_strategy=str(data.get("retrieval_strategy", "multi")),
            initial_collections=list(data.get("initial_collections", []) or []),
            must_search_collections=list(data.get("must_search_collections", []) or []),
            expand_if_weak=_bool(data.get("expand_if_weak", True)),
            allow_global_fallback=_bool(data.get("allow_global_fallback", True)),
            collection_reasoning=str(data.get("collection_reasoning", "")),
        )

        return normalize_route_decision(decision, routing_question)

    except Exception as e:
        return normalize_route_decision(
            _fallback_route(routing_question, reason=f"router_error: {repr(e)}"),
            routing_question,
        )


# ============================================================
# DEBUG
# ============================================================

def debug_route_question(question: str):
    d = route_question(question)
    print(json.dumps(asdict(d), ensure_ascii=False, indent=2))
    return d


def test_cell4_router():
    tests = [
        "كم عدد الطالبات المتوقع تخرجهن؟",
        "كم عدد المتوقع تدريبهم عند كل مرشدة؟",
        "من المرشدة التي لديها أكبر عدد من الطلاب المتعثرين؟",
        "ما هي الطالبات المساعدات؟",
        "هل توجد خدمات دعم نفسي للطالبات؟",
        "حلل الطالب 2216908 واذكر التوصية وفق الأنظمة",
        "هل الطالبة رقم 123456 تحتاج دعم؟",
        "ما شروط التحويل؟",
        "كم عدد ساعات خطة الذكاء الاصطناعي؟",
        "متى يبدأ الحذف والإضافة؟",
    ]

    for q in tests:
        print("=" * 100)
        print("QUESTION:", q)
        debug_route_question(q)
def smart_autocorrect(question: str) -> str:
    question = basic_autocorrect(question)
    question = ai_autocorrect(question)
    question = remove_repeated_phrases(question)
    return question

print("✅ Cell 4 loaded: Semantic Student Record / RAG / Hybrid Router")

✅ Cell 4 loaded: Semantic Student Record / RAG / Hybrid Router


In [ ]:
# ============================================================
# CELL 5: RISK TOOL + SUPPORT TOOL (ALIGNED / FIXED VERSION)
# ============================================================

# =========================
# RISK MODEL CONFIG
# =========================
RISK_LABEL_MAP = {
    0: "خطورة مرتفعة",
    1: "خطورة منخفضة",
    2: "خطورة متوسطة",
}

RISK_FEATURE_ORDER = [
    "Attendance (%)",
    "Participation_Score",
    "Quizzes_Avg",
    "Assignments_Avg",
    "Midterm_Score",
    "Projects_Score",
]

RISK_CANONICAL_TO_MODEL = {
    "attendance": "Attendance (%)",
    "participation": "Participation_Score",
    "quizzes": "Quizzes_Avg",
    "assignments": "Assignments_Avg",
    "midterm": "Midterm_Score",
    "projects": "Projects_Score",
}


# =========================
# SUPPORT MODEL CONFIG
# =========================
SUPPORT_LABEL_MAP_AR = {
    0: "لا يحتاج دعم",
    1: "يحتاج دعم",
}

SUPPORT_CANONICAL_TO_MODEL = {
    "study_hours_per_week": "Study Hours Per Week",
    "social_media_usage_hours_per_day": "Social Media Usage (Hours per day)",
    "sleep_duration_hours_per_night": "Sleep Duration (Hours per night)",
    "physical_exercise_hours_per_week": "Physical Exercise (Hours per week)",
    "family_support": "Family Support",
    "financial_stress": "Financial Stress",
    "peer_pressure": "Peer Pressure",
    "mental_stress_level": "Mental Stress Level",
    "diet_quality": "Diet Quality",
    "cognitive_distortions": "Cognitive Distortions",
}

HIGH_BAD_FEATURES = {
    "Mental Stress Level",
    "Cognitive Distortions",
    "Financial Stress",
    "Peer Pressure",
    "Social Media Usage (Hours per day)",
}

LOW_BAD_FEATURES = {
    "Sleep Duration (Hours per night)",
    "Family Support",
    "Diet Quality",
    "Physical Exercise (Hours per week)",
    "Study Hours Per Week",
}


# =========================
# RUNTIME CACHES
# =========================
_risk_model = None
_risk_scaler = None

_support_model = None
_support_scaler = None
_support_feature_order = None
_support_threshold = None
_support_numeric_cols = None
_support_feature_means = None
_support_feature_stds = None


# =========================
# HELPERS
# =========================
def count_risk_features(risk_features: Dict[str, Any]) -> int:
    return sum(
        1 for f in RISK_CANONICAL_FEATURES
        if risk_features.get(f) is not None
    )


def count_support_features(support_features: Dict[str, Any]) -> int:
    return sum(
        1 for k, v in support_features.items()
        if k != "student_id" and v is not None
    )


def map_risk_features_to_model_schema(features: Dict[str, Any]) -> Dict[str, Any]:
    if not features:
        return {}

    mapped = {}

    # لو دخلت الأسماء بصيغة النموذج مباشرة
    for model_name in RISK_FEATURE_ORDER:
        if model_name in features and features[model_name] is not None:
            mapped[model_name] = features[model_name]

    # canonical -> model
    for canonical_name, model_name in RISK_CANONICAL_TO_MODEL.items():
        if canonical_name in features and features[canonical_name] is not None and model_name not in mapped:
            mapped[model_name] = features[canonical_name]

    return mapped


def map_support_features_to_model_schema(features: Dict[str, Any]) -> Dict[str, Any]:
    if not features:
        return {}

    mapped = {}

    # لو دخلت الأسماء بصيغة النموذج مباشرة
    if _support_feature_order:
        for col in _support_feature_order:
            if col in features and features[col] is not None:
                mapped[col] = features[col]

    # canonical -> model
    for canonical_name, model_name in SUPPORT_CANONICAL_TO_MODEL.items():
        if canonical_name in features and features[canonical_name] is not None and model_name not in mapped:
            mapped[model_name] = features[canonical_name]

    return mapped


# =========================
# LOAD RISK ARTIFACTS
# =========================
def load_risk_artifacts():
    global _risk_model, _risk_scaler

    if _risk_model is not None and _risk_scaler is not None:
        return

    if not os.path.exists(RISK_MODEL_ZIP):
        raise FileNotFoundError(f"ملف نموذج الخطورة غير موجود: {RISK_MODEL_ZIP}")

    if not os.path.exists(RISK_SCALER_PATH):
        raise FileNotFoundError(f"ملف scaler الخاص بنموذج الخطورة غير موجود: {RISK_SCALER_PATH}")

    model = TabNetClassifier()
    model.load_model(RISK_MODEL_ZIP)

    scaler = joblib.load(RISK_SCALER_PATH)

    _risk_model = model
    _risk_scaler = scaler



# =========================
# LOAD SUPPORT ARTIFACTS
# =========================
def load_support_artifacts():
    global _support_model, _support_scaler
    global _support_feature_order, _support_threshold
    global _support_numeric_cols, _support_feature_means, _support_feature_stds

    if (
        _support_model is not None
        and _support_scaler is not None
        and _support_feature_order is not None
    ):
        return

    if tf is None:
        raise RuntimeError("TensorFlow غير متوفر. ثبّت TensorFlow أولًا لتشغيل نموذج الدعم.")

    required_paths = {
        "SUPPORT_MODEL_PATH": SUPPORT_MODEL_PATH,
        "SUPPORT_SCALER_PATH": SUPPORT_SCALER_PATH,
        "SUPPORT_PPINFO_PATH": SUPPORT_PPINFO_PATH,
        "SUPPORT_AECOLS_PATH": SUPPORT_AECOLS_PATH,
    }

    missing_files = [name for name, path in required_paths.items() if not os.path.exists(path)]
    if missing_files:
        raise FileNotFoundError("ملفات نموذج الدعم التالية غير موجودة: " + ", ".join(missing_files))

    _support_model = tf.keras.models.load_model(SUPPORT_MODEL_PATH, compile=False)
    _support_scaler = joblib.load(SUPPORT_SCALER_PATH)

    with open(SUPPORT_AECOLS_PATH, "r", encoding="utf-8") as f:
        _support_feature_order = json.load(f)

    with open(SUPPORT_PPINFO_PATH, "r", encoding="utf-8") as f:
        prep = json.load(f)

    _support_threshold = prep.get("threshold")
    if _support_threshold is None:
        _support_threshold = DEFAULT_SUPPORT_THRESHOLD

    _support_numeric_cols = prep.get("numeric_cols", _support_feature_order)

    if hasattr(_support_scaler, "mean_") and hasattr(_support_scaler, "scale_"):
        _support_feature_means = dict(zip(_support_numeric_cols, _support_scaler.mean_))
        _support_feature_stds = dict(zip(_support_numeric_cols, _support_scaler.scale_))
    else:
        _support_feature_means = {}
        _support_feature_stds = {}



# =========================
# SUPPORT RISK HELPERS
# =========================
def directional_univariate_risk(features: Dict[str, Any]) -> float:
    scores = []

    for col in (_support_numeric_cols or []):
        if col not in features or features[col] is None:
            continue

        mu = _support_feature_means.get(col)
        sd = _support_feature_stds.get(col)

        if mu is None or sd is None or sd == 0:
            continue

        val = float(features[col])

        if col in HIGH_BAD_FEATURES:
            z = (val - mu) / sd
            if z > 1:
                scores.append(float(z))

        elif col in LOW_BAD_FEATURES:
            z = (mu - val) / sd
            if z > 1:
                scores.append(float(z))

    return max(scores) if scores else 0.0


def top_stress_factor(features: Dict[str, Any]) -> Optional[str]:
    best_name = None
    best_z = 0.0

    for col in (_support_numeric_cols or []):
        if col not in features or features[col] is None:
            continue

        mu = _support_feature_means.get(col)
        sd = _support_feature_stds.get(col)

        if mu is None or sd is None or sd == 0:
            continue

        val = float(features[col])

        if col in HIGH_BAD_FEATURES:
            z = max(0.0, (val - mu) / sd)
        elif col in LOW_BAD_FEATURES:
            z = max(0.0, (mu - val) / sd)
        else:
            z = 0.0

        if z > best_z:
            best_z = z
            best_name = col

    return best_name


# =========================
# RISK TOOL
# =========================
def run_risk_tool(features: Dict[str, Any]) -> Dict[str, Any]:
    try:
        load_risk_artifacts()

        mapped_features = map_risk_features_to_model_schema(features)
        missing = [f for f in RISK_FEATURE_ORDER if f not in mapped_features or mapped_features[f] is None]

        if missing:
            return make_tool_response(
                ok=False,
                tool_name="risk_level",
                task_type="risk_prediction",
                summary_ar="تعذر تشغيل نموذج الخطورة لأن بعض القيم المطلوبة غير متوفرة.",
                structured_output={},
                features_used=mapped_features,
                missing_features=missing,
                warnings=["يجب توفير جميع مدخلات نموذج الخطورة قبل تشغيله."],
                error="القيم الناقصة لتشغيل نموذج الخطورة: " + ", ".join(to_ar_feature_list(missing)),
                model_type="TabNetClassifier",
                provenance={
                    "source": "risk_model",
                    "model_type": "TabNetClassifier",
                    "artifacts": {
                        "model_path": RISK_MODEL_ZIP,
                        "scaler_path": RISK_SCALER_PATH,
                    }
                },
            )

        df = pd.DataFrame(
            [[float(mapped_features[f]) for f in RISK_FEATURE_ORDER]],
            columns=RISK_FEATURE_ORDER
        )

        scaled = _risk_scaler.transform(df)

        pred = int(_risk_model.predict(scaled)[0])
        label = RISK_LABEL_MAP.get(pred, f"Class_{pred}")

        confidence = None
        probabilities = None
        proba = None

        if hasattr(_risk_model, "predict_proba"):
            proba = _risk_model.predict_proba(scaled)[0]
            confidence = float(np.max(proba))
            probabilities = {
                RISK_LABEL_MAP[i]: safe_round(proba[i], 4)
                for i in range(len(proba))
                if i in RISK_LABEL_MAP
            }

        summary_ar = f"تنبأ نموذج الخطورة بأن مستوى الطالب هو: {label}."
        if confidence is not None:
            summary_ar += f" درجة الثقة التقريبية: {safe_round(confidence, 3)}."

        return make_tool_response(
            ok=True,
            tool_name="risk_level",
            task_type="risk_prediction",
            summary_ar=summary_ar,
            structured_output={
                "risk_label_ar": label,
                "confidence": safe_round(confidence, 4),
                "class_probabilities": probabilities,
            },
            features_used=mapped_features,
            warnings=[],
            error=None,
            model_type="TabNetClassifier",
            provenance={
                "source": "risk_model",
                "model_type": "TabNetClassifier",
                "artifacts": {
                    "model_path": RISK_MODEL_ZIP,
                    "scaler_path": RISK_SCALER_PATH,
                }
            },
            raw_output={
                "pred_class": pred,
                "probabilities_raw": proba.tolist() if proba is not None else None,
            },
        )

    except Exception as e:
        return make_tool_response(
            ok=False,
            tool_name="risk_level",
            task_type="risk_prediction",
            summary_ar="فشل تشغيل نموذج الخطورة.",
            structured_output={},
            features_used=features,
            missing_features=[],
            warnings=[],
            error=f"فشل تشغيل نموذج الخطورة: {str(e)}",
            model_type="TabNetClassifier",
            provenance={
                "source": "risk_model",
                "model_type": "TabNetClassifier",
                "artifacts": {
                    "model_path": RISK_MODEL_ZIP,
                    "scaler_path": RISK_SCALER_PATH,
                }
            },
        )


# =========================
# SUPPORT TOOL
# =========================
def run_support_tool(features: Dict[str, Any]) -> Dict[str, Any]:
    try:
        load_support_artifacts()

        mapped_features = map_support_features_to_model_schema(features)

        missing = [
            f for f in _support_feature_order
            if f not in mapped_features or mapped_features[f] is None
        ]

        if missing:
            return make_tool_response(
                ok=False,
                tool_name="support",
                task_type="support_recommendation",
                summary_ar="تعذر تشغيل نموذج الدعم لأن بعض القيم المطلوبة غير متوفرة.",
                structured_output={},
                features_used=mapped_features,
                missing_features=missing,
                warnings=["يجب توفير جميع مدخلات نموذج الدعم قبل تشغيله."],
                error="القيم الناقصة لتشغيل نموذج الدعم: " + ", ".join(to_ar_feature_list(missing)),
                model_type="Hybrid Autoencoder + Directional Univariate",
                provenance={
                    "source": "support_model",
                    "model_type": "Hybrid Autoencoder + Directional Univariate",
                    "artifacts": {
                        "model_path": SUPPORT_MODEL_PATH,
                        "scaler_path": SUPPORT_SCALER_PATH,
                        "preprocess_info_path": SUPPORT_PPINFO_PATH,
                        "ae_columns_path": SUPPORT_AECOLS_PATH,
                    }
                },
            )

        df = pd.DataFrame(
            [[float(mapped_features[f]) for f in _support_feature_order]],
            columns=_support_feature_order
        )

        scaled = _support_scaler.transform(df)
        recon = _support_model.predict(scaled, verbose=0)

        ae_risk = float(np.mean((scaled - recon) ** 2))
        uni_risk = float(directional_univariate_risk(mapped_features))
        final_risk = float(0.6 * ae_risk + 0.4 * uni_risk)

        pred = 1 if final_risk >= float(_support_threshold) else 0
        label_ar = SUPPORT_LABEL_MAP_AR.get(pred, str(pred))

        confidence = None
        if _support_threshold not in (None, 0):
            diff = abs(final_risk - float(_support_threshold))
            confidence = float(min(diff / float(_support_threshold), 1.0))

        primary_factor = top_stress_factor(mapped_features)
        primary_factor_ar = to_ar_feature_name(primary_factor) if primary_factor else None

        summary_ar = f"تشير نتيجة نموذج الدعم إلى أن الطالب: {label_ar}."
        if confidence is not None:
            summary_ar += f" درجة الثقة التقريبية: {safe_round(confidence, 3)}."

        warnings = []
        if _support_threshold == DEFAULT_SUPPORT_THRESHOLD:
            warnings.append("تم استخدام threshold افتراضي لعدم توفر قيمة محفوظة في ملف الإعدادات.")

        return make_tool_response(
            ok=True,
            tool_name="support",
            task_type="support_recommendation",
            summary_ar=summary_ar,
            structured_output={
                "support_label_ar": label_ar,
                "ae_risk": safe_round(ae_risk, 6),
                "univariate_risk": safe_round(uni_risk, 6),
                "final_risk": safe_round(final_risk, 6),
                "threshold": safe_round(_support_threshold, 6),
                "confidence": safe_round(confidence, 4),
                "primary_stress_factor": primary_factor_ar,
            },
            features_used=mapped_features,
            missing_features=[],
            warnings=warnings,
            error=None,
            model_type="Hybrid Autoencoder + Directional Univariate",
            provenance={
                "source": "support_model",
                "model_type": "Hybrid Autoencoder + Directional Univariate",
                "artifacts": {
                    "model_path": SUPPORT_MODEL_PATH,
                    "scaler_path": SUPPORT_SCALER_PATH,
                    "preprocess_info_path": SUPPORT_PPINFO_PATH,
                    "ae_columns_path": SUPPORT_AECOLS_PATH,
                }
            },
            raw_output={
                "ae_risk": ae_risk,
                "univariate_risk": uni_risk,
                "final_risk": final_risk,
                "pred_class": pred,
                "primary_stress_factor_raw": primary_factor,
            },
        )

    except Exception as e:
        return make_tool_response(
            ok=False,
            tool_name="support",
            task_type="support_recommendation",
            summary_ar="فشل تشغيل نموذج الدعم.",
            structured_output={},
            features_used=features,
            missing_features=[],
            warnings=[],
            error=f"فشل تشغيل نموذج الدعم: {str(e)}",
            model_type="Hybrid Autoencoder + Directional Univariate",
            provenance={
                "source": "support_model",
                "model_type": "Hybrid Autoencoder + Directional Univariate",
                "artifacts": {
                    "model_path": SUPPORT_MODEL_PATH,
                    "scaler_path": SUPPORT_SCALER_PATH,
                    "preprocess_info_path": SUPPORT_PPINFO_PATH,
                    "ae_columns_path": SUPPORT_AECOLS_PATH,
                }
            },
        )


print(" Cell 5 fixed version loaded successfully.")

 Cell 5 fixed version loaded successfully.


In [ ]:
import json
from dataclasses import asdict
from typing import Any, Dict, List


# ============================================================
# LIGHT FINAL ANSWER PROMPT
# ============================================================

STUDENT_ANALYSIS_SYSTEM_PROMPT = """
أنت مساعد أكاديمي لجامعة جدة.

قواعد الإجابة:
1) أجب بلغة المستخدم وبأسلوب واضح ومباشر.
2) أجب فقط على المطلوب في السؤال، ولا تضف تفاصيل لم تُطلب.
3) إذا طلب المستخدم العدد فقط، اذكر العدد فقط.
4) إذا طلب المستخدم الأسماء صراحة مثل: "مع الأسماء"، "اعرض الأسماء"، "من هم"، "مين"، عندها فقط اعرض الأسماء.
5) إذا كانت بيانات SQL موجودة في sql_data، لا تقل إن معلومات الطالب أو البيانات غير متوفرة.
6) إذا كان السؤال عن طالب محدد، فبيانات الطالب تأتي من sql_data وليس من retrieval.
7) إذا كان السؤال يطلب توصية وفق الأنظمة، استخدم sql_data لحالة الطالب واستخدم retrieval للأنظمة والسياسات.
8) إذا لم تظهر قاعدة نظامية واضحة في retrieval، قل ذلك بوضوح، ثم أعطِ توصية مبنية على بيانات الطالب المتاحة.
9) لا تذكر أسماء داخلية مثل sql_data أو retrieval في الإجابة.
10) لا تخترع معلومات غير موجودة.
""".strip()


# ============================================================
# HELPERS
# ============================================================

def safe_intent_name(route_decision: Any) -> str:
    if hasattr(route_decision, "primary_intent"):
        return route_decision.primary_intent
    if isinstance(route_decision, dict):
        return route_decision.get("primary_intent", "unknown")
    return "unknown"


def merge_used_sources(retrieval_sources: Dict[str, List[str]]) -> List[str]:
    if not retrieval_sources:
        return []

    if "kb" in retrieval_sources or "web" in retrieval_sources:
        return retrieval_sources.get("kb", []) + retrieval_sources.get("web", [])

    return (
        retrieval_sources.get("kb", [])
        + retrieval_sources.get("uj_web", [])
        + retrieval_sources.get("full_web", [])
    )


def is_simple_user_question(question: str) -> bool:
    q = normalize_query_for_tools(question).strip().lower()

    starters = [
        "كم",
        "ما هو",
        "ماهي",
        "ما هي",
        "من هو",
        "من هي",
        "أين",
        "متى",
        "عدد",
        "هل",
    ]

    analysis_markers = [
        "حلل", "تحليل", "قارن", "اشرح", "فسر",
        "لماذا", "ليش", "سبب", "استنتج",
    ]

    if any(x in q for x in analysis_markers):
        return False

    return any(q.startswith(s) for s in starters)

def question_requests_names(question: str) -> bool:
    q = normalize_query_for_tools(question).strip().lower()

    markers = [
        "مع الأسماء",
        "مع الاسماء",
        "الأسماء",
        "الاسماء",
        "اعرض الأسماء",
        "اعرض الاسماء",
        "من هم",
        "مين",
        "who",
        "names",
        "list",
    ]

    return any(m in q for m in markers)


def question_needs_student_policy_analysis(question: str) -> bool:
    q = normalize_query_for_tools(question).strip().lower()

    sid = None
    try:
        sid = extract_student_id(q)
    except Exception:
        sid = None

    if not sid:
        return False

    markers = [
        "حلل",
        "تحليل",
        "التوصية",
        "التوصيه",
        "وفق الأنظمة",
        "وفق الانظمة",
        "وفق الانظمه",
        "وفق النظام",
        "الأنظمة",
        "الانظمة",
        "الانظمه",
        "دعم",
        "متعثر",
        "خطورة",
        "خطر",
        "مؤهل",
        "تحويل",
        "حالة الطالب",
        "حالة الطالبة",
    ]

    return any(m in q for m in markers)
def llm_student_analysis_answer(question: str, payload_json: str) -> str:
    user_prompt = f"""
السؤال:
{question}

المعطيات المتاحة:
{payload_json}

اكتب الجواب النهائي المناسب للمستخدم فقط.

تعليمات:
- إذا كان السؤال بسيطًا، أجب باختصار شديد.
- إذا كان السؤال يحتاج تحليلًا، قدّم تحليلًا مختصرًا فقط.
- لا تستخدم عناوين أو أقسام إلا عند الضرورة.
- لا تشرح طريقة عمل النظام.
- لا تذكر تفاصيل داخلية غير مهمة للمستخدم.
""".strip()

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": STUDENT_ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=max(MAX_OUTPUT_TOKENS, 300),
    )

    return (resp.choices[0].message.content or "").strip()

def extract_sql_direct_answer(sql_result: Any, question: str = "") -> str:
    q = normalize_query_for_tools(question).strip().lower()

    def format_rows(rows):
        if not rows:
            return ""

        first = rows[0]

        if isinstance(first, dict) and "course_key" in first:
            lines = ["المواد التي قد تحتاج فتحًا استثنائيًا:"]
            for row in rows:
                course_key = row.get("course_key", "")
                course_label = row.get("course_label", "")
                count = row.get("expected_graduate_need_count", row.get("count", ""))

                line = f"- {course_key}"
                if course_label:
                    line += f" - {course_label}"
                if count != "":
                    line += f" — عدد الطالبات المحتاجات: {count}"

                lines.append(line)

            return "\n".join(lines)

        if isinstance(first, dict) and "advisor" in first:
            value_keys = [k for k in first.keys() if k != "advisor"]
            value_key = value_keys[0] if value_keys else None

            if not value_key:
                return str(rows)

            if any(x in q for x in ["أكبر", "اكبر", "أعلى", "اعلى", "أكثر", "اكثر"]):
                return f"المرشدة الأعلى هي {first.get('advisor')} بعدد {first.get(value_key)}."

            lines = ["النتائج حسب المرشدة:"]
            for row in rows:
                lines.append(f"- {row.get('advisor', 'غير محدد')}: {row.get(value_key, '')}")

            return "\n".join(lines)

        if isinstance(first, dict) and "student_id" in first:
            if question_requests_names(question):
                lines = [f"عدد النتائج هو {len(rows)}", "", "الأسماء:"]
                for i, s in enumerate(rows, start=1):
                    name = s.get("student_name") or ""
                    sid = s.get("student_id") or ""
                    advisor = s.get("advisor") or ""

                    line = f"{i}. {name}".strip()
                    if sid:
                        line += f" - الرقم: {sid}"
                    if advisor:
                        line += f" - المرشدة: {advisor}"
                    lines.append(line)

                return "\n".join(lines)

            return f"عدد النتائج هو {len(rows)}."

        return str(rows)

    if isinstance(sql_result, list):
        return format_rows(sql_result)

    if not isinstance(sql_result, dict):
        return ""

    ans = sql_result.get("answer_ar") or sql_result.get("answer")
    ans = ans.strip() if isinstance(ans, str) else ""

    if isinstance(sql_result.get("data"), list):
        return format_rows(sql_result["data"])

    students = sql_result.get("students") or []
    if not students and isinstance(sql_result.get("data"), dict):
        students = sql_result["data"].get("students") or []

    if students:
        if question_requests_names(question):
            lines = [ans if ans else f"عدد النتائج هو {len(students)}", "", "الأسماء:"]
            for i, s in enumerate(students, start=1):
                name = s.get("student_name") or ""
                sid = s.get("student_id") or ""
                advisor = s.get("advisor") or ""

                line = f"{i}. {name}".strip()
                if sid:
                    line += f" - الرقم: {sid}"
                if advisor:
                    line += f" - المرشدة: {advisor}"
                lines.append(line)

            return "\n".join(lines)

        return ans.replace("مع عرض الأسماء", "").strip() or f"عدد النتائج هو {len(students)}."

    return ans
def looks_relevant_to_training_companies(question: str, sources: List[str]) -> bool:
    q = normalize_text(question)

    if not any(x in q for x in ["تدريب", "شركة", "شركات", "coop", "intern"]):
        return True

    bad_markers = ["elective:", "course:", "degree_plans"]
    if sources and all(any(m in s for m in bad_markers) for s in sources):
        return False

    return True


# ============================================================
# LIGHT REASONING / SYNTHESIS HELPERS
# self-contained fallback helpers for Cell 6
# ============================================================

def compare_student_against_policy(student_data: Dict[str, Any], policy_texts: List[str]) -> Dict[str, Any]:
    return {
        "mode": "compare_student_data_against_policy",
        "student_data_present": bool(student_data),
        "policy_texts_present": bool(policy_texts),
        "notes": "Light fallback comparison helper."
    }


def match_student_to_degree_plan(student_courses: List[Any], degree_plans: List[Any]) -> Dict[str, Any]:
    return {
        "mode": "mixed_lookup",
        "student_courses_count": len(student_courses or []),
        "degree_plans_count": len(degree_plans or []),
        "notes": "Light fallback degree-plan matcher."
    }


def build_recommendation_from_evidence(
    student_data: Dict[str, Any] = None,
    risk_result: Dict[str, Any] = None,
    support_result: Dict[str, Any] = None,
    rag_texts: List[str] = None,
) -> Dict[str, Any]:
    student_data = student_data or {}
    risk_result = risk_result or {}
    support_result = support_result or {}
    rag_texts = rag_texts or []

    recommendations = []

    risk_label = ((risk_result or {}).get("structured_output") or {}).get("risk_label_ar")
    support_label = ((support_result or {}).get("structured_output") or {}).get("support_label_ar")

    if risk_label == "خطورة مرتفعة":
        recommendations.append("يوصى بمتابعة الطالب أكاديميًا بشكل قريب.")
        recommendations.append("يوصى بمراجعة أسباب انخفاض الأداء في الحضور والاختبارات والمشاركة.")

    if support_label == "يحتاج دعم":
        recommendations.append("يوصى بتقديم دعم أكاديمي وإرشادي للطالب.")
    elif (support_result or {}).get("ok") is False:
        recommendations.append("بعض بيانات الدعم غير مكتملة؛ يفضل استكمالها قبل إصدار حكم نهائي على الحاجة للدعم.")

    if not recommendations and rag_texts:
        recommendations.append("يوصى بالالتزام بالضوابط الأكاديمية ذات الصلة الواردة في اللائحة.")

    if not recommendations:
        recommendations.append("لا توجد توصية حاسمة حاليًا، ويُفضل جمع معلومات إضافية عند الحاجة.")

    return {
        "mode": "policy_grounded_recommendation" if rag_texts else "predictive_assessment",
        "recommendations": recommendations,
        "risk_label": risk_label,
        "support_label": support_label,
    }


def fuse_evidence(
    request_type: str,
    sql_result: Dict[str, Any] = None,
    rag_result: Dict[str, Any] = None,
    model_result: Dict[str, Any] = None,
    reasoning_result: Dict[str, Any] = None,
) -> Dict[str, Any]:
    return {
        "request_type": request_type,
        "sql_data": sql_result,
        "rag_data": rag_result,
        "model_data": model_result,
        "reasoning_result": reasoning_result,
        "available_sources": [
            name for name, obj in [
                ("SQL", sql_result),
                ("RAG", rag_result),
                ("MODEL", model_result),
            ] if obj
        ],
        "notes": []
    }

def question_should_force_sql(question: str) -> bool:
    q = normalize_query_for_tools(question).strip().lower()

    count_markers = [
        "كم عدد",
        "عدد",
        "احصاء",
        "إحصاء",
        "اجمالي",
        "إجمالي",
    ]

    sql_markers = [
        "طالب",
        "طالبة",
        "طالبات",
        "طلاب",
        "مقرر",
        "المسجلين",
        "المسجلات",
        "متوقع تخرج",
        "المتوقع تخرجهم",
        "المتوقع تخرجهن",
        "تدريبهم",
        "تدريبهن",
        "كل مرشده",
        "كل مرشدة",
        "كل مرشد",
        "عند كل مرشده",
        "عند كل مرشدة",
        "متعثر",
        "متعثرين",
        "متعثرات",
    ]

    return any(x in q for x in count_markers) and any(x in q for x in sql_markers)
# ============================================================
# EXECUTE SINGLE QUESTION
# ============================================================
def build_no_answer_reason(
    run_sql: bool,
    run_rag: bool,
    run_risk: bool,
    run_support: bool,
) -> str:
    reasons = []

    if run_sql:
        reasons.append("لا توجد بيانات كافية في سجلات الطلاب")

    if run_rag:
        reasons.append("لا توجد معلومات كافية في الأنظمة أو قاعدة المعرفة")

    if run_risk or run_support:
        reasons.append("لا توجد بيانات كافية لتشغيل نموذج التحليل")

    if not reasons:
        reasons.append("لم يتم تحديد مصدر مناسب للإجابة")

    return "عذرًا، لا يمكنني تقديم إجابة دقيقة بسبب:\n- " + "\n- ".join(reasons)


REPLAN_CONFIDENCE_THRESHOLD = 0.40
MAX_REPLAN_ATTEMPTS = 1


def get_route_confidence(route_decision: Any) -> float:
    try:
        if hasattr(route_decision, "confidence"):
            return float(route_decision.confidence)
        if isinstance(route_decision, dict):
            return float(route_decision.get("confidence", 0.0))
    except Exception:
        return 0.0
    return 0.0


def answer_is_weak(text: str) -> bool:
    if not text:
        return True

    weak_markers = [
        "لا أملك",
        "غير كافية",
        "غير متوفرة",
        "لم يتم العثور",
        "لا توجد معلومات",
    ]

    return any(m in text for m in weak_markers)


def should_replan_once(
    route_decision: Any,
    sql_result: Any = None,
    retrieval_answer: str = "",
    risk_result: Any = None,
    support_result: Any = None,
) -> bool:
    confidence = get_route_confidence(route_decision)

    sql_weak = False
    if isinstance(sql_result, dict):
        sql_weak = sql_result.get("ok") is False

    rag_weak = answer_is_weak(retrieval_answer)

    risk_weak = isinstance(risk_result, dict) and risk_result.get("ok") is False
    support_weak = isinstance(support_result, dict) and support_result.get("ok") is False

    return (
        confidence < REPLAN_CONFIDENCE_THRESHOLD
        or sql_weak
        or rag_weak
        or risk_weak
        or support_weak
    )


def explain_no_answer_after_replan(
    run_sql: bool,
    run_rag: bool,
    run_risk: bool,
    run_support: bool,
    sql_result: Any = None,
    retrieval_answer: str = "",
    risk_result: Any = None,
    support_result: Any = None,
    replan_attempted: bool = False,
) -> str:
    reasons = []

    if run_sql:
        if not sql_result:
            reasons.append("لم تُرجع قاعدة بيانات الطلاب نتيجة كافية.")
        elif isinstance(sql_result, dict) and sql_result.get("ok") is False:
            reasons.append("تعذر الحصول على نتيجة موثوقة من مسار سجلات الطلاب.")

    if run_rag:
        if not retrieval_answer:
            reasons.append("لم تُرجع قاعدة المعرفة أو الاسترجاع نتيجة مناسبة.")
        elif answer_is_weak(retrieval_answer):
            reasons.append("المعلومات المسترجعة من قاعدة المعرفة غير كافية للإجابة بثقة.")

    if run_risk and isinstance(risk_result, dict) and risk_result.get("ok") is False:
        reasons.append("تعذر تشغيل نموذج الخطورة أو كانت بياناته غير كافية.")

    if run_support and isinstance(support_result, dict) and support_result.get("ok") is False:
        reasons.append("تعذر تشغيل نموذج الدعم أو كانت بياناته غير كافية.")

    if replan_attempted:
        reasons.append("تمت محاولة إعادة صياغة السؤال وإعادة المعالجة مرة واحدة، لكنها لم تنتج إجابة كافية.")

    if not reasons:
        reasons.append("لم تتوفر أدلة كافية من المصادر المتاحة للإجابة بثقة.")

    return "عذرًا، لا يمكنني تقديم إجابة دقيقة للأسباب التالية:\n- " + "\n- ".join(reasons)
def execute_single_question(
    question: str,
    protos: Dict[str, list],
    retrieval_mode: str = "router",
) -> Dict[str, Any]:

    original_question = question
    rewritten_question = rewrite_query_with_memory(question)

    route_decision = route_question(rewritten_question)
    intent_name = safe_intent_name(route_decision)

    run_sql = getattr(route_decision, "needs_sql", False)
    run_rag = getattr(route_decision, "needs_rag", False)
    run_risk = getattr(route_decision, "needs_risk_model", False)
    run_support = getattr(route_decision, "needs_support_model", False)

    pure_sql_count_question = (
        question_should_force_sql(rewritten_question)
        and not question_needs_student_policy_analysis(rewritten_question)
    )

    if pure_sql_count_question:
        run_sql = True
        run_rag = False
        run_risk = False
        run_support = False

    force_sql = question_should_force_sql(rewritten_question)
    force_student_policy_analysis = question_needs_student_policy_analysis(rewritten_question)

    if force_student_policy_analysis:
        run_sql = True
        run_rag = True
    elif force_sql:
        run_sql = True

    sql_result = None
    rag_result = None
    risk_result = None
    support_result = None

    retrieval_answer = ""
    retrieval_trace = None
    retrieval_sources = {"kb": [], "web": []}

    replan_attempts = 0

    try:
        state = get_conversation_state()
    except Exception:
        state = None

    agent_decision_trace = {
        "O_t": original_question,
        "rewritten_observation": rewritten_question,
        "Z_t": asdict(route_decision) if hasattr(route_decision, "__dataclass_fields__") else route_decision,
        "pi_actions": [],
        "execution_feedback": {},
        "memory_state": {
            "conversation_memory": "langchain_history_used_in_rewriting",
            "system_state": {
                "student_id": state.last_student_id if state else None,
                "advisor": state.last_advisor_name if state else None,
                "active_topic": state.active_topic if state else None,
            }
        }
    }

    if run_sql:
        agent_decision_trace["pi_actions"].append("student_records")
    if run_rag:
        agent_decision_trace["pi_actions"].append("retrieval")
    if run_risk:
        agent_decision_trace["pi_actions"].append("risk_model")
    if run_support:
        agent_decision_trace["pi_actions"].append("support_model")
    # ========================================================
    # 1) Execute SQL if needed
    # ========================================================
    if run_sql:
        try:
            sql_result = sql_student_answering(rewritten_question)

            if isinstance(sql_result, dict):
                sql_result.setdefault("answer_ar", sql_result.get("answer"))
                sql_result.setdefault("task_type", "sql_student_answering")
                sql_result.setdefault("use_sql", True)

        except Exception as e:
            sql_result = {
                "ok": False,
                "source": "sql",
                "answer": "لا أملك إجابة كافية من البيانات الحالية.",
                "answer_ar": None,
                "error": str(e),
            }

    # ========================================================
    # 1.5) Direct student-model fallback for student + policy analysis
    # ========================================================
    if force_student_policy_analysis:
        sid = None

        try:
            sid = extract_student_id(rewritten_question)
        except Exception:
            sid = None

        sql_answer_now = extract_sql_direct_answer(sql_result, rewritten_question)

        sql_is_weak = (
            not isinstance(sql_result, dict)
            or not sql_result.get("ok", False)
            or not sql_answer_now
            or "لا أملك" in sql_answer_now
            or "غير كافية" in sql_answer_now
            or "غير متوفرة" in sql_answer_now
        )

        if sid and sql_is_weak:
            try:
                sql_result = student_model_analysis(
                    student_id=sid,
                    question=rewritten_question,
                    model_type="auto",
                    prefer_db=True,
                )

                if isinstance(sql_result, dict):
                    sql_result.setdefault("answer_ar", sql_result.get("answer"))
                    sql_result.setdefault("task_type", "student_model_analysis_direct_fallback")
                    sql_result.setdefault("use_sql", True)

            except Exception as e:
                sql_result = {
                    "ok": False,
                    "source": "sql",
                    "answer": None,
                    "answer_ar": None,
                    "error": str(e),
                }

    # ========================================================
    # 2) Execute RAG if needed
    # ========================================================
    if run_rag:
        rag_result = answer(
            query=rewritten_question,
            protos=protos,
            mode=retrieval_mode,
            return_debug=True,
            use_deep=True,
            use_web=True,
        )

        if isinstance(rag_result, dict):
            retrieval_answer = rag_result.get("answer", "")
            retrieval_trace = rag_result.get("trace")

            agents = rag_result.get("agents", {}) or {}
            kb_agent_data = agents.get("kb_agent", {}) or {}
            web_agent_data = agents.get("tavily_agent", {}) or {}

            retrieval_sources = {
                "kb": kb_agent_data.get("sources", []) or [],
                "web": web_agent_data.get("sources", []) or [],
            }

            all_sources = retrieval_sources.get("kb", []) + retrieval_sources.get("web", [])
            if not looks_relevant_to_training_companies(rewritten_question, all_sources):
                retrieval_answer = ""
                retrieval_sources = {"kb": [], "web": []}

    # ========================================================
    # 3) ReAct-style second step
    # ========================================================
    sql_answer = extract_sql_direct_answer(sql_result, rewritten_question)

    sql_failed = (
        run_sql
        and (
            not sql_result
            or not isinstance(sql_result, dict)
            or not sql_result.get("ok", False)
            or not sql_answer
            or "لا أملك" in sql_answer
        )
    )

    rag_failed = (
        run_rag
        and (
            not retrieval_answer
            or "لا أملك" in retrieval_answer
            or "غير كافية" in retrieval_answer
        )
    )

    if sql_failed and not run_rag:
        rag_result = answer(
            query=rewritten_question,
            protos=protos,
            mode=retrieval_mode,
            return_debug=True,
            use_deep=True,
            use_web=True,
        )

        if isinstance(rag_result, dict):
            retrieval_answer = rag_result.get("answer", "")
            retrieval_trace = rag_result.get("trace")

            agents = rag_result.get("agents", {}) or {}
            kb_agent_data = agents.get("kb_agent", {}) or {}
            web_agent_data = agents.get("tavily_agent", {}) or {}

            retrieval_sources = {
                "kb": kb_agent_data.get("sources", []) or [],
                "web": web_agent_data.get("sources", []) or [],
            }

        run_rag = True

    if rag_failed and not run_sql:
        try:
            sql_result = sql_student_answering(rewritten_question)

            if isinstance(sql_result, dict):
                sql_result.setdefault("answer_ar", sql_result.get("answer"))
                sql_result.setdefault("task_type", "sql_student_answering")
                sql_result.setdefault("use_sql", True)

            run_sql = True

        except Exception as e:
            sql_result = {
                "ok": False,
                "source": "sql",
                "answer": "لا أملك إجابة كافية من البيانات الحالية.",
                "answer_ar": None,
                "error": str(e),
            }

    # ========================================================
    # 4) Resolve student context only if models needed
    # ========================================================
    student_id = None
    student_profile = None
    data_source = None
    risk_features = {}
    support_features = {}

    if run_risk or run_support:
        resolved = resolve_student_context(question=rewritten_question)

        student_id = resolved.get("student_id")
        student_profile = resolved.get("profile")
        data_source = resolved.get("data_source")

        risk_features = resolved.get("risk_features", {})
        support_features = resolved.get("support_features", {})

        if run_risk:
            risk_input = {k: v for k, v in risk_features.items() if k != "student_id"}
            risk_result = run_risk_tool(risk_input)

        if run_support:
            support_input = {k: v for k, v in support_features.items() if k != "student_id"}
            support_result = run_support_tool(support_input)

    # ========================================================
    # 4.5) Bounded confidence-based replanning loop
    # Runs at most once and never blocks the system
    # ========================================================
    if should_replan_once(
        route_decision=route_decision,
        sql_result=sql_result,
        retrieval_answer=retrieval_answer,
        risk_result=risk_result,
        support_result=support_result,
    ) and replan_attempts < MAX_REPLAN_ATTEMPTS:

        replan_attempts += 1

        try:
            repaired_question = rewrite_query_with_memory(rewritten_question)

            if repaired_question and repaired_question != rewritten_question:
                repaired_route_decision = route_question(repaired_question)

                rewritten_question = repaired_question
                route_decision = repaired_route_decision
                intent_name = safe_intent_name(route_decision)

                run_sql = getattr(route_decision, "needs_sql", False)
                run_rag = getattr(route_decision, "needs_rag", False)
                run_risk = getattr(route_decision, "needs_risk_model", False)
                run_support = getattr(route_decision, "needs_support_model", False)

                sql_result = None
                rag_result = None
                risk_result = None
                support_result = None

                retrieval_answer = ""
                retrieval_trace = None
                retrieval_sources = {"kb": [], "web": []}

                if run_sql:
                    try:
                        sql_result = sql_student_answering(rewritten_question)

                        if isinstance(sql_result, dict):
                            sql_result.setdefault("answer_ar", sql_result.get("answer"))
                            sql_result.setdefault("task_type", "sql_student_answering_replanned")
                            sql_result.setdefault("use_sql", True)

                    except Exception as e:
                        sql_result = {
                            "ok": False,
                            "source": "sql",
                            "answer": "لا أملك إجابة كافية من البيانات الحالية.",
                            "answer_ar": None,
                            "error": str(e),
                        }

                if run_rag:
                    rag_result = answer(
                        query=rewritten_question,
                        protos=protos,
                        mode=retrieval_mode,
                        return_debug=True,
                        use_deep=True,
                        use_web=True,
                    )

                    if isinstance(rag_result, dict):
                        retrieval_answer = rag_result.get("answer", "")
                        retrieval_trace = rag_result.get("trace")

                        agents = rag_result.get("agents", {}) or {}
                        kb_agent_data = agents.get("kb_agent", {}) or {}
                        web_agent_data = agents.get("tavily_agent", {}) or {}

                        retrieval_sources = {
                            "kb": kb_agent_data.get("sources", []) or [],
                            "web": web_agent_data.get("sources", []) or [],
                        }

                        all_sources = retrieval_sources.get("kb", []) + retrieval_sources.get("web", [])
                        if not looks_relevant_to_training_companies(rewritten_question, all_sources):
                            retrieval_answer = ""
                            retrieval_sources = {"kb": [], "web": []}

                if run_risk or run_support:
                    resolved = resolve_student_context(question=rewritten_question)

                    student_id = resolved.get("student_id")
                    student_profile = resolved.get("profile")
                    data_source = resolved.get("data_source")

                    risk_features = resolved.get("risk_features", {})
                    support_features = resolved.get("support_features", {})

                    if run_risk:
                        risk_input = {k: v for k, v in risk_features.items() if k != "student_id"}
                        risk_result = run_risk_tool(risk_input)

                    if run_support:
                        support_input = {k: v for k, v in support_features.items() if k != "student_id"}
                        support_result = run_support_tool(support_input)

        except Exception:
            pass

        # ========================================================
    # 4.6) Agent decision feedback trace
    # Records execution feedback after original execution
    # and optional bounded replanning
    # ========================================================
    agent_decision_trace["execution_feedback"] = {
        "sql_ok": sql_result.get("ok") if isinstance(sql_result, dict) else None,
        "rag_has_answer": bool(retrieval_answer),
        "risk_ok": risk_result.get("ok") if isinstance(risk_result, dict) else None,
        "support_ok": support_result.get("ok") if isinstance(support_result, dict) else None,
        "replanning_attempted": replan_attempts > 0,
        "replanning_attempts": replan_attempts,
    }
    # ========================================================
    # 5) Build final payload
    # ========================================================
    reasoning_result = None

    fused_payload = fuse_evidence(
        request_type=intent_name,
        sql_result=sql_result,
        rag_result={
            "answer": retrieval_answer,
            "trace": retrieval_trace,
            "sources": retrieval_sources,
        } if (retrieval_answer or retrieval_trace or merge_used_sources(retrieval_sources)) else None,
        model_result={
            "risk_result": risk_result,
            "support_result": support_result,
        } if (risk_result or support_result) else None,
        reasoning_result=reasoning_result,
    )

    payload_json = json.dumps({
        "question": original_question,
        "rewritten_question": rewritten_question,
        "route_decision": asdict(route_decision) if hasattr(route_decision, "__dataclass_fields__") else route_decision,
        "student_resolution": {
            "student_id": student_id,
            "profile": student_profile,
            "data_source": data_source,
        },
        "sql_data": sql_result,
        "retrieval": {
            "answer": retrieval_answer,
            "trace": retrieval_trace,
            "sources": retrieval_sources,
        },
        "risk_result": risk_result,
        "support_result": support_result,
        "reasoning": reasoning_result,
        "fusion": fused_payload,
        "student_features": {
            "risk_features": risk_features,
            "support_features": support_features,
        },
        "replanning": {
            "attempted": replan_attempts > 0,
            "attempts": replan_attempts,
            "threshold": REPLAN_CONFIDENCE_THRESHOLD,
        },
      "agent_decision_trace": agent_decision_trace,
    }, ensure_ascii=False, indent=2)

    sql_direct = extract_sql_direct_answer(sql_result, original_question)

    # ========================================================
    # 6) Final answer priority
    # ========================================================
    if run_sql and sql_direct and not run_rag and not run_risk and not run_support:
        if question_requests_names(original_question) or is_simple_user_question(original_question):
            final_answer = sql_direct
        else:
            final_answer = llm_student_analysis_answer(original_question, payload_json)

    elif run_rag and retrieval_answer and not run_sql and not run_risk and not run_support:
        final_answer = retrieval_answer

    elif sql_direct or retrieval_answer or risk_result or support_result:
        final_answer = llm_student_analysis_answer(original_question, payload_json)

    else:
        final_answer = explain_no_answer_after_replan(
            run_sql=run_sql,
            run_rag=run_rag,
            run_risk=run_risk,
            run_support=run_support,
            sql_result=sql_result,
            retrieval_answer=retrieval_answer,
            risk_result=risk_result,
            support_result=support_result,
            replan_attempted=replan_attempts > 0,
        )

    save_turn_to_memory(original_question, final_answer)

    update_conversation_state(
    question=original_question,
    route_decision=route_decision,
    sql_result=sql_result,
    retrieval_answer=retrieval_answer,
    final_answer=final_answer,
    )

    return {
        "question": original_question,
        "rewritten_question": rewritten_question,
        "student_id": student_id,
        "intent": intent_name,
        "route_decision": asdict(route_decision) if hasattr(route_decision, "__dataclass_fields__") else route_decision,
        "answer": final_answer,
        "sql": sql_result,
        "retrieval": {
            "answer": retrieval_answer,
            "trace": retrieval_trace,
            "sources": retrieval_sources,
        },
        "predictive_models": {
            "risk_result": risk_result,
            "support_result": support_result,
        },
        "reasoning": reasoning_result,
        "fusion": fused_payload,
        "student_features": {
            "risk_features": risk_features,
            "support_features": support_features,
        },
        "replanning": {
            "attempted": replan_attempts > 0,
            "attempts": replan_attempts,
            "threshold": REPLAN_CONFIDENCE_THRESHOLD,

        },
    }
    # ============================================================
# MAIN ENTRY
# ============================================================
def rag_student_answer(
    question: str,
    protos: Dict[str, list],
    retrieval_mode: str = "router",

):

    original_question = question
    corrected_question = smart_autocorrect(question)
    return execute_single_question(
        corrected_question,
        protos,
        retrieval_mode,
    )

print(" Cell 6 final light version loaded successfully.")

 Cell 6 final light version loaded successfully.


In [ ]:
# ============================================================
# CELL 700: ACADEMIC COURSE REGISTRATION ADVISING LAYER
# ============================================================

from typing import Dict, Any, List
import math

# ------------------------------------------------------------
# 1. PREREQUISITE CHECKING
# ------------------------------------------------------------
def check_course_prerequisites(student_id: str, course_key: str) -> Dict[str, Any]:
    """يفحص ما إذا كانت الطالبة قد اجتازت المتطلبات السابقة لمقرر معين"""
    student_courses = get_student_courses(student_id)
    course_info = fetch_one("SELECT course_key, course_label, prerequisites FROM courses WHERE course_key = ?", (course_key,))

    if not course_info:
        return {"ok": False, "answer": f"المقرر {course_key} غير موجود في قاعدة البيانات."}

    prereqs_raw = course_info.get("prerequisites")
    if not prereqs_raw or str(prereqs_raw).strip() == "":
        return {"ok": True, "eligible": True, "answer": f"المقرر {course_key} ليس له متطلبات سابقة. يمكن للطالبة تنزيله."}

    # تبسيط فحص المتطلبات (افتراض أن المتطلبات مفصولة بفاصلة)
    prereqs = [p.strip() for p in str(prereqs_raw).split(",") if p.strip()]
    passed_courses = {sc["course_key"] for sc in student_courses if sc["status"] in ['P', 'معادلة']}

    missing_prereqs = [p for p in prereqs if p not in passed_courses]

    if missing_prereqs:
        return {
            "ok": True,
            "eligible": False,
            "missing": missing_prereqs,
            "answer": f"لا يمكن اقتراح المادة {course_key} للطالبة لأنها لم تجتز المتطلبات السابقة: {', '.join(missing_prereqs)}."
        }

    return {
        "ok": True,
        "eligible": True,
        "answer": f"الطالبة استوفت المتطلبات السابقة ({', '.join(prereqs)}). يمكن اقتراح المادة {course_key} لها."
    }

# ------------------------------------------------------------
# 2. COURSE REGISTRATION SUGGESTION
# ------------------------------------------------------------
def suggest_student_registration(student_id: str) -> Dict[str, Any]:
    """يحلل حالة الطالبة ويقترح المواد المسموح بتنزيلها بناءً على الخطة والمتطلبات"""
    student_info = get_student_basic_info(student_id)
    if not student_info:
        return {"ok": False, "answer": "الطالبة غير موجودة."}

    summary = get_student_course_completion_summary(student_id)
    remaining = summary.get("remaining_courses", [])

    suggestions = []
    warnings = []

    if student_info.get("graduating_this_year") == 'Y':
        warnings.append("تنبيه: الطالبة متوقع تخرجها، يجب إعطاؤها أولوية في المواد المتبقية.")

    for rc in remaining:
        c_key = rc.get("course_key")
        check = check_course_prerequisites(student_id, c_key)
        if check.get("eligible"):
            suggestions.append(c_key)

    answer = f"بناءً على تحليل خطة الطالبة {student_id}، نقترح المواد التالية للتنزيل: {', '.join(suggestions)}."
    if warnings:
        answer += "\n" + "\n".join(warnings)

    return {
        "ok": True,
        "student_id": student_id,
        "suggestions": suggestions,
        "warnings": warnings,
        "answer": answer
    }

# ------------------------------------------------------------
# 3. SECTION OPENING & CAPACITY ANALYSIS
# ------------------------------------------------------------
def analyze_section_needs(course_key: str, capacity_per_section: int = 25) -> Dict[str, Any]:
    """يقترح عدد الشعب بناءً على عدد الطالبات المحتاجات (خاصة الخريجات)"""
    need_count = exceptional_opening("count_course", course_key=course_key)

    if isinstance(need_count, dict): # Handle error/unexpected return
        need_count = need_count.get("count", 0)

    if need_count == 0:
        return {"ok": True, "answer": f"لا يوجد طالبات خريجات محتاجات للمقرر {course_key} حالياً."}

    sections_needed = math.ceil(need_count / capacity_per_section)

    distribution = []
    remaining_students = need_count
    for i in range(sections_needed):
        alloc = min(capacity_per_section, remaining_students)
        distribution.append(f"شعبة {i+1}: {alloc} طالبة")
        remaining_students -= alloc

    answer = (
        f"المادة {course_key} تحتاج فتح استثنائي.\n"
        f"عدد الطالبات المحتاجات (متوقع تخرجهن): {need_count}\n"
        f"المقترح: فتح {sections_needed} شعبة.\n"
        f"التوزيع المقترح:\n- " + "\n- ".join(distribution)
    )

    return {
        "ok": True,
        "course_key": course_key,
        "total_needed": need_count,
        "suggested_sections": sections_needed,
        "answer": answer
    }

print(" Cell 700: Course Registration Advising Layer loaded.")

 Cell 700: Course Registration Advising Layer loaded.


In [ ]:
import os
import re
import sqlite3
from typing import Any, Dict, List, Optional, Tuple


# ============================================================
# SQLITE STUDENT DATA LAYER (UNIFIED, SAFE, PRODUCTION-READY) cell 7
# ============================================================


# ------------------------------------------------------------
# DB VALIDATION
# ------------------------------------------------------------

REQUIRED_TABLES = {
    "students",
    "courses",
    "student_courses",
    "risk_features",
    "support_features",
}


def _db_file_exists() -> bool:
    return os.path.exists(DB_PATH)


def _raw_get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def _list_tables_raw() -> List[str]:
    if not _db_file_exists():
        return []

    try:
        with _raw_get_connection() as conn:
            cur = conn.cursor()
            cur.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                  AND name NOT LIKE 'sqlite_%'
                ORDER BY name
            """)
            rows = cur.fetchall()
            return [r["name"] for r in rows]
    except sqlite3.Error:
        return []


def validate_database_or_raise() -> None:
    if not _db_file_exists():
        raise FileNotFoundError(f"Database file not found: {DB_PATH}")

    existing_tables = set(_list_tables_raw())
    missing = REQUIRED_TABLES - existing_tables

    if missing:
        raise RuntimeError(
            "Database is connected, but required tables are missing. "
            f"Missing: {sorted(missing)} | Existing: {sorted(existing_tables)}"
        )


# ------------------------------------------------------------
# CONNECTION
# ------------------------------------------------------------

def get_db_connection() -> sqlite3.Connection:
    validate_database_or_raise()
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


# ------------------------------------------------------------
# ROW HELPERS
# ------------------------------------------------------------

def row_to_dict(row: Optional[sqlite3.Row]) -> Optional[Dict[str, Any]]:
    return dict(row) if row is not None else None


def rows_to_dicts(rows: List[sqlite3.Row]) -> List[Dict[str, Any]]:
    return [dict(r) for r in rows] if rows else []


# ------------------------------------------------------------
# CORE QUERY HELPERS
# ------------------------------------------------------------

def fetch_one(query: str, params: Tuple = ()) -> Optional[Dict[str, Any]]:
    try:
        with get_db_connection() as conn:
            cur = conn.cursor()
            cur.execute(query, params)
            row = cur.fetchone()
            return row_to_dict(row)
    except Exception as e:
        print(f"SQLite error in fetch_one: {e}")
        return None


def fetch_all(query: str, params: Tuple = ()) -> List[Dict[str, Any]]:
    try:
        with get_db_connection() as conn:
            cur = conn.cursor()
            cur.execute(query, params)
            rows = cur.fetchall()
            return rows_to_dicts(rows)
    except Exception as e:
        print(f"SQLite error in fetch_all: {e}")
        return []


def run_read_query(query: str, params: Tuple = ()) -> List[Dict[str, Any]]:
    """
    Execute a safe read-only SELECT query.
    """
    q = (query or "").strip().lower()

    if not q.startswith("select"):
        print("Blocked query: only SELECT statements are allowed.")
        return []

    blocked_keywords = [
        "insert",
        "update",
        "delete",
        "drop",
        "alter",
        "create",
        "replace",
        "attach",
        "detach",
        "pragma",
    ]

    if any(word in q for word in blocked_keywords):
        print("Blocked query: contains unsafe SQL keyword.")
        return []

    return fetch_all(query, params)


def _one_value(sql: str, params: tuple = (), default=0):
    row = fetch_one(sql, params)
    if not row:
        return default
    return list(row.values())[0]


def _rows(sql: str, params: tuple = ()):
    rows = fetch_all(sql, params)
    return rows or []


# ------------------------------------------------------------
# STATUS HELPERS
# ------------------------------------------------------------

# UPDATED: backlog logic fixed (failed + risk)
def _is_failed_status_expr(alias: str = "sc") -> str:
    """
    Backlog failure logic ONLY.

    Backlog failure includes:
    - N
    - any status starting with N, e.g. N%, N مواضيع

    Does NOT include:
    - R
    - NULL
    - empty

    IMPORTANT:
    Remaining courses and backlog are not the same concept.
    """
    return f"""(
        TRIM({alias}.status) = 'N'
        OR TRIM({alias}.status) LIKE 'N%'
    )"""


# UPDATED: backlog logic fixed (failed + risk)
def _is_high_risk_expr(alias: str = "rf") -> str:
    """
    Academic Risk Model.

    High risk if any core academic risk feature is below 50.
    """
    return f"""(
        {alias}.attendance < 50
        OR {alias}.midterm < 50
        OR {alias}.assignments < 50
        OR {alias}.quizzes < 50
    )"""


# UPDATED: backlog logic fixed (failed + risk)
def _is_backlog_or_high_risk_expr(sc_alias: str = "sc", rf_alias: str = "rf") -> str:
    """
    Student is BACKLOG / STRUGGLING only if:
    - failed course status N / N%
    OR
    - high academic risk.
    """
    return f"""(
        {_is_failed_status_expr(sc_alias)}
        OR {_is_high_risk_expr(rf_alias)}
    )"""


def _is_not_passed_status_expr(alias: str = "sc") -> str:
    """
    Remaining / unfinished course logic.

    Remaining includes:
    - NULL
    - empty status
    - R
    - N
    - any status starting with N, e.g. 'N مواضيع'

    IMPORTANT:
    This is REMAINING COURSES logic, not backlog-only logic.
    """
    return f"""(
        {alias}.status IS NULL
        OR TRIM({alias}.status) = ''
        OR TRIM({alias}.status) = 'R'
        OR TRIM({alias}.status) = 'N'
        OR TRIM({alias}.status) LIKE 'N%'
    )"""


def _is_completed_status_expr(alias: str = "sc") -> str:
    """
    Completed course logic.

    Completed includes:
    - P
    - معادلة
    """
    return f"""(
        TRIM({alias}.status) = 'P'
        OR TRIM({alias}.status) = 'معادلة'
    )"""


def _is_unknown_status_expr(alias: str = "sc") -> str:
    """
    Unknown course statuses that should not drive main backlog/completion logic.
    """
    return f"""(
        {alias}.status IS NOT NULL
        AND TRIM({alias}.status) != ''
        AND TRIM({alias}.status) NOT IN ('P', 'معادلة', 'R', 'N')
        AND TRIM({alias}.status) NOT LIKE 'N%'
    )"""


def _needs_training_expr(alias: str = "s") -> str:
    """
    Training logic based on students.cohort_rank.

    Needs training:
    - NULL
    - empty
    - NP
    """
    return f"""(
        {alias}.cohort_rank IS NULL
        OR TRIM({alias}.cohort_rank) = ''
        OR TRIM({alias}.cohort_rank) = 'NP'
    )"""


def _training_completed_expr(alias: str = "s") -> str:
    """
    Completed training:
    - IP
    """
    return f"""(
        TRIM({alias}.cohort_rank) = 'IP'
    )"""


def _training_in_progress_expr(alias: str = "s") -> str:
    """
    No in-progress training status exists in this dataset.
    IP means completed, not in progress.
    """
    return "(1 = 0)"


def _gq_norm(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    value = str(value).strip()
    return value or None


def a(name: Optional[str]) -> Optional[str]:
    n = _gq_norm(name)

    if not n:
        return None

    patterns = [
        r"^\s*د\.\s*",
        r"^\s*دكتور\s+",
        r"^\s*دكتورة\s+",
        r"^\s*الدكتور\s+",
        r"^\s*الدكتورة\s+",
        r"^\s*الدكتوره\s+",
    ]

    for p in patterns:
        n = re.sub(p, "", n, flags=re.IGNORECASE).strip()

    return n or None


# ------------------------------------------------------------
# RESULT HELPERS
# ------------------------------------------------------------

def _count_students_result(
    rows: List[Dict[str, Any]],
    advisor_name: Optional[str] = None,
    answer: Optional[str] = None,
) -> Dict[str, Any]:
    students = []
    seen = set()

    for row in rows or []:
        sid = row.get("student_id")
        if sid in seen:
            continue
        seen.add(sid)
        students.append({
            "student_id": sid,
            "student_name": row.get("student_name"),
            "advisor": row.get("advisor"),
        })

    count = len(students)

    return {
        "ok": True,
        "count": count,
        "students": students,
        "advisor": advisor_name,
        "answer": answer or f"عدد الطالبات هو {count} مع عرض الأسماء",
    }


def _students_base_select(extra_cols: str = "") -> str:
    return f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor
            {extra_cols}
        FROM students s
    """


def _advisor_filter_sql(advisor_name: Optional[str], params: List[Any], alias: str = "s") -> str:
    if advisor_name:
        params.append(advisor_name)
        return f" AND {alias}.advisor = ? "
    return ""


# ------------------------------------------------------------
# SCHEMA HELPERS
# ------------------------------------------------------------

def table_exists(table_name: str) -> bool:
    row = fetch_one(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table' AND name = ?
        """,
        (table_name,)
    )
    return row is not None


def get_table_columns(table_name: str) -> List[str]:
    try:
        with get_db_connection() as conn:
            cur = conn.cursor()
            cur.execute(f"PRAGMA table_info({table_name})")
            rows = cur.fetchall()
            return [r["name"] for r in rows]
    except Exception as e:
        print(f"SQLite error in get_table_columns: {e}")
        return []


def get_all_tables_safe() -> List[str]:
    try:
        validate_database_or_raise()
        return _list_tables_raw()
    except Exception as e:
        print(f"Database validation error: {e}")
        return []


# ------------------------------------------------------------
# STUDENT LOOKUPS
# ------------------------------------------------------------

def student_exists(student_id: str) -> bool:
    row = fetch_one(
        """
        SELECT student_id
        FROM students
        WHERE student_id = ?
        """,
        (student_id,)
    )
    return row is not None


def get_student_basic_info(student_id: str) -> Optional[Dict[str, Any]]:
    return fetch_one(
        """
        SELECT
            student_id,
            student_name,
            serial,
            advisor,
            cohort_rank,
            hours_completed,
            notes,
            graduating_this_year,
            has_medical_condition,
            needs_accommodation,
            is_synthetic,
            remaining_hours
        FROM students
        WHERE student_id = ?
        """,
        (student_id,)
    )


def get_student_courses(student_id: str) -> List[Dict[str, Any]]:
    return fetch_all(
        """
        SELECT
            student_id,
            course_key,
            course_label,
            status,
            is_synthetic
        FROM student_courses
        WHERE student_id = ?
        """,
        (student_id,)
    )


def get_student_risk_features(student_id: str) -> Optional[Dict[str, Any]]:
    return fetch_one(
        """
        SELECT
            student_id,
            attendance,
            participation,
            quizzes,
            assignments,
            midterm,
            projects
        FROM risk_features
        WHERE student_id = ?
        """,
        (student_id,)
    )


def get_student_support_features(student_id: str) -> Optional[Dict[str, Any]]:
    return fetch_one(
        """
        SELECT
            student_id,
            study_hours_per_week,
            social_media_usage_hours_per_day,
            sleep_duration_hours_per_night,
            physical_exercise_hours_per_week,
            family_support,
            financial_stress,
            peer_pressure,
            mental_stress_level,
            diet_quality,
            cognitive_distortions
        FROM support_features
        WHERE student_id = ?
        """,
        (student_id,)
    )


def load_student_profile(student_id: str) -> Dict[str, Any]:
    student_info = get_student_basic_info(student_id)
    risk_features_db = get_student_risk_features(student_id)
    support_features_db = get_student_support_features(student_id)
    courses = get_student_courses(student_id)

    return {
        "ok": True,
        "student_id": student_id,
        "found": student_info is not None,
        "student_info": student_info,
        "risk_features": risk_features_db,
        "support_features": support_features_db,
        "courses": courses,
    }

# ------------------------------------------------------------
# STUDENT COUNTS
# ------------------------------------------------------------

def student_count(metric: str, advisor_name: Optional[str] = None) -> Dict[str, Any]:
    params = []
    advisor_sql = _advisor_filter_sql(advisor_name, params, "s")

    if metric == "all_students":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE 1=1 {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "expected_graduates":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE s.graduating_this_year = 'Y'
            {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "medical":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE s.has_medical_condition = 1
            {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "accommodation":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE s.needs_accommodation = 1
            {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "backlog_students":
        return backlog_students_with_names(advisor_name)

    if metric == "training_needed":
        return training_needed_students_with_names(advisor_name)

    if metric == "training_completed":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE {_training_completed_expr("s")}
            {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "training_in_progress":
        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE {_training_in_progress_expr("s")}
            {advisor_sql}
            ORDER BY s.student_name
        """, tuple(params))
        return _count_students_result(rows, advisor_name)

    if metric == "by_advisor":
        if not advisor_name:
            return _count_students_result([], advisor_name)

        rows = _rows("""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            WHERE s.advisor = ?
            ORDER BY s.student_name
        """, (advisor_name,))
        return _count_students_result(rows, advisor_name)

    raise ValueError(f"Unsupported student_count metric: {metric}")


# ------------------------------------------------------------
# COURSE FUNCTIONS
# ------------------------------------------------------------

def get_course_enrollment_count(course_key: str) -> Dict[str, Any]:
    rows = _rows("""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor
        FROM student_courses sc
        JOIN students s
          ON s.student_id = sc.student_id
        WHERE sc.course_key = ?
        ORDER BY s.student_name
    """, (course_key,))

    return _count_students_result(
        rows,
        answer=f"عدد الطالبات المسجلات في المقرر {course_key} هو {len(rows)} مع عرض الأسماء"
    )


def students_not_passed_course(course_key: str) -> Dict[str, Any]:
    rows = _rows(f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        JOIN students s
          ON s.student_id = sc.student_id
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.course_key = ?
          AND {_is_not_passed_status_expr("sc")}
        ORDER BY s.student_name
    """, (course_key,))

    return {
        "ok": True,
        "course_key": course_key,
        "count": len(rows),
        "students": rows,
        "advisor": None,
        "answer": f"عدد الطالبات اللاتي لم يجتزن/لم ينهين المقرر {course_key} هو {len(rows)} مع عرض الأسماء."
    }


def course_lookup(course_key: str) -> Dict[str, Any]:
    row = fetch_one("""
        SELECT course_key, course_label, credits, prerequisites
        FROM courses
        WHERE course_key = ?
    """, (course_key,))

    if not row:
        return {
            "ok": False,
            "course_key": course_key,
            "answer": "لم يتم العثور على المقرر."
        }

    return {
        "ok": True,
        "course": row,
        "answer": f"{row.get('course_key')} - {row.get('course_label')}"
    }


def get_student_course_completion_summary(student_id: str) -> Dict[str, Any]:
    completed = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_completed_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    remaining = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_not_passed_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    failed = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_failed_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    unknown = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_unknown_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    return {
        "ok": True,
        "student_id": student_id,
        "completed_courses": completed,
        "remaining_courses": remaining,
        "failed_courses": failed,
        "unknown_courses": unknown,
        "completed_count": len(completed),
        "remaining_count": len(remaining),
        "failed_count": len(failed),
        "unknown_count": len(unknown),
        "answer": (
            f"عدد المواد المجتازة/المعادلة: {len(completed)}، "
            f"وعدد المواد المتبقية: {len(remaining)}، "
            f"وعدد مواد التعثر الفعلية: {len(failed)}."
        )
    }


# ------------------------------------------------------------
# TRAINING FUNCTIONS
# ------------------------------------------------------------

def training_students(
    status: str,
    advisor_name: Optional[str] = None,
    mode: str = "count",
):
    params = []
    advisor_sql = _advisor_filter_sql(advisor_name, params, "s")

    if status in {
        "needs_training",
        "needed",
        "not_passed",
        "not_taken",
        "np",
        "missing",
    }:
        condition_sql = _needs_training_expr("s")

    elif status in {
        "passed",
        "completed",
        "ip",
    }:
        condition_sql = _training_completed_expr("s")

    elif status in {
        "in_progress",
    }:
        condition_sql = _training_in_progress_expr("s")

    else:
        raise ValueError(f"Unsupported training status: {status}")

    rows = _rows(f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor,
            s.cohort_rank
        FROM students s
        WHERE {condition_sql}
        {advisor_sql}
        ORDER BY s.student_name
    """, tuple(params))

    if mode == "count":
        return _count_students_result(rows, advisor_name)

    return rows

def get_student_training_status(student_id: str) -> Dict[str, Any]:
    row = fetch_one("""
        SELECT
            student_id,
            student_name,
            advisor,
            cohort_rank
        FROM students
        WHERE student_id = ?
        LIMIT 1
    """, (student_id,))

    if not row:
        return {
            "ok": False,
            "student_id": student_id,
            "answer": "لم يتم العثور على الطالبة."
        }

    cohort_rank = row.get("cohort_rank")
    normalized = str(cohort_rank).strip() if cohort_rank is not None else ""

    if normalized == "IP":
        training_status = "completed"
        answer = "الطالبة أنهت التدريب."
    elif normalized == "NP":
        training_status = "needs_training"
        answer = "الطالبة لم تجتز التدريب وتحتاج إلى تدريب."
    elif normalized == "":
        training_status = "needs_training"
        answer = "الطالبة لم تأخذ التدريب بعد وتحتاج إلى تدريب."
    else:
        training_status = "unknown"
        answer = "حالة التدريب غير واضحة."

    return {
        "ok": True,
        "student_id": student_id,
        "training_status": training_status,
        "answer": answer,
        "data": row,
    }
# ------------------------------------------------------------
# NEW FUNCTIONS
# ------------------------------------------------------------

# NEW: added backlog_students_with_names
# UPDATED: backlog logic fixed (failed + risk)
def backlog_students_with_names(advisor_name: Optional[str] = None) -> Dict[str, Any]:
    params = []
    advisor_sql = _advisor_filter_sql(advisor_name, params, "s")

    rows = _rows(f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor
        FROM students s
        LEFT JOIN student_courses sc
          ON sc.student_id = s.student_id
        LEFT JOIN risk_features rf
          ON rf.student_id = s.student_id
        WHERE {_is_backlog_or_high_risk_expr("sc", "rf")}
        {advisor_sql}
        ORDER BY s.student_name
    """, tuple(params))

    return _count_students_result(
        rows,
        advisor_name,
        f"عدد الطالبات المتعثرات أو عاليّات الخطورة هو {len(rows)} مع عرض الأسماء"
    )


# NEW: added training_needed_students_with_names
def training_needed_students_with_names(advisor_name: Optional[str] = None) -> Dict[str, Any]:
    params = []
    advisor_sql = _advisor_filter_sql(advisor_name, params, "s")

    rows = _rows(f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor
        FROM students s
        WHERE {_needs_training_expr("s")}
        {advisor_sql}
        ORDER BY s.student_name
    """, tuple(params))

    return _count_students_result(
        rows,
        advisor_name,
        f"عدد الطالبات المحتاجات للتدريب هو {len(rows)} مع عرض الأسماء"
    )


# NEW: added students_with_advisor_summary
# UPDATED: backlog logic fixed (failed + risk)
def students_with_advisor_summary(advisor_name: str) -> Dict[str, Any]:
    students = _rows("""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor,
            s.cohort_rank,
            s.graduating_this_year,
            s.hours_completed,
            s.remaining_hours
        FROM students s
        WHERE s.advisor = ?
        ORDER BY s.student_name
    """, (advisor_name,))

    backlog = backlog_students_with_names(advisor_name)
    training_needed = training_needed_students_with_names(advisor_name)

    expected_graduates = _rows("""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor
        FROM students s
        WHERE s.advisor = ?
          AND s.graduating_this_year = 'Y'
        ORDER BY s.student_name
    """, (advisor_name,))

    return {
        "ok": True,
        "advisor": advisor_name,
        "count": len(students),
        "students": students,
        "summary": {
            "total_students": len(students),
            "backlog_or_high_risk_students": backlog["count"],
            "training_needed_students": training_needed["count"],
            "expected_graduates": len(expected_graduates),
        },
        "backlog_or_high_risk_students": backlog["students"],
        "training_needed_students": training_needed["students"],
        "expected_graduates": expected_graduates,
        "answer": f"عدد طالبات المرشدة {advisor_name} هو {len(students)} مع عرض الأسماء والملخص."
    }


# ------------------------------------------------------------
# ADVISOR LOGIC
# ------------------------------------------------------------

def advisor_students(
    advisor_name: str,
    filter: str = "all",
    limit: Optional[int] = None,
):
    params = [advisor_name]
    filter_sql = ""

    if filter == "expected_graduates":
        filter_sql = " AND s.graduating_this_year = 'Y' "

    elif filter == "backlog":
        # UPDATED: backlog logic fixed (failed + risk)
        filter_sql = f"""
            AND (
                EXISTS (
                    SELECT 1
                    FROM student_courses sc
                    WHERE sc.student_id = s.student_id
                      AND {_is_failed_status_expr("sc")}
                )
                OR EXISTS (
                    SELECT 1
                    FROM risk_features rf
                    WHERE rf.student_id = s.student_id
                      AND {_is_high_risk_expr("rf")}
                )
            )
        """

    elif filter in {"training_needed", "needs_training"}:
        filter_sql = f" AND {_needs_training_expr('s')} "

    elif filter in {"training_completed", "completed", "ip"}:
        filter_sql = f" AND {_training_completed_expr('s')} "

    elif filter in {"training_in_progress", "in_progress"}:
        filter_sql = f" AND {_training_in_progress_expr('s')} "

    elif filter != "all":
        raise ValueError(f"Unsupported advisor_students filter: {filter}")

    limit_sql = ""
    if limit:
        limit_sql = " LIMIT ? "
        params.append(int(limit))

    rows = _rows(f"""
        SELECT DISTINCT
            s.student_id,
            s.student_name,
            s.advisor,
            s.cohort_rank,
            s.graduating_this_year,
            s.hours_completed,
            s.remaining_hours
        FROM students s
        WHERE s.advisor = ?
        {filter_sql}
        ORDER BY s.student_name
        {limit_sql}
    """, tuple(params))

    return rows


def advisor_ranking(metric: str, limit: Optional[int] = None):
    limit = int(limit or 10)

    if metric == "student_count":
        return _rows("""
            SELECT advisor, COUNT(DISTINCT student_id) AS student_count
            FROM students
            WHERE advisor IS NOT NULL AND TRIM(advisor) != ''
            GROUP BY advisor
            ORDER BY student_count DESC
            LIMIT ?
        """, (limit,))

    if metric == "expected_graduates":
        return _rows("""
            SELECT advisor, COUNT(DISTINCT student_id) AS expected_graduates_count
            FROM students
            WHERE graduating_this_year = 'Y'
              AND advisor IS NOT NULL
              AND TRIM(advisor) != ''
            GROUP BY advisor
            ORDER BY expected_graduates_count DESC
            LIMIT ?
        """, (limit,))

    if metric == "backlog_students":
        # UPDATED: backlog logic fixed (failed + risk)
        return _rows(f"""
            SELECT
                s.advisor,
                COUNT(DISTINCT s.student_id) AS backlog_student_count
            FROM students s
            LEFT JOIN student_courses sc
              ON sc.student_id = s.student_id
            LEFT JOIN risk_features rf
              ON rf.student_id = s.student_id
            WHERE s.advisor IS NOT NULL
              AND TRIM(s.advisor) != ''
              AND {_is_backlog_or_high_risk_expr("sc", "rf")}
            GROUP BY s.advisor
            ORDER BY backlog_student_count DESC
            LIMIT ?
        """, (limit,))

    if metric == "training_needed":
        return _rows(f"""
            SELECT
                s.advisor,
                COUNT(DISTINCT s.student_id) AS training_needed_count
            FROM students s
            WHERE s.advisor IS NOT NULL
              AND TRIM(s.advisor) != ''
              AND {_needs_training_expr("s")}
            GROUP BY s.advisor
            ORDER BY training_needed_count DESC
            LIMIT ?
        """, (limit,))

    if metric == "training_completed":
        return _rows(f"""
            SELECT
                s.advisor,
                COUNT(DISTINCT s.student_id) AS training_completed_count
            FROM students s
            WHERE s.advisor IS NOT NULL
              AND TRIM(s.advisor) != ''
              AND {_training_completed_expr("s")}
            GROUP BY s.advisor
            ORDER BY training_completed_count DESC
            LIMIT ?
        """, (limit,))

    raise ValueError(f"Unsupported advisor_ranking metric: {metric}")


def students_count_by_advisor():
    rows = _rows("""
        SELECT
            advisor,
            COUNT(DISTINCT student_id) AS total_students
        FROM students
        WHERE advisor IS NOT NULL
          AND TRIM(advisor) != ''
        GROUP BY advisor
        ORDER BY total_students DESC, advisor
    """)
    return rows


def backlog_students_count_by_advisor():
    # UPDATED: backlog logic fixed (failed + risk)
    return _rows(f"""
        SELECT
            s.advisor,
            COUNT(DISTINCT s.student_id) AS backlog_students
        FROM students s
        LEFT JOIN student_courses sc
          ON sc.student_id = s.student_id
        LEFT JOIN risk_features rf
          ON rf.student_id = s.student_id
        WHERE s.advisor IS NOT NULL
          AND TRIM(s.advisor) != ''
          AND {_is_backlog_or_high_risk_expr("sc", "rf")}
        GROUP BY s.advisor
        ORDER BY backlog_students DESC, s.advisor
    """)


def training_needed_count_by_advisor():
    return _rows(f"""
        SELECT
            s.advisor,
            COUNT(DISTINCT s.student_id) AS training_needed_students
        FROM students s
        WHERE s.advisor IS NOT NULL
          AND TRIM(s.advisor) != ''
          AND {_needs_training_expr("s")}
        GROUP BY s.advisor
        ORDER BY training_needed_students DESC, s.advisor
    """)


def training_completed_count_by_advisor():
    return _rows(f"""
        SELECT
            s.advisor,
            COUNT(DISTINCT s.student_id) AS training_completed_students
        FROM students s
        WHERE s.advisor IS NOT NULL
          AND TRIM(s.advisor) != ''
          AND {_training_completed_expr("s")}
        GROUP BY s.advisor
        ORDER BY training_completed_students DESC, s.advisor
    """)


def training_in_progress_count_by_advisor():
    return _rows(f"""
        SELECT
            s.advisor,
            COUNT(DISTINCT s.student_id) AS training_in_progress_students
        FROM students s
        WHERE s.advisor IS NOT NULL
          AND TRIM(s.advisor) != ''
          AND {_training_in_progress_expr("s")}
        GROUP BY s.advisor
        ORDER BY training_in_progress_students DESC, s.advisor
    """)


def advisor_full_summary():
    # UPDATED: backlog logic fixed (failed + risk)
    return _rows(f"""
        SELECT
            s.advisor,

            COUNT(DISTINCT s.student_id) AS total_students,

            COUNT(DISTINCT CASE
                WHEN {_is_backlog_or_high_risk_expr("sc", "rf")}
                THEN s.student_id
            END) AS backlog_students,

            COUNT(DISTINCT CASE
                WHEN {_needs_training_expr("s")}
                THEN s.student_id
            END) AS training_needed_students,

            COUNT(DISTINCT CASE
                WHEN {_training_completed_expr("s")}
                THEN s.student_id
            END) AS training_completed_students,

            COUNT(DISTINCT CASE
                WHEN {_training_in_progress_expr("s")}
                THEN s.student_id
            END) AS training_in_progress_students,

            COUNT(DISTINCT CASE
                WHEN s.graduating_this_year = 'Y'
                THEN s.student_id
            END) AS expected_graduates

        FROM students s
        LEFT JOIN student_courses sc
          ON sc.student_id = s.student_id
        LEFT JOIN risk_features rf
          ON rf.student_id = s.student_id

        WHERE s.advisor IS NOT NULL
          AND TRIM(s.advisor) != ''

        GROUP BY s.advisor
        ORDER BY total_students DESC, s.advisor
    """)


# ------------------------------------------------------------
# ACADEMIC CASE
# ------------------------------------------------------------

def student_academic_case(student_id: str, case_type: str = "backlog") -> Dict[str, Any]:
    if case_type not in {"backlog", "delay", "graduation_risk"}:
        raise ValueError(f"Unsupported case_type: {case_type}")

    failed = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_failed_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    unfinished = _rows(f"""
        SELECT
            sc.student_id,
            sc.course_key,
            COALESCE(c.course_label, sc.course_label) AS course_label,
            sc.status
        FROM student_courses sc
        LEFT JOIN courses c
          ON c.course_key = sc.course_key
        WHERE sc.student_id = ?
          AND {_is_not_passed_status_expr("sc")}
        ORDER BY sc.course_key
    """, (student_id,))

    risk_row = fetch_one(f"""
        SELECT
            rf.student_id,
            rf.attendance,
            rf.midterm,
            rf.assignments,
            rf.quizzes,
            rf.participation,
            rf.projects
        FROM risk_features rf
        WHERE rf.student_id = ?
          AND {_is_high_risk_expr("rf")}
        LIMIT 1
    """, (student_id,))

    has_backlog = len(failed) > 0 or risk_row is not None

    return {
        "ok": True,
        "student_id": student_id,
        "answer": (
            "نعم، لدى الطالبة تعثر/خطورة أكاديمية."
            if has_backlog
            else "لا، لا تظهر مؤشرات تعثر أو خطورة أكاديمية للطالبة."
        ),
        "data": {
            "student_id": student_id,
            "has_backlog": has_backlog,
            "failed_courses": failed,
            "unfinished_courses": unfinished,
            "high_risk_features": risk_row,
        },
    }
# ------------------------------------------------------------
# FACTOR ANALYSIS PLACEHOLDER
# ------------------------------------------------------------

def student_factor_analysis(
    analysis_type: str,
    advisor_name: Optional[str] = None,
):
    if analysis_type == "backlog_factors":
        # UPDATED: backlog logic fixed (failed + risk)
        params = []
        advisor_sql = _advisor_filter_sql(advisor_name, params, "s")

        return _rows(f"""
            SELECT
                s.advisor,
                COUNT(DISTINCT s.student_id) AS backlog_student_count,
                AVG(s.hours_completed) AS avg_hours_completed,
                AVG(s.remaining_hours) AS avg_remaining_hours,
                AVG(rf.attendance) AS avg_attendance,
                AVG(rf.midterm) AS avg_midterm,
                AVG(rf.assignments) AS avg_assignments,
                AVG(rf.quizzes) AS avg_quizzes
            FROM students s
            LEFT JOIN student_courses sc
              ON sc.student_id = s.student_id
            LEFT JOIN risk_features rf
              ON rf.student_id = s.student_id
            WHERE {_is_backlog_or_high_risk_expr("sc", "rf")}
            {advisor_sql}
            GROUP BY s.advisor
            ORDER BY backlog_student_count DESC
        """, tuple(params))

    raise ValueError(f"Unsupported analysis_type: {analysis_type}")


# ------------------------------------------------------------
# EXCEPTIONAL OPENING
# ------------------------------------------------------------

def exceptional_opening(
    mode: str,
    course_key: Optional[str] = None,
    min_students: Optional[int] = None,
):
    min_students = int(min_students or 5)

    if mode == "check_course":
        if not course_key:
            raise ValueError("course_key is required for check_course")

        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            JOIN student_courses sc
              ON sc.student_id = s.student_id
            WHERE s.graduating_this_year = 'Y'
              AND sc.course_key = ?
              AND {_is_not_passed_status_expr("sc")}
            ORDER BY s.student_name
        """, (course_key,))

        count = len(rows)

        return {
            "ok": True,
            "course_key": course_key,
            "count": count,
            "students": rows,
            "advisor": None,
            "should_open": count >= min_students,
            "answer": (
                f"نعم، المقرر {course_key} يحتاج فتحًا استثنائيًا؛ عدد المحتاجات له {count} مع عرض الأسماء."
                if count >= min_students
                else f"لا، المقرر {course_key} لا يحقق شرط الفتح الاستثنائي؛ عدد المحتاجات له {count} مع عرض الأسماء."
            )
        }

    if mode == "count_course":
        if not course_key:
            raise ValueError("course_key is required for count_course")

        rows = _rows(f"""
            SELECT DISTINCT
                s.student_id,
                s.student_name,
                s.advisor
            FROM students s
            JOIN student_courses sc
              ON sc.student_id = s.student_id
            WHERE s.graduating_this_year = 'Y'
              AND sc.course_key = ?
              AND {_is_not_passed_status_expr("sc")}
            ORDER BY s.student_name
        """, (course_key,))

        return _count_students_result(
            rows,
            answer=f"عدد الطالبات المتوقع تخرجهن والمحتجات للمقرر {course_key} هو {len(rows)} مع عرض الأسماء"
        )

    if mode == "list":
        return _rows(f"""
            SELECT
                sc.course_key,
                COALESCE(c.course_label, sc.course_label) AS course_label,
                COUNT(DISTINCT s.student_id) AS expected_graduate_need_count
            FROM students s
            JOIN student_courses sc
              ON sc.student_id = s.student_id
            LEFT JOIN courses c
              ON c.course_key = sc.course_key
            WHERE s.graduating_this_year = 'Y'
              AND {_is_not_passed_status_expr("sc")}
            GROUP BY sc.course_key, COALESCE(c.course_label, sc.course_label)
            HAVING expected_graduate_need_count >= ?
            ORDER BY expected_graduate_need_count DESC
        """, (min_students,))

    raise ValueError(f"Unsupported exceptional_opening mode: {mode}")


# ------------------------------------------------------------
# DEBUG / HEALTH CHECK
# ------------------------------------------------------------

def check_database_health() -> Dict[str, Any]:
    result = {
        "db_path": DB_PATH,
        "file_exists": _db_file_exists(),
        "tables": _list_tables_raw() if _db_file_exists() else [],
        "required_tables": sorted(REQUIRED_TABLES),
        "ok": False,
        "missing_tables": [],
    }

    try:
        validate_database_or_raise()
        result["ok"] = True
    except Exception as e:
        existing = set(result["tables"])
        result["missing_tables"] = sorted(REQUIRED_TABLES - existing)
        result["error"] = str(e)

    return result


print(" SQL FUNCTIONS LAYER loaded successfully.")

 SQL FUNCTIONS LAYER loaded successfully.


In [ ]:
# ============================================================
# CELL 8: UNIFIED STUDENT CONTEXT + MODEL INPUT LAYER
# ============================================================

from typing import Dict, Any, Optional


# ------------------------------------------------------------
# SAFE OPTIONAL DEPENDENCY HELPERS
# ------------------------------------------------------------
def _safe_call_dict_function(function_name: str, *args, **kwargs) -> Dict[str, Any]:
    fn = globals().get(function_name)
    if callable(fn):
        try:
            result = fn(*args, **kwargs)
            return result if isinstance(result, dict) else {}
        except Exception:
            return {}
    return {}


def _safe_call_any_function(function_name: str, *args, **kwargs):
    fn = globals().get(function_name)
    if callable(fn):
        try:
            return fn(*args, **kwargs)
        except Exception:
            return None
    return None


def _safe_extract_risk_features_from_query(text: str) -> Dict[str, Any]:
    return _safe_call_dict_function("extract_risk_features_from_query", text)


def _safe_extract_support_features_from_query(text: str) -> Dict[str, Any]:
    return _safe_call_dict_function("extract_support_features_from_query", text)


def _safe_count_features(function_name: str, features: Dict[str, Any]) -> int:
    fn = globals().get(function_name)
    if callable(fn):
        try:
            return int(fn(features))
        except Exception:
            pass

    return sum(
        1
        for key, value in (features or {}).items()
        if key != "student_id" and value is not None
    )


def _safe_count_risk_features(features: Dict[str, Any]) -> int:
    return _safe_count_features("count_risk_features", features)


def _safe_count_support_features(features: Dict[str, Any]) -> int:
    return _safe_count_features("count_support_features", features)


# ------------------------------------------------------------
# TEXT / ENTITY HELPERS
# ------------------------------------------------------------
def _safe_normalize_student_context_text(text: str) -> str:
    if not text:
        return ""

    fn = globals().get("normalize_sql_text")
    if callable(fn):
        try:
            result = fn(text)
            return result if isinstance(result, str) else str(text).strip()
        except Exception:
            return str(text).strip()

    return str(text).strip()


def extract_student_id_from_question_8(text: str) -> Optional[str]:
    if not text:
        return None

    fn = globals().get("extract_student_id")
    if callable(fn):
        try:
            return fn(text)
        except Exception:
            return None

    return None


def extract_student_name_from_question_8(text: str) -> Optional[str]:
    fn = globals().get("extract_student_name_from_question")
    if callable(fn):
        try:
            return fn(text)
        except Exception:
            return None
    return None


def _safe_fuzzy_match_student_name_8(name: Optional[str]) -> Optional[str]:
    if not name:
        return None

    fn = globals().get("fuzzy_match_student_name")
    if callable(fn):
        try:
            result = fn(name)
            if getattr(result, "matched", False):
                return getattr(result, "matched_value", None) or name
            if isinstance(result, dict) and result.get("matched"):
                return result.get("matched_value") or name
        except Exception:
            pass

    return name


# ------------------------------------------------------------
# DB -> CANONICAL FEATURE SHAPE
# ------------------------------------------------------------
def db_risk_to_canonical_features(db_row: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not db_row:
        return {}

    return {
        "student_id": db_row.get("student_id"),
        "attendance": db_row.get("attendance"),
        "participation": db_row.get("participation"),
        "quizzes": db_row.get("quizzes"),
        "assignments": db_row.get("assignments"),
        "midterm": db_row.get("midterm"),
        "projects": db_row.get("projects"),
    }


def db_support_to_canonical_features(db_row: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not db_row:
        return {}

    return {
        "student_id": db_row.get("student_id"),
        "study_hours_per_week": db_row.get("study_hours_per_week"),
        "social_media_usage_hours_per_day": db_row.get("social_media_usage_hours_per_day"),
        "sleep_duration_hours_per_night": db_row.get("sleep_duration_hours_per_night"),
        "physical_exercise_hours_per_week": db_row.get("physical_exercise_hours_per_week"),
        "family_support": db_row.get("family_support"),
        "financial_stress": db_row.get("financial_stress"),
        "peer_pressure": db_row.get("peer_pressure"),
        "mental_stress_level": db_row.get("mental_stress_level"),
        "diet_quality": db_row.get("diet_quality"),
        "cognitive_distortions": db_row.get("cognitive_distortions"),
    }


# ------------------------------------------------------------
# FEATURE MERGING
# ------------------------------------------------------------
def merge_feature_sources(
    db_features: Optional[Dict[str, Any]],
    manual_features: Optional[Dict[str, Any]],
    prefer_db: bool = True
) -> Dict[str, Any]:
    db_features = db_features or {}
    manual_features = manual_features or {}

    merged = {}
    all_keys = set(db_features.keys()) | set(manual_features.keys())

    for key in all_keys:
        if key == "student_id":
            merged[key] = db_features.get(key) or manual_features.get(key)
            continue

        db_value = db_features.get(key)
        manual_value = manual_features.get(key)

        if prefer_db:
            merged[key] = db_value if db_value is not None else manual_value
        else:
            merged[key] = manual_value if manual_value is not None else db_value

    return merged


# ------------------------------------------------------------
# DATA SOURCE LABELING
# ------------------------------------------------------------
def infer_resolution_data_source(
    student_id: Optional[str],
    profile_found: bool,
    has_manual_risk: bool,
    has_manual_support: bool,
) -> str:
    has_manual = bool(has_manual_risk or has_manual_support)

    if profile_found and has_manual:
        return "mixed"

    if profile_found:
        return "database"

    if has_manual:
        return "manual"

    if student_id and not profile_found:
        return "student_id_not_found"

    return "none"


# ------------------------------------------------------------
# PROFILE HELPERS
# ------------------------------------------------------------
def _safe_load_student_profile(student_id: Optional[str]) -> Dict[str, Any]:
    if not student_id:
        return {"found": False}

    fn = globals().get("load_student_profile")
    if callable(fn):
        try:
            result = fn(student_id)
            return result if isinstance(result, dict) else {"found": False}
        except Exception:
            return {"found": False}

    return {"found": False}


def _safe_lookup_student_profile_by_name(student_name: Optional[str]) -> Dict[str, Any]:
    """
    Best-effort fallback إذا توفر lookup بالاسم.
    """
    if not student_name:
        return {"found": False}

    # 1) direct helper if exists
    fn = globals().get("load_student_profile_by_name")
    if callable(fn):
        try:
            result = fn(student_name)
            return result if isinstance(result, dict) else {"found": False}
        except Exception:
            pass

    # 2) try DB if fetch_all exists
    fetch_all_fn = globals().get("fetch_all")
    if callable(fetch_all_fn):
        try:
            rows = fetch_all_fn(
                """
                SELECT student_id
                FROM students
                WHERE student_name = ?
                LIMIT 1
                """,
                (student_name,)
            )
            if rows:
                sid = rows[0].get("student_id")
                if sid:
                    return _safe_load_student_profile(sid)
        except Exception:
            pass

    return {"found": False}


def _extract_profile_summary(profile: Dict[str, Any]) -> Dict[str, Any]:
    if not profile:
        return {}

    # shapes supported:
    # 1) profile["student_info"] = {...}
    # 2) profile["student"] = {...}
    # 3) profile["student_profile"] = {...}
    # 4) profile itself contains fields directly
    student = None

    if isinstance(profile.get("student_info"), dict):
        student = profile.get("student_info")
    elif isinstance(profile.get("student"), dict):
        student = profile.get("student")
    elif isinstance(profile.get("student_profile"), dict):
        student = profile.get("student_profile")
    elif isinstance(profile.get("profile"), dict):
        student = profile.get("profile")
    else:
        student = profile

    def pick_from_student(*keys):
        for key in keys:
            if isinstance(student, dict) and student.get(key) is not None:
                return student.get(key)
        return None

    summary = {
        "student_id": pick_from_student("student_id", "id") or profile.get("student_id"),
        "student_name": pick_from_student("student_name", "name", "full_name"),
        "advisor": pick_from_student("advisor", "advisor_name", "academic_advisor"),
        "gpa": pick_from_student("gpa", "GPA", "student_gpa"),
        "status": pick_from_student("status", "student_status", "academic_status", "cohort_rank"),
        "level": pick_from_student("level", "student_level"),
        "major": pick_from_student("major", "specialization", "program"),
        "hours_completed": pick_from_student("hours_completed", "completed_hours", "earned_hours"),
        "remaining_hours": pick_from_student("remaining_hours", "hours_remaining"),
        "graduating_this_year": pick_from_student("graduating_this_year"),
        "has_medical_condition": pick_from_student("has_medical_condition"),
        "needs_accommodation": pick_from_student("needs_accommodation"),
        "notes": pick_from_student("notes"),
    }

    # fallback from top-level profile if present
    for key in [
        "student_name",
        "advisor",
        "gpa",
        "status",
        "level",
        "major",
        "hours_completed",
        "remaining_hours",
        "graduating_this_year",
        "has_medical_condition",
        "needs_accommodation",
        "notes",
    ]:
        if summary.get(key) is None and profile.get(key) is not None:
            summary[key] = profile.get(key)

    return summary

# ------------------------------------------------------------
# MODEL INPUT BUILDERS
# ------------------------------------------------------------
def build_risk_model_input(
    *,
    student_id: Optional[str],
    db_risk_features: Optional[Dict[str, Any]],
    manual_risk_features: Optional[Dict[str, Any]],
    prefer_db: bool = True
) -> Dict[str, Any]:
    merged = merge_feature_sources(
        db_features=db_risk_features,
        manual_features=manual_risk_features,
        prefer_db=prefer_db
    )

    merged["student_id"] = merged.get("student_id") or student_id

    return {
        "student_id": student_id,
        "model_name": "risk",
        "features": merged,
        "feature_count": _safe_count_risk_features(merged),
        "has_features": _safe_count_risk_features(merged) > 0,
        "source_policy": "prefer_db_fill_missing_from_manual" if prefer_db else "prefer_manual_fill_missing_from_db",
    }


def build_support_model_input(
    *,
    student_id: Optional[str],
    db_support_features: Optional[Dict[str, Any]],
    manual_support_features: Optional[Dict[str, Any]],
    prefer_db: bool = True
) -> Dict[str, Any]:
    merged = merge_feature_sources(
        db_features=db_support_features,
        manual_features=manual_support_features,
        prefer_db=prefer_db
    )

    merged["student_id"] = merged.get("student_id") or student_id

    return {
        "student_id": student_id,
        "model_name": "support",
        "features": merged,
        "feature_count": _safe_count_support_features(merged),
        "has_features": _safe_count_support_features(merged) > 0,
        "source_policy": "prefer_db_fill_missing_from_manual" if prefer_db else "prefer_manual_fill_missing_from_db",
    }


# ------------------------------------------------------------
# MAIN CONTEXT RESOLUTION
# ------------------------------------------------------------
def resolve_student_context(
    *,
    student_id: Optional[str] = None,
    student_name: Optional[str] = None,
    question: Optional[str] = None,
    prefer_db: bool = True
) -> Dict[str, Any]:
    question_normalized = _safe_normalize_student_context_text(question or "")

    if not student_id and question_normalized:
        student_id = extract_student_id_from_question_8(question_normalized)

    if not student_name and question_normalized:
        student_name = extract_student_name_from_question_8(question_normalized)

    if student_name:
        student_name = _safe_fuzzy_match_student_name_8(student_name)

    manual_risk = _safe_extract_risk_features_from_query(question_normalized)
    manual_support = _safe_extract_support_features_from_query(question_normalized)

    has_manual_risk = _safe_count_risk_features(manual_risk) > 0
    has_manual_support = _safe_count_support_features(manual_support) > 0

    profile = None
    profile_found = False
    warnings = []

    # load by student_id first
    if student_id:
        profile = _safe_load_student_profile(student_id)
        profile_found = bool((profile or {}).get("found"))

    # fallback by student_name
    if not profile_found and student_name:
        profile = _safe_lookup_student_profile_by_name(student_name)
        profile_found = bool((profile or {}).get("found"))

        if profile_found and not student_id:
            student = profile.get("student") or {}
            student_id = student.get("student_id") or profile.get("student_id")

    if student_id and not profile_found:
        warnings.append(f"لم يتم العثور على الطالب برقم {student_id} في قاعدة البيانات.")

    db_risk = {}
    db_support = {}
    profile_summary = {}

    if profile_found:
        db_risk = db_risk_to_canonical_features((profile or {}).get("risk_features"))
        db_support = db_support_to_canonical_features((profile or {}).get("support_features"))
        profile_summary = _extract_profile_summary(profile)

    final_risk = merge_feature_sources(
        db_features=db_risk,
        manual_features=manual_risk,
        prefer_db=prefer_db
    )

    final_support = merge_feature_sources(
        db_features=db_support,
        manual_features=manual_support,
        prefer_db=prefer_db
    )

    data_source = infer_resolution_data_source(
        student_id=student_id,
        profile_found=profile_found,
        has_manual_risk=has_manual_risk,
        has_manual_support=has_manual_support,
    )

    if data_source == "mixed":
        warnings.append(
            "تم اعتماد بيانات قاعدة البيانات كأساس، مع استخدام القيم اليدوية فقط لتعبئة الخصائص الناقصة عند الحاجة."
        )

    risk_model_input = build_risk_model_input(
        student_id=student_id,
        db_risk_features=db_risk,
        manual_risk_features=manual_risk,
        prefer_db=prefer_db
    )

    support_model_input = build_support_model_input(
        student_id=student_id,
        db_support_features=db_support,
        manual_support_features=manual_support,
        prefer_db=prefer_db
    )

    return {
        "ok": True,
        "question": question,
        "question_normalized": question_normalized,
        "student_id": student_id,
        "student_name": student_name,
        "profile": profile,
        "profile_found": profile_found,
        "profile_summary": profile_summary,
        "risk_features": final_risk,
        "support_features": final_support,
        "manual_risk_features": manual_risk,
        "manual_support_features": manual_support,
        "db_risk_features": db_risk,
        "db_support_features": db_support,
        "risk_model_input": risk_model_input,
        "support_model_input": support_model_input,
        "data_source": data_source,
        "merge_policy": "prefer_db_fill_missing_from_manual" if prefer_db else "prefer_manual_fill_missing_from_db",
        "has_manual_risk": has_manual_risk,
        "has_manual_support": has_manual_support,
        "warnings": warnings,
        "reference_source": "student_id_or_name_resolution",
        "reference_matches": [x for x in [student_id, student_name] if x],
    }


def student_model_analysis(
    student_id=None,
    question=None,
    model_type="auto",
    prefer_db=True,
):
    ctx = resolve_student_context(
        student_id=student_id,
        question=question,
        prefer_db=prefer_db,
    )

    sid = ctx.get("student_id") or student_id

    if not sid:
        return {
            "ok": False,
            "student_id": None,
            "answer": "لم أستطع تحديد الطالبة المطلوبة من السؤال.",
            "data": {"context": ctx},
        }

    academic_case = student_academic_case(sid, "backlog")
    course_summary = get_student_course_completion_summary(sid)

    risk_input = ctx.get("risk_model_input") or {}
    support_input = ctx.get("support_model_input") or {}

    risk_features = risk_input.get("features", risk_input)
    support_features = support_input.get("features", support_input)

    risk_result = None
    support_result = None

    if risk_features:
        risk_result = run_risk_tool(risk_features)

    if support_features:
        support_result = run_support_tool(support_features)

    answer_parts = []

    if isinstance(academic_case, dict):
        answer_parts.append(academic_case.get("answer", ""))

    if isinstance(course_summary, dict):
        answer_parts.append(course_summary.get("answer", ""))

    if isinstance(risk_result, dict) and risk_result.get("ok"):
        risk_label = risk_result.get("structured_output", {}).get("risk_label_ar")
        if risk_label:
            answer_parts.append(f"تشير نتيجة نموذج الخطورة إلى أن الحالة: {risk_label}.")

    if isinstance(support_result, dict) and support_result.get("ok"):
        structured = support_result.get("structured_output", {})
        support_label = structured.get("support_label_ar")
        primary_factor = structured.get("primary_stress_factor")

        if support_label:
            answer_parts.append(f"تشير نتيجة نموذج الدعم إلى أن الطالبة: {support_label}.")

        if primary_factor:
            answer_parts.append(f"أبرز عامل مؤثر: {primary_factor}.")

    return {
        "ok": True,
        "student_id": sid,
        "answer": "\n".join([p for p in answer_parts if p]),
        "data": {
            "context": ctx,
            "academic_case": academic_case,
            "course_summary": course_summary,
            "risk_result": risk_result,
            "support_result": support_result,
        },
    }

    # CONVENIENCE ACCESSORS FOR EXECUTION LAYER
# ------------------------------------------------------------
def resolve_student_profile_only(student_id: Optional[str] = None, question: Optional[str] = None) -> Dict[str, Any]:
    context = resolve_student_context(student_id=student_id, question=question, prefer_db=True)
    return {
        "ok": context["ok"],
        "student_id": context["student_id"],
        "student_name": context["student_name"],
        "profile_found": context["profile_found"],
        "profile": context["profile"],
        "profile_summary": context["profile_summary"],
        "warnings": context["warnings"],
    }


def resolve_student_model_inputs(student_id: Optional[str] = None, question: Optional[str] = None) -> Dict[str, Any]:
    context = resolve_student_context(student_id=student_id, question=question, prefer_db=True)
    return {
        "ok": context["ok"],
        "student_id": context["student_id"],
        "profile_found": context["profile_found"],
        "risk_model_input": context["risk_model_input"],
        "support_model_input": context["support_model_input"],
        "warnings": context["warnings"],
    }


print(" Cell  9 unified student context + model input layer loaded successfully.")

 Cell  9 unified student context + model input layer loaded successfully.


In [ ]:
import json
import re
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Dict, List, Optional, Union


# ============================================================
# cell 9 STUDENT ADVISING FUNCTION ROUTER + REASONING LAYER
# ============================================================
# Objective:
# - Decide whether a student advising question needs a logic function.
# - Select the best allowed function.
# - Extract and validate required arguments.
# - Optionally let an LLM reviewer improve the decision.
# - Optionally execute the selected function.
#
# Important:
# - This layer is NOT retrieval.
# - Retrieval/embedding should happen before this layer.
# - This layer is for logic, counting, filtering, grouping, model input,
#   and academic decision functions.
# ============================================================


# ============================================================
# ALLOWED LOGIC FUNCTIONS
# ============================================================
# Keep only public logic functions here.
# Do not add helper functions like fetch_one, fetch_all, _rows, etc.
# Names must match the real function names loaded in globals().

ALLOWED_FUNCTION_NAMES = {
    "training_students",
    "get_student_training_status",
    "student_academic_case",
    "advisor_students",
    "advisor_ranking",
    "student_count",
    "get_course_enrollment_count",
    "students_not_passed_course",
    "exceptional_opening",
    "student_factor_analysis",
    "student_model_analysis",
    "students_count_by_advisor",
    "backlog_students_count_by_advisor",
    "training_needed_count_by_advisor",
    "training_completed_count_by_advisor",
    "advisor_full_summary",
    "get_student_course_completion_summary",
    "resolve_student_profile_only",
    "resolve_student_model_inputs",
    "course_lookup",
}


# ============================================================
# DATACLASSES
# ============================================================

@dataclass
class FunctionDecision:
    function_name: Optional[str] = None
    arguments: Dict[str, Any] = field(default_factory=dict)
    needs_resolution: bool = False
    missing_arguments: List[str] = field(default_factory=list)
    confidence: float = 0.0
    reason: str = ""


@dataclass
class FunctionExecutionResult:
    ok: bool
    function_name: Optional[str]
    arguments: Dict[str, Any]
    result: Any = None
    error: Optional[str] = None
    decision: Optional[Dict[str, Any]] = None


# ============================================================
# FUNCTION REGISTRY
# ============================================================

def build_function_registry(namespace: Optional[Dict[str, Any]] = None) -> Dict[str, Callable]:
    """
    Build registry from already-loaded functions.
    Only functions listed in ALLOWED_FUNCTION_NAMES can be exposed.
    """
    ns = namespace if namespace is not None else globals()
    registry: Dict[str, Callable] = {}

    for name in ALLOWED_FUNCTION_NAMES:
        fn = ns.get(name)
        if callable(fn):
            registry[name] = fn

    return registry


# ============================================================
# NORMALIZATION HELPERS
# ============================================================

def normalize_text(text: Optional[str]) -> str:
    if not text:
        return ""

    text = str(text).strip()
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = re.sub(r"\s+", " ", text)
    return text


def contains_any(text: str, terms: List[str]) -> bool:
    return any(term in text for term in terms)


def extract_student_id(question: str) -> Optional[str]:
    q = question or ""

    patterns = [
        r"\b\d{5,12}\b",
        r"\bS\d{3,12}\b",
        r"\bstudent[_\-]?\d{3,12}\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, q, flags=re.IGNORECASE)
        if match:
            return match.group(0)

    return None


def extract_course_key(question: str) -> Optional[str]:
    q = question or ""

    patterns = [
        r"\bCOURSE[_\-]?\d+\b",
        r"\b[A-Z]{2,10}[_\-]?\d{2,10}\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, q, flags=re.IGNORECASE)
        if match:
            return match.group(0).replace("-", "_").upper()

    return None


def extract_limit(question: str) -> Optional[int]:
    q = question or ""

    patterns = [
        r"(?:اول|أول|اعلى|اعلي|top)\s+(\d+)",
        r"(?:limit|حد)\s*[:=]?\s*(\d+)",
        r"\b(\d+)\s+(?:مرشدات|مرشدين|دكاتره|دكتوره|دكتور|نتائج|طلاب|طالبات)\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, q, flags=re.IGNORECASE)
        if match:
            try:
                return int(match.group(1))
            except Exception:
                return None

    return None


def is_group_by_advisor_phrase(value: Optional[str]) -> bool:
    if not value:
        return False

    n = normalize_text(value)
    n = re.sub(r"[؟\?,،.]+", "", n).strip()

    forbidden = {
        "كل مرشده",
        "كل مرشد",
        "كل دكتوره",
        "كل دكتور",
        "جميع المرشدات",
        "جميع المرشدين",
        "جميع الدكاتره",
        "كل",
        "الكل",
    }

    return (
        n in forbidden
        or n.startswith("كل مرشده")
        or n.startswith("كل دكتوره")
        or n.startswith("كل مرشد")
        or n.startswith("كل دكتور")
    )


def extract_advisor_name(question: str) -> Optional[str]:
    raw = question or ""
    q_norm = normalize_text(raw)

    group_by_phrases = [
        "كل مرشده",
        "كل دكتوره",
        "كل مرشد",
        "كل دكتور",
        "عند كل مرشده",
        "عند كل دكتوره",
        "حسب المرشده",
        "حسب المرشد",
        "لكل مرشده",
        "لكل دكتوره",
    ]

    if contains_any(q_norm, group_by_phrases):
        return None

    patterns = [
        r"(?:عند|لدى|لدي|مع)\s+(?!كل\s+(?:مرشدة|مرشده|دكتورة|دكتوره|مرشد|دكتور))([^؟\?\n,،]+)",
        r"(?:مرشده|مرشدة|الدكتوره|الدكتورة|دكتوره|دكتورة|د\.)\s+(?!كل\b)([^؟\?\n,،]+)",
        r"advisor\s*[:=]\s*([^؟\?\n,،]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, raw, flags=re.IGNORECASE)
        if match:
            value = re.sub(r"\s+", " ", match.group(1).strip())
            value = re.sub(r"^(كل|جميع)\s+", "", value).strip()

            if not value:
                continue

            if is_group_by_advisor_phrase(value):
                continue

            normalized = normalize_text(value)
            if normalized in {"مرشده", "مرشد", "دكتوره", "دكتور", "كل"}:
                continue

            return value

    return None


# ============================================================
# FEATURE EXTRACTION FOR REASONING
# ============================================================

def extract_question_features(question: str) -> Dict[str, Any]:
    """
    Extract semantic flags used by the routing/reasoning layer.
    These are not final answers; they are signals to choose the right logic function.
    """
    raw = question or ""
    q = normalize_text(raw)

    student_id = extract_student_id(raw)
    course_key = extract_course_key(raw)
    advisor_name = extract_advisor_name(raw)
    limit = extract_limit(raw)

    has_count = contains_any(q, [
        "عدد", "كم", "احصاء", "احصائيه", "اجمالي", "مجموع",
        "count", "total",
    ])

    has_list_request = contains_any(q, [
        "من", "مين", "اعرض", "اعرضي", "قائمه", "قائمة", "اذكر", "اذكري",
        "show", "list", "who",
    ])

    has_group_by_advisor = (
        contains_any(q, [
            "عند كل مرشده",
            "لكل مرشده",
            "حسب المرشده",
            "حسب المرشد",
            "لكل دكتوره",
            "عند كل دكتوره",
            "كل مرشده",
            "كل دكتوره",
            "كل مرشد",
            "كل دكتور",
            "by advisor",
            "group by advisor",
            "grouped by advisor",
        ])
        or (
            contains_any(q, ["كل مرشده", "كل دكتوره", "كل مرشد", "كل دكتور"])
            and contains_any(q, ["عدد", "متعثر", "تدريب", "طالبات", "طلاب"])
        )
    )

    has_advisor_summary = contains_any(q, [
        "ملخص المرشدات",
        "ملخص المرشده",
        "ملخص لكل مرشده",
        "تقرير المرشدات",
        "تقرير لكل مرشده",
        "advisor summary",
        "advisor full summary",
        "summary by advisor",
    ])

    has_backlog = contains_any(q, [
        "متعثر", "متاخر", "متاخرات", "باقي", "باقيه", "متبقي",
        "متبقيه", "غير مجتاز", "غير مجتازه", "لم تجتز", "لم يجتاز",
        "لم ينجح", "لم تنجح", "راسب", "راسبه", "backlog", "remaining",
        "not passed", "unfinished", "failed",
    ])

    has_training = contains_any(q, [
        "تدريب", "التدريب", "تعاوني", "الصيفي", "summer training",
        "training", "cohort_rank",
    ])

    has_training_needed = has_training and contains_any(q, [
        "يحتاج", "يحتاجون", "تحتاج", "احتياج",
        "ما اخذ", "ما اخذت", "لم تاخذ", "لم يأخذ", "لم ياخذ",
        "لم تجتز", "لم يجتاز", "غير مجتاز", "غير مجتازه",
        "ما خلص", "ما خلصت", "لم تكمل", "لم يكمل",
        "غير مكتمل", "غير مكتمله", "not taken", "needs", "needed",
        "np",
    ])

    has_training_completed = has_training and contains_any(q, [
        "انهى", "انهت", "انهي", "خلص", "خلصت", "اكمل", "اكملت",
        "اجتاز", "اجتازت", "completed", "passed", "ip",
    ])
    has_course_completion_split = (
    student_id is not None
    and (
        contains_any(q, [
            "المواد الباقيه",
            "المواد الباقية",
            "المواد المتبقيه",
            "المواد المتبقية",
            "المواد التي درس",
            "المواد اللتي درس",
            "المواد اللي درس",
            "المواد التي درست",
            "المواد اللتي درست",
            "المواد اللي درست",
            "ماهي المواد",
            "ما هي المواد",
            "وش المواد",
            "ايش المواد",
            "المقررات التي درس",
            "المقررات اللتي درس",
            "المقررات اللي درس",
            "المقررات التي درست",
            "المقررات اللتي درست",
            "المقررات اللي درست",
            "اللي خلصتها",
            "اللي خلصها",
            "خلصتها",
            "خلصها",
            "انجزتها",
            "انجزها",
            "المجتازه",
            "المجتازة",
            "completed courses",
            "remaining courses",
            "courses taken",
            "studied courses",
            "course history",
            "student courses",
        ])
        or (
            contains_any(q, ["باقيه", "باقية", "متبقيه", "متبقية", "متبقي"])
            and contains_any(q, ["خلصتها", "خلصها", "المجتازه", "المجتازة", "completed"])
        )
    )
)#)

    has_course_lookup = course_key is not None and contains_any(q, [
        "اسم المقرر", "بيانات المقرر", "معلومات المقرر", "ساعات", "الساعات",
        "متطلب", "متطلبات", "prerequisite", "prerequisites", "course info",
    ])

    has_course_not_passed = course_key is not None and contains_any(q, [
        "لم يجتزن", "لم تجتز", "لم يجتاز", "غير مجتاز", "غير مجتازه",
        "راسب", "راسبه", "متعثر", "متعثره", "not passed", "failed",
        "remaining",
    ])

    has_exceptional_opening = contains_any(q, [
        "فتح استثنائي", "استثنائي", "exceptional opening", "فتح المقرر",
    ])

    has_factor_analysis = contains_any(q, [
        "عوامل", "اسباب", "أسباب", "تحليل العوامل", "factor", "factors",
    ])

    has_model_analysis = contains_any(q, [
        "نموذج", "model", "prediction", "تنبؤ", "تحليل النموذج",
        "تحليل ذكي", "ai analysis",
    ])

    has_profile_request = student_id is not None and contains_any(q, [
        "اعرض بيانات", "بيانات الطالبه", "بيانات الطالبة",
        "معلومات الطالبه", "معلومات الطالبة",
        "الساعات المكتمله", "الساعات المكتملة",
        "الساعات المنجزه", "الساعات المنجزة",
        "مرشده", "مرشدة", "ملاحظات",
    ])

    has_student_condition_analysis = student_id is not None and contains_any(q, [
        "وضع الطالبه", "وضع الطالبة",
        "حلل", "حللي", "تحليل",
        "خطر", "خطوره", "خطورة",
        "دعم", "تحتاج دعم",
        "سبب التعثر", "اسباب التعثر",
    ])

    has_ranking = contains_any(q, [
        "ترتيب", "اعلى", "اعلي", "اقل", "top", "ranking", "rank",
    ])

    return {
        "raw_question": raw,
        "normalized_question": q,
        "student_id": student_id,
        "course_key": course_key,
        "advisor_name": advisor_name,
        "limit": limit,
        "has_count": has_count,
        "has_list_request": has_list_request,
        "has_group_by_advisor": has_group_by_advisor,
        "has_advisor_summary": has_advisor_summary,
        "has_backlog": has_backlog,
        "has_training": has_training,
        "has_training_needed": has_training_needed,
        "has_training_completed": has_training_completed,
        "has_course_completion_split": has_course_completion_split,
        "has_course_lookup": has_course_lookup,
        "has_course_not_passed": has_course_not_passed,
        "has_exceptional_opening": has_exceptional_opening,
        "has_factor_analysis": has_factor_analysis,
        "has_model_analysis": has_model_analysis,
        "has_profile_request": has_profile_request,
        "has_student_condition_analysis": has_student_condition_analysis,
        "has_ranking": has_ranking,
    }


# ============================================================
# DETERMINISTIC FUNCTION SELECTION
# ============================================================

def deterministic_function_selector(question: str) -> FunctionDecision:
    """
    First-pass deterministic reasoning.
    This should be stable and predictable.
    The optional LLM reviewer can improve it, but this must work alone.
    """
    features = extract_question_features(question)

    student_id = features["student_id"]
    course_key = features["course_key"]
    advisor_name = features["advisor_name"]
    limit = features["limit"]

    if not question or not str(question).strip():
        return FunctionDecision(
            function_name=None,
            arguments={},
            needs_resolution=True,
            missing_arguments=["question"],
            confidence=0.0,
            reason="empty question",
        )

    # 1) Advisor grouped summaries must be handled before individual advisor extraction.
    if features["has_advisor_summary"]:
        return FunctionDecision(
            function_name="advisor_full_summary",
            arguments={},
            confidence=0.98,
            reason="question asks for full advisor-level summary",
        )

    if features["has_group_by_advisor"] and features["has_training_needed"]:
        return FunctionDecision(
            function_name="training_needed_count_by_advisor",
            arguments={},
            confidence=0.99,
            reason="question asks for training-needed counts grouped by advisor",
        )

    if features["has_group_by_advisor"] and features["has_training_completed"]:
        return FunctionDecision(
            function_name="training_completed_count_by_advisor",
            arguments={},
            confidence=0.99,
            reason="question asks for completed-training counts grouped by advisor",
        )

    if features["has_group_by_advisor"] and features["has_backlog"]:
        return FunctionDecision(
            function_name="backlog_students_count_by_advisor",
            arguments={},
            confidence=0.99,
            reason="question asks for backlog/high-risk counts grouped by advisor",
        )

    if features["has_group_by_advisor"] and features["has_count"]:
        return FunctionDecision(
            function_name="students_count_by_advisor",
            arguments={},
            confidence=0.99,
            reason="question asks for student counts grouped by advisor",
        )

    # 2) Course-specific logic.
    if features["has_course_completion_split"]:
        return FunctionDecision(
            function_name="get_student_course_completion_summary",
            arguments={"student_id": student_id} if student_id else {},
            needs_resolution=student_id is None,
            missing_arguments=[] if student_id else ["student_id"],
            confidence=0.99 if student_id else 0.60,
            reason="question asks to split student's completed and remaining courses",
        )

    if features["has_course_not_passed"] and course_key:
        return FunctionDecision(
            function_name="students_not_passed_course",
            arguments={"course_key": course_key},
            confidence=0.93,
            reason="question asks for students who have not passed a specific course",
        )

    if features["has_course_lookup"] and course_key:
        return FunctionDecision(
            function_name="course_lookup",
            arguments={"course_key": course_key},
            confidence=0.90,
            reason="question asks for course information",
        )

    if course_key and features["has_count"]:
        return FunctionDecision(
            function_name="get_course_enrollment_count",
            arguments={"course_key": course_key},
            confidence=0.86,
            reason="question asks for course enrollment count",
        )

    if features["has_exceptional_opening"]:
        args = {"mode": "list"}
        if course_key:
            args = {"mode": "check_course", "course_key": course_key}

        return FunctionDecision(
            function_name="exceptional_opening",
            arguments=args,
            confidence=0.93,
            reason="question asks for exceptional course opening logic",
        )

    # 3) Student-specific logic.
    if features["has_training"] and student_id:
        return FunctionDecision(
            function_name="get_student_training_status",
            arguments={"student_id": student_id},
            confidence=0.95,
            reason="question asks for a specific student's training status",
        )

    if features["has_backlog"] and student_id:
        return FunctionDecision(
            function_name="student_academic_case",
            arguments={"student_id": student_id, "case_type": "backlog"},
            confidence=0.95,
            reason="question asks for a specific student's academic backlog/high-risk case",
        )

    if features["has_profile_request"]:
        return FunctionDecision(
            function_name="resolve_student_profile_only",
            arguments={"student_id": student_id, "question": question},
            confidence=0.90,
            reason="question asks for basic student profile fields",
        )

    if features["has_model_analysis"] or features["has_student_condition_analysis"]:
        args: Dict[str, Any] = {
            "question": question,
            "model_type": "auto",
            "prefer_db": True,
        }
        if student_id:
            args["student_id"] = student_id

        return FunctionDecision(
            function_name="student_model_analysis",
            arguments=args,
            needs_resolution=student_id is None,
            missing_arguments=[] if student_id else ["student_id"],
            confidence=0.92 if student_id else 0.60,
            reason="question asks for student reasoning/model-based analysis",
        )

    # 4) Training aggregate logic.
    if features["has_training_needed"] and features["has_count"]:
        args = {"status": "needs_training", "mode": "count"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="training_students",
            arguments=args,
            confidence=0.92,
            reason="question asks to count students needing training",
        )

    if features["has_training_completed"] and features["has_count"]:
        args = {"status": "completed", "mode": "count"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="training_students",
            arguments=args,
            confidence=0.92,
            reason="question asks to count students who completed training",
        )

    # 5) Ranking and factor analysis.
    if features["has_factor_analysis"]:
        args = {"analysis_type": "backlog_factors"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="student_factor_analysis",
            arguments=args,
            confidence=0.88,
            reason="question asks for backlog factor summary",
        )

    if features["has_ranking"] and features["has_backlog"]:
        args = {"metric": "backlog_students"}
        if limit:
            args["limit"] = limit

        return FunctionDecision(
            function_name="advisor_ranking",
            arguments=args,
            confidence=0.90,
            reason="question asks for advisor ranking by backlog/high-risk students",
        )

    if features["has_ranking"] and features["has_training_needed"]:
        args = {"metric": "training_needed"}
        if limit:
            args["limit"] = limit

        return FunctionDecision(
            function_name="advisor_ranking",
            arguments=args,
            confidence=0.90,
            reason="question asks for advisor ranking by students needing training",
        )

    if features["has_ranking"] and features["has_training_completed"]:
        args = {"metric": "training_completed"}
        if limit:
            args["limit"] = limit

        return FunctionDecision(
            function_name="advisor_ranking",
            arguments=args,
            confidence=0.90,
            reason="question asks for advisor ranking by completed training",
        )

    if features["has_ranking"] and features["has_count"]:
        args = {"metric": "student_count"}
        if limit:
            args["limit"] = limit

        return FunctionDecision(
            function_name="advisor_ranking",
            arguments=args,
            confidence=0.90,
            reason="question asks for advisor ranking by student count",
        )

    # 6) General counts.
    if features["has_count"] and features["has_backlog"]:
        args = {"metric": "backlog_students"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="student_count",
            arguments=args,
            confidence=0.88,
            reason="question asks to count backlog/high-risk students",
        )

    if features["has_count"] and features["has_training_needed"]:
        args = {"status": "needs_training", "mode": "count"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="training_students",
            arguments=args,
            confidence=0.88,
            reason="question asks to count students needing training",
        )

    if features["has_count"]:
        args = {"metric": "all_students"}
        if advisor_name:
            args["advisor_name"] = advisor_name

        return FunctionDecision(
            function_name="student_count",
            arguments=args,
            confidence=0.82,
            reason="question asks for a general student count",
        )

    return FunctionDecision(
        function_name=None,
        arguments={},
        needs_resolution=True,
        missing_arguments=["function_name"],
        confidence=0.30,
        reason="no suitable logic function was selected; answer may belong to retrieval/reasoning outside this router",
    )


# ============================================================
# ARABIC LLM FUNCTION REVIEWER PROMPT
# ============================================================

LLM_FUNCTION_REVIEWER_SYSTEM_PROMPT = """
أنت مراجع دلالي متخصص في اختيار الدالة المناسبة داخل نظام إرشاد طالبات.

هذه الطبقة ليست طبقة استرجاع.
الاسترجاع بالـ embeddings يتم خارج هذه الطبقة.
هذه الطبقة تُستخدم فقط عندما يحتاج السؤال إلى منطق، حساب، فلترة، تجميع، قرار أكاديمي، أو تجهيز مدخلات نموذج.

مهمتك:
راجع قرار deterministic_function_selector ثم أعد القرار النهائي بصيغة JSON فقط.

ممنوع:
- لا تجب عن سؤال المستخدم.
- لا تنفذ أي دالة.
- لا تكتب SQL.
- لا تشرح خارج JSON.
- لا تختر دالة غير موجودة في القائمة المسموحة.
- لا تختر RETRIEVAL.
- لا تختر helper functions.

حلّل معنى السؤال، لا تعتمد فقط على الكلمات المفتاحية.

الدوال المسموحة فقط:
- training_students
- get_student_training_status
- student_academic_case
- advisor_students
- advisor_ranking
- student_count
- get_course_enrollment_count
- students_not_passed_course
- exceptional_opening
- student_factor_analysis
- student_model_analysis
- students_count_by_advisor
- backlog_students_count_by_advisor
- training_needed_count_by_advisor
- training_completed_count_by_advisor
- advisor_full_summary
- get_student_course_completion_summary
- resolve_student_profile_only
- resolve_student_model_inputs
- course_lookup

قواعد المجال:

منطق التدريب في students.cohort_rank:
- IP = أنهت / اجتازت التدريب
- NP = لم تجتز أو لم تكمل التدريب وتحتاج تدريب
- NULL = لم تأخذ التدريب بعد وتحتاج تدريب
- القيمة الفارغة = لم تأخذ التدريب بعد وتحتاج تدريب
- لا يوجد P للتدريب
- لا تعتبر IP قيد التنفيذ

منطق حالة المواد في student_courses.status:
- P = مكتملة / مجتازة
- معادلة = مكتملة أو معادلة ولا تُحسب تعثرًا
- R = مسجلة حاليًا وتُعتبر باقية
- N = تعثر / غير مكتملة
- أي قيمة تبدأ بـ N مثل N مواضيع = تعثر / غير مكتملة
- NULL = باقية / لم تؤخذ
- القيمة الفارغة = باقية / لم تؤخذ
- الحالات غير المعروفة لا تقود القرار الأساسي

اختيار الدوال:

- عدد الطالبات عند كل مرشدة:
  students_count_by_advisor

- عدد الطالبات المتعثرات أو عاليّات الخطورة عند كل مرشدة:
  backlog_students_count_by_advisor

- عدد الطالبات اللاتي يحتجن تدريبًا عند كل مرشدة:
  training_needed_count_by_advisor

- عدد الطالبات اللاتي أنهين التدريب عند كل مرشدة:
  training_completed_count_by_advisor

- ملخص كامل حسب المرشدة:
  advisor_full_summary

- بيانات مقرر محدد:
  course_lookup
  requires course_key

- عدد الطالبات المرتبطات بمقرر:
  get_course_enrollment_count
  requires course_key

- الطالبات غير المجتازات لمقرر محدد:
  students_not_passed_course
  requires course_key

- المواد الباقية والمكتملة لطالبة:
  get_student_course_completion_summary
  requires student_id

- حالة تدريب طالبة محددة:
  get_student_training_status
  requires student_id

- تعثر طالبة محددة:
  student_academic_case
  requires student_id and case_type = backlog

- تحليل نموذج / تحليل حالة طالبة:
  student_model_analysis
  requires student_id, question, model_type = auto, prefer_db = true

- عدد الطالبات أو المتعثرات أو الحالات الطبية أو المتوقع تخرجهن:
  student_count
  requires metric

- طالبات التدريب بشكل عام أو عند مرشدة:
  training_students
  requires status

- ترتيب المرشدات:
  advisor_ranking
  requires metric

- فتح استثنائي:
  exceptional_opening
  requires mode

- تحليل عوامل التعثر:
  student_factor_analysis
  requires analysis_type

قواعد التجميع حسب المرشدة:
هذه العبارات تعني تجميعًا، وليست اسم مرشدة:
- كل مرشدة
- كل دكتورة
- كل مرشد
- كل دكتور
- عند كل مرشدة
- عند كل دكتورة
- حسب المرشدة
- لكل مرشدة

لا تضع هذه العبارات في advisor_name.

إذا كانت الدالة المناسبة تحتاج وسيطًا غير موجود، لا تخمن.
ضع needs_resolution = true و missing_arguments.

إذا لم تكن هناك دالة مناسبة:
ضع function_name = null و needs_resolution = true و missing_arguments = ["function_name"].

إخراج JSON فقط، بدون Markdown.

الشكل المطلوب:
{
  "function_name": "اسم الدالة أو null",
  "arguments": {},
  "needs_resolution": false,
  "missing_arguments": [],
  "confidence": 0.0,
  "reason": "سبب قصير ودقيق"
}
""".strip()


# ============================================================
# LLM REVIEWER
# ============================================================

def build_function_reviewer_user_payload(question: str, decision: FunctionDecision) -> str:
    return json.dumps(
        {
            "question": question,
            "deterministic_function_decision": asdict(decision),
            "allowed_functions": sorted(ALLOWED_FUNCTION_NAMES),
            "required_output_schema": {
                "function_name": "allowed function name or null",
                "arguments": {},
                "needs_resolution": False,
                "missing_arguments": [],
                "confidence": 0.0,
                "reason": "short reason",
            },
        },
        ensure_ascii=False,
    )


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    cleaned = text.strip()

    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start != -1 and end != -1 and end > start:
        try:
            parsed = json.loads(cleaned[start:end + 1])
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return None

    return None


def parse_function_decision(
    payload: Union[str, Dict[str, Any], FunctionDecision, None]
) -> Optional[FunctionDecision]:
    if payload is None:
        return None

    if isinstance(payload, FunctionDecision):
        return payload

    if isinstance(payload, str):
        data = extract_json_object(payload)
        if data is None:
            return None
    elif isinstance(payload, dict):
        data = payload
    else:
        return None

    return FunctionDecision(
        function_name=data.get("function_name"),
        arguments=data.get("arguments") or {},
        needs_resolution=bool(data.get("needs_resolution", False)),
        missing_arguments=list(data.get("missing_arguments") or []),
        confidence=float(data.get("confidence", 0.0) or 0.0),
        reason=str(data.get("reason", "")),
    )


def llm_function_reviewer(
    question: str,
    initial_decision: FunctionDecision,
    *,
    client: Optional[Any] = None,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.0,
) -> FunctionDecision:
    """
    Optional semantic reviewer.
    If no client is provided, the deterministic decision is used as-is.
    """
    if client is None:
        return initial_decision

    try:
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": LLM_FUNCTION_REVIEWER_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": build_function_reviewer_user_payload(question, initial_decision),
                },
            ],
        )

        content = response.choices[0].message.content
        reviewed = parse_function_decision(content)

        if reviewed is None:
            return initial_decision

        return reviewed

    except Exception:
        return initial_decision


# ============================================================
# VALIDATOR
# ============================================================

FUNCTION_REQUIRED_ARGUMENTS = {
    "training_students": ["status"],
    "get_student_training_status": ["student_id"],
    "student_academic_case": ["student_id", "case_type"],
    "advisor_students": ["advisor_name"],
    "advisor_ranking": ["metric"],
    "student_count": ["metric"],
    "get_course_enrollment_count": ["course_key"],
    "students_not_passed_course": ["course_key"],
    "exceptional_opening": ["mode"],
    "student_factor_analysis": ["analysis_type"],
    "student_model_analysis": ["student_id", "question", "model_type", "prefer_db"],
    "students_count_by_advisor": [],
    "backlog_students_count_by_advisor": [],
    "training_needed_count_by_advisor": [],
    "training_completed_count_by_advisor": [],
    "advisor_full_summary": [],
    "resolve_student_profile_only": [],
    "resolve_student_model_inputs": [],
    "get_student_course_completion_summary": ["student_id"],
    "course_lookup": ["course_key"],
}


def validate_function_decision(decision: FunctionDecision) -> FunctionDecision:
    if not decision.function_name:
        decision.needs_resolution = True
        decision.missing_arguments = ["function_name"]
        decision.confidence = min(float(decision.confidence or 0.0), 0.30)
        if not decision.reason:
            decision.reason = "no function selected"
        return decision

    if decision.function_name not in ALLOWED_FUNCTION_NAMES:
        return FunctionDecision(
            function_name=None,
            arguments={},
            needs_resolution=True,
            missing_arguments=["function_name"],
            confidence=min(float(decision.confidence or 0.0), 0.20),
            reason="invalid function name",
        )

    required = FUNCTION_REQUIRED_ARGUMENTS.get(decision.function_name, [])
    missing = [
        arg for arg in required
        if arg not in decision.arguments or decision.arguments.get(arg) in (None, "")
    ]

    decision.missing_arguments = missing
    decision.needs_resolution = len(missing) > 0

    try:
        decision.confidence = max(0.0, min(1.0, float(decision.confidence)))
    except Exception:
        decision.confidence = 0.0

    return decision


# ============================================================
# MISSING ARGUMENT RESOLVER
# ============================================================

def resolve_missing_arguments(question: str, decision: FunctionDecision) -> FunctionDecision:
    if not decision.function_name:
        return validate_function_decision(decision)

    features = extract_question_features(question)
    args = dict(decision.arguments or {})

    if "student_id" in decision.missing_arguments and features["student_id"]:
        args["student_id"] = features["student_id"]

    if "course_key" in decision.missing_arguments and features["course_key"]:
        args["course_key"] = features["course_key"]

    if "advisor_name" in decision.missing_arguments and features["advisor_name"]:
        args["advisor_name"] = features["advisor_name"]

    if "limit" in decision.missing_arguments and features["limit"]:
        args["limit"] = features["limit"]

    if decision.function_name == "student_academic_case":
        args.setdefault("case_type", "backlog")

    if decision.function_name == "student_count":
        if "metric" not in args:
            if features["has_backlog"]:
                args["metric"] = "backlog_students"
            else:
                args["metric"] = "all_students"

        if features["advisor_name"] and "advisor_name" not in args:
            args["advisor_name"] = features["advisor_name"]

    if decision.function_name == "get_course_enrollment_count":
        if "course_key" not in args and features["course_key"]:
            args["course_key"] = features["course_key"]

    if decision.function_name == "course_lookup":
        if "course_key" not in args and features["course_key"]:
            args["course_key"] = features["course_key"]

    if decision.function_name == "training_students":
        if "status" not in args:
            if features["has_training_completed"]:
                args["status"] = "completed"
            else:
                args["status"] = "needs_training"

        args.setdefault(
            "mode",
            "list" if features.get("has_list_request") else "count",
        )

        if features["advisor_name"] and "advisor_name" not in args:
            args["advisor_name"] = features["advisor_name"]

    if decision.function_name == "advisor_ranking":
        if "metric" not in args:
            if features["has_backlog"]:
                args["metric"] = "backlog_students"
            elif features["has_training_needed"]:
                args["metric"] = "training_needed"
            elif features["has_training_completed"]:
                args["metric"] = "training_completed"
            else:
                args["metric"] = "student_count"

        if features["limit"] and "limit" not in args:
            args["limit"] = features["limit"]

    if decision.function_name == "students_not_passed_course":
        if "course_key" not in args and features["course_key"]:
            args["course_key"] = features["course_key"]

    if decision.function_name == "exceptional_opening":
        args.setdefault("mode", "list")

        if args.get("mode") in {"check_course", "count_course"}:
            if "course_key" not in args and features["course_key"]:
                args["course_key"] = features["course_key"]

    if decision.function_name == "student_factor_analysis":
        args.setdefault("analysis_type", "backlog_factors")

        if features["advisor_name"] and "advisor_name" not in args:
            args["advisor_name"] = features["advisor_name"]

    if decision.function_name == "student_model_analysis":
        args["question"] = question
        args["model_type"] = "auto"
        args["prefer_db"] = True

        if "student_id" not in args and features["student_id"]:
            args["student_id"] = features["student_id"]

    if decision.function_name == "get_student_course_completion_summary":
        if "student_id" not in args and features["student_id"]:
            args["student_id"] = features["student_id"]

    if decision.function_name in {"resolve_student_profile_only", "resolve_student_model_inputs"}:
        args["question"] = question

        if "student_id" not in args and features["student_id"]:
            args["student_id"] = features["student_id"]

    if decision.function_name == "get_student_training_status":
        if "student_id" not in args and features["student_id"]:
            args["student_id"] = features["student_id"]

    decision.arguments = args
    return validate_function_decision(decision)


# ============================================================
# EXECUTOR
# ============================================================

def execute_function_decision(
    decision: FunctionDecision,
    *,
    function_registry: Optional[Dict[str, Callable]] = None,
) -> FunctionExecutionResult:
    if decision.needs_resolution:
        return FunctionExecutionResult(
            ok=False,
            function_name=decision.function_name,
            arguments=decision.arguments,
            error=f"Missing required arguments: {decision.missing_arguments}",
            decision=asdict(decision),
        )

    registry = function_registry or build_function_registry()
    fn = registry.get(decision.function_name or "")

    if not callable(fn):
        return FunctionExecutionResult(
            ok=False,
            function_name=decision.function_name,
            arguments=decision.arguments,
            error=f"Function not found in registry: {decision.function_name}",
            decision=asdict(decision),
        )

    try:
        result = fn(**decision.arguments)
        return FunctionExecutionResult(
            ok=True,
            function_name=decision.function_name,
            arguments=decision.arguments,
            result=result,
            decision=asdict(decision),
        )
    except Exception as e:
        return FunctionExecutionResult(
            ok=False,
            function_name=decision.function_name,
            arguments=decision.arguments,
            error=str(e),
            decision=asdict(decision),
        )


# ============================================================
# PUBLIC ENTRYPOINT
# ============================================================

def function_router(
    question: str,
    *,
    execute: bool = False,
    client: Optional[Any] = None,
    model: str = "gpt-4.1-mini",
    function_registry: Optional[Dict[str, Callable]] = None,
) -> Union[Dict[str, Any], FunctionExecutionResult]:
    deterministic_decision = deterministic_function_selector(question)

    reviewed_decision = llm_function_reviewer(
        question,
        deterministic_decision,
        client=client,
        model=model,
        temperature=0.0,
    )

    validated_decision = validate_function_decision(reviewed_decision)
    resolved_decision = resolve_missing_arguments(question, validated_decision)
    final_decision = validate_function_decision(resolved_decision)

    if not execute:
        return asdict(final_decision)

    return execute_function_decision(
        final_decision,
        function_registry=function_registry,
    )


def route_student_question(
    question: str,
    *,
    execute: bool = False,
    client: Optional[Any] = None,
    model: str = "gpt-4.1-mini",
    function_registry: Optional[Dict[str, Callable]] = None,
) -> Union[Dict[str, Any], FunctionExecutionResult]:
    return function_router(
        question=question,
        execute=execute,
        client=client,
        model=model,
        function_registry=function_registry,
    )


# ============================================================
# DEBUG FUNCTION
# ============================================================

def debug_function_router(
    question: str,
    *,
    client: Optional[Any] = None,
    model: str = "gpt-4.1-mini",
) -> Dict[str, Any]:
    features = extract_question_features(question)

    deterministic_decision = deterministic_function_selector(question)

    reviewed_decision = llm_function_reviewer(
        question,
        deterministic_decision,
        client=client,
        model=model,
        temperature=0.0,
    )

    validated_decision = validate_function_decision(reviewed_decision)
    resolved_decision = resolve_missing_arguments(question, validated_decision)
    final_decision = validate_function_decision(resolved_decision)

    return {
        "question": question,
        "features": features,
        "deterministic_decision": asdict(deterministic_decision),
        "reviewed_decision": asdict(reviewed_decision),
        "validated_decision": asdict(validated_decision),
        "resolved_decision": asdict(resolved_decision),
        "final_decision": asdict(final_decision),
        "allowed_functions": sorted(ALLOWED_FUNCTION_NAMES),
    }


def debug_route_student_question(
    question: str,
    *,
    client: Optional[Any] = None,
    model: str = "gpt-4.1-mini",
) -> Dict[str, Any]:
    return debug_function_router(
        question=question,
        client=client,
        model=model,
    )


print(" Student advising function router + reasoning layer loaded successfully.")


 Student advising function router + reasoning layer loaded successfully.


In [ ]:
# ============================================================
# STRICT RETRIEVAL-FIRST LAYER
# Existing Vector DB only.
# LangChain Chroma + E5.
# No rebuild. No SQL. No functions.
# cell 10 cell 10

# Architecture:
# retrieval search → grounded LLM judge → answered / needs_functions / no_answer
# ============================================================

from typing import Any, Dict, List, Optional, Literal
import os
import re
import json
import zipfile
from dataclasses import dataclass, asdict

try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except Exception:
    from langchain_community.embeddings import HuggingFaceEmbeddings


# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

RETRIEVAL_STATUS_ANSWERED = "answered"
RETRIEVAL_STATUS_NEEDS_FUNCTIONS = "needs_functions"
RETRIEVAL_STATUS_NO_ANSWER = "no_answer"

RETRIEVAL_NO_ANSWER = "لا توجد معلومات كافية في البيانات المتاحة."
RETRIEVAL_NEEDS_FUNCTIONS = "هذا السؤال يحتاج مسار FUNCTIONS وليس retrieval."

VECTOR_ZIP_PATH = "data/chroma_students_dbv22.zip"
VECTOR_DB_PATH = "data/chroma_students_db_v2"
VECTOR_COLLECTION_NAME = "students_v2"
EMBED_MODEL = "intfloat/multilingual-e5-large"


# ------------------------------------------------------------
# RESULT OBJECT
# ------------------------------------------------------------

@dataclass
class RetrievalDecision:
    status: Literal["answered", "needs_functions", "no_answer"]
    answer: str
    confidence: float
    reason: str
    documents_used: List[Dict[str, Any]]
    retrieval: Dict[str, Any]


# ------------------------------------------------------------
# EMBEDDINGS
# ------------------------------------------------------------

class E5HuggingFaceEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        texts = [
            t if str(t).startswith("passage:") else f"passage: {t}"
            for t in texts
        ]
        return super().embed_documents(texts)

    def embed_query(self, text: str) -> List[float]:
        text = text if str(text).startswith("query:") else f"query: {text}"
        return super().embed_query(text)


def build_embeddings():
    return E5HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )


# ------------------------------------------------------------
# VECTOR DB
# ------------------------------------------------------------

def ensure_vector_db_ready(
    zip_path: str = VECTOR_ZIP_PATH,
    db_path: str = VECTOR_DB_PATH,
) -> str:
    if os.path.exists(db_path) and os.listdir(db_path):
        return db_path

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Vector DB zip not found: {zip_path}")

    os.makedirs(db_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(db_path)

    return db_path


def get_chroma_collection(
    db_path: str = VECTOR_DB_PATH,
    collection_name: str = VECTOR_COLLECTION_NAME,
):
    db_path = ensure_vector_db_ready(
        zip_path=VECTOR_ZIP_PATH,
        db_path=db_path,
    )

    embeddings = build_embeddings()

    return Chroma(
        collection_name=collection_name,
        persist_directory=db_path,
        embedding_function=embeddings,
    )


# ------------------------------------------------------------
# QUERY HELPERS
# ------------------------------------------------------------

def _normalize_ar(text: Optional[str]) -> str:
    if not text:
        return ""

    text = str(text).strip()
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه").replace("ى", "ي")
    text = re.sub(r"\s+", " ", text)
    return text


def _contains_any(text: str, terms: List[str]) -> bool:
    return any(term in text for term in terms)


def _extract_student_id(question: str) -> Optional[str]:
    m = re.search(r"\b\d{5,12}\b", str(question))
    return m.group(0) if m else None


def _extract_course_key(question: str) -> Optional[str]:
    m = re.search(r"\bCOURSE[_\-]?\d+\b", str(question), re.IGNORECASE)
    return m.group(0).replace("-", "_").upper() if m else None


def question_requires_functions(question: str) -> bool:
    q = _normalize_ar(question)

    logic_terms = [
        "عدد",
        "كم عدد",
        "احصاء",
        "احصائيه",
        "اجمالي",
        "مجموع",
        "حسب",
        "لكل",
        "عند كل",
        "كل مرشده",
        "كل دكتوره",
        "ترتيب",
        "اعلى",
        "اقل",
        "قارن",
        "مقارنه",
        "تحليل",
        "عوامل",
        "اسباب",
        "نموذج",
        "تنبؤ",
        "متعثر",
        "متعثرات",
        "متاخر",
        "غير مجتاز",
        "لم تجتز",
        "لم يجتاز",
        "المواد الباقيه",
        "المواد الباقية",
        "اللي خلصتها",
        "المجتازه",
        "المجتازة",
        "يحتاجون تدريب",
        "تحتاج تدريب",
        "لم تجتز التدريب",
        "فتح استثنائي",
        "استثنائي",
        "count",
        "group",
        "ranking",
        "analysis",
        "compare",
        "model",
        "prediction",
        "backlog",
        "remaining courses",
        "completed courses",
    ]

    return _contains_any(q, logic_terms)


def question_is_direct_lookup(question: str) -> bool:
    q = _normalize_ar(question)

    direct_terms = [
        "ما اسم",
        "اسم الطالبه",
        "اسم الطالب",
        "من مرشده",
        "من مرشدة",
        "مرشده الطالبه",
        "مرشدة الطالبه",
        "اعرض بيانات",
        "بيانات الطالبه",
        "ملاحظات",
        "الساعات المكتمله",
        "الساعات المنجزه",
        "الحاله الصحيه",
        "الحالة الصحية",
        "هل لديها حاله طبيه",
        "هل تحتاج تسهيلات",
        "student name",
        "advisor",
        "notes",
        "basic info",
        "profile",
    ]

    return _contains_any(q, direct_terms)


# ------------------------------------------------------------
# FILTER HELPERS
# ------------------------------------------------------------

def _build_metadata_filter(question: str) -> Optional[Dict[str, Any]]:
    filters = []

    student_id = _extract_student_id(question)
    course_key = _extract_course_key(question)

    if student_id:
        filters.append({"student_id": student_id})

    if course_key:
        filters.append({"course_key": course_key})

    if not filters:
        return None

    if len(filters) == 1:
        return filters[0]

    return {"$and": filters}


# ------------------------------------------------------------
# RETRIEVAL
# ------------------------------------------------------------

def retrieve_student_documents(
    question: str,
    collection,
    top_k: int = 10,
) -> List[Dict[str, Any]]:
    metadata_filter = _build_metadata_filter(question)

    results = collection.similarity_search_with_score(
        query=question,
        k=top_k,
        filter=metadata_filter,
    )

    retrieved = []

    for doc, distance in results:
        metadata = doc.metadata or {}
        text = doc.page_content or ""

        if not str(text).strip():
            continue

        retrieved.append({
            "text": text,
            "metadata": metadata,
            "distance": distance,
            "student_id": metadata.get("student_id"),
            "student_name": metadata.get("student_name"),
            "advisor": metadata.get("advisor"),
            "doc_type": metadata.get("doc_type"),
            "course_key": metadata.get("course_key"),
        })

    return retrieved


def _split_text_for_deep_search(
    text: str,
    chunk_size: int = 1200,
    overlap: int = 250,
) -> List[str]:
    text = str(text).strip()

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

        if start < 0:
            start = 0

        if start >= len(text):
            break

    return chunks


def deep_search_retrieved_documents(
    question: str,
    documents: List[Dict[str, Any]],
    max_documents: int = 5,
    max_chars_per_document: int = 1800,
) -> List[Dict[str, Any]]:
    q = str(question)

    important_terms = []
    for token in re.split(r"\s+|،|,|\?|؟|\.", q):
        token = token.strip()
        if len(token) >= 3:
            important_terms.append(token)

    student_id = _extract_student_id(question)
    course_key = _extract_course_key(question)

    deep_docs = []

    for doc in documents:
        text = str(doc.get("text", "")).strip()
        metadata = doc.get("metadata") or {}

        if not text:
            continue

        chunks = _split_text_for_deep_search(text)

        best_chunk = ""
        best_score = -999999

        for chunk in chunks:
            score = 0

            for term in important_terms:
                if term in chunk:
                    score += 3

            if student_id and student_id in chunk:
                score += 10

            if course_key and course_key in chunk.upper():
                score += 10

            if student_id and str(metadata.get("student_id")) == student_id:
                score += 8

            if course_key and str(metadata.get("course_key", "")).upper() == course_key:
                score += 8

            distance = doc.get("distance")
            if isinstance(distance, (int, float)):
                score += max(0, 5 - float(distance))

            if score > best_score:
                best_score = score
                best_chunk = chunk

        new_doc = dict(doc)
        new_doc["deep_score"] = best_score
        new_doc["text"] = best_chunk[:max_chars_per_document]
        deep_docs.append(new_doc)

    deep_docs.sort(
        key=lambda d: (
            d.get("deep_score", 0),
            -float(d.get("distance", 999999))
            if isinstance(d.get("distance"), (int, float))
            else -999999,
        ),
        reverse=True,
    )

    return deep_docs[:max_documents]


def build_retrieval_context(documents: List[Dict[str, Any]]) -> str:
    blocks = []

    for i, doc in enumerate(documents, start=1):
        metadata = doc.get("metadata") or {}

        header = (
            f"[Document {i}]\n"
            f"student_id: {metadata.get('student_id')}\n"
            f"student_name: {metadata.get('student_name')}\n"
            f"advisor: {metadata.get('advisor')}\n"
            f"doc_type: {metadata.get('doc_type')}\n"
            f"course_key: {metadata.get('course_key')}\n"
            f"distance: {doc.get('distance')}\n"
            f"deep_score: {doc.get('deep_score')}\n"
        )

        blocks.append(header + "\n" + str(doc.get("text", "")).strip())

    return "\n\n---\n\n".join(blocks).strip()


# ------------------------------------------------------------
# STRICT GROUNDED LLM
# ------------------------------------------------------------

RETRIEVAL_JUDGE_SYSTEM_PROMPT = """
أنت طبقة استرجاع صارمة داخل نظام إرشاد أكاديمي.

مهمتك الوحيدة:
تفحص السياق المسترجع أولًا، ثم تقرر هل يمكن الإجابة مباشرة منه أم لا.

لا ترفض السؤال لمجرد أنه يحتوي على كلمات مثل:
عدد، كم، تدريب، تعثر، مواد، مرشدة.

يجب أن تتبع هذا الترتيب بدقة:
1. افحص السياق المسترجع.
2. إذا كانت الإجابة موجودة مباشرة وصراحة في السياق، أعد status = answered.
3. إذا لم تكن الإجابة موجودة مباشرة، وكان السؤال يحتاج منطقًا أو حسابًا أو تجميعًا أو فلترة أو تحليلًا، أعد status = needs_functions.
4. إذا لم تكن الإجابة موجودة مباشرة، ولا يناسبها مسار FUNCTIONS، أعد status = no_answer.

ممنوع تمامًا:
- لا تحسب
- لا تجمع
- لا ترتب
- لا تقارن
- لا تستنتج
- لا تستخدم SQL
- لا تستخدم الدوال
- لا تستخدم معرفة خارج السياق
- لا تخمن
- لا تدمج معلومات طالبات مختلفات
- لا تبني حكمًا منطقيًا من عدة سجلات

تعريف الإجابة المباشرة:
الإجابة المباشرة هي معلومة ظاهرة بوضوح في السياق، مثل اسم طالبة، مرشدة، ملاحظة، حالة مكتوبة، قيمة حقل، أو ملخص مكتوب مسبقًا.

تعريف ما يحتاج FUNCTIONS:
إذا لم تكن الإجابة المباشرة موجودة، وكان السؤال يحتاج:
- عدد
- تجميع حسب مرشدة
- فلترة
- تعثر
- تدريب
- تحليل
- مقارنة
- ترتيب
- تقسيم مواد باقية ومكتملة
- أي منطق غير موجود مباشرة كسطر أو حقل واضح في السياق

فأعد needs_functions.

المسارات الممكنة:

1. answered
استخدمها فقط إذا كانت الإجابة موجودة نصيًا وبوضوح في السياق المسترجع.

2. needs_functions
استخدمها فقط بعد فشل الإجابة المباشرة من السياق، وكان السؤال يحتاج منطقًا أو حسابًا أو تجميعًا أو فلترة أو تحليلًا.

3. no_answer
استخدمها إذا لم توجد معلومات كافية في السياق للإجابة المباشرة، والسؤال لا يحتاج مسار FUNCTIONS.

أعد JSON فقط بهذا الشكل:

{
  "status": "answered | needs_functions | no_answer",
  "answer": "الإجابة العربية أو رسالة مناسبة",
  "confidence": 0.0,
  "reason": "سبب قصير"
}

إذا كان status = needs_functions، يجب أن تكون answer:
"هذا السؤال يحتاج مسار FUNCTIONS وليس retrieval."

إذا كان status = no_answer، يجب أن تكون answer:
"لا توجد معلومات كافية في البيانات المتاحة."

إذا كان status = answered، أجب بإيجاز وبالعربية اعتمادًا فقط على السياق.
""".strip()


def _parse_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    cleaned = text.strip()

    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start != -1 and end != -1 and end > start:
        try:
            parsed = json.loads(cleaned[start:end + 1])
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return None

    return None


def judge_retrieval_answerability(
    *,
    question: str,
    context: str,
    client,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    if not context.strip():
        if question_requires_functions(question):
            return {
                "status": RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
                "answer": RETRIEVAL_NEEDS_FUNCTIONS,
                "confidence": 0.90,
                "reason": "empty context and question requires logic",
            }

        return {
            "status": RETRIEVAL_STATUS_NO_ANSWER,
            "answer": RETRIEVAL_NO_ANSWER,
            "confidence": 0.0,
            "reason": "empty retrieval context",
        }

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},

        messages=[
            {
                "role": "system",
                "content": RETRIEVAL_JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
السؤال:
{question}

هل السؤال يبدو أنه يحتاج FUNCTIONS إذا لم توجد إجابة مباشرة؟
{"نعم" if question_requires_functions(question) else "لا"}

السياق المسترجع:
{context}
""".strip(),
            },
        ],
    )

    raw = response.choices[0].message.content.strip()
    parsed = _parse_json_object(raw)

    if not parsed:
        if question_requires_functions(question):
            return {
                "status": RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
                "answer": RETRIEVAL_NEEDS_FUNCTIONS,
                "confidence": 0.70,
                "reason": "invalid LLM JSON and question requires logic",
            }

        return {
            "status": RETRIEVAL_STATUS_NO_ANSWER,
            "answer": RETRIEVAL_NO_ANSWER,
            "confidence": 0.0,
            "reason": "LLM returned invalid JSON",
        }

    status = parsed.get("status")
    answer = str(parsed.get("answer", "")).strip()
    reason = str(parsed.get("reason", "")).strip()

    try:
        confidence = float(parsed.get("confidence", 0.0))
    except Exception:
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))

    if status not in {
        RETRIEVAL_STATUS_ANSWERED,
        RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
        RETRIEVAL_STATUS_NO_ANSWER,
    }:
        if question_requires_functions(question):
            return {
                "status": RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
                "answer": RETRIEVAL_NEEDS_FUNCTIONS,
                "confidence": 0.70,
                "reason": "invalid retrieval status and question requires logic",
            }

        return {
            "status": RETRIEVAL_STATUS_NO_ANSWER,
            "answer": RETRIEVAL_NO_ANSWER,
            "confidence": 0.0,
            "reason": "invalid retrieval status",
        }

    if status == RETRIEVAL_STATUS_NEEDS_FUNCTIONS:
        answer = RETRIEVAL_NEEDS_FUNCTIONS

    if status == RETRIEVAL_STATUS_NO_ANSWER:
        if question_requires_functions(question):
            status = RETRIEVAL_STATUS_NEEDS_FUNCTIONS
            answer = RETRIEVAL_NEEDS_FUNCTIONS
            confidence = max(confidence, 0.75)
            reason = reason or "direct answer not found and question requires logic"
        else:
            answer = RETRIEVAL_NO_ANSWER

    if status == RETRIEVAL_STATUS_ANSWERED and not answer:
        if question_requires_functions(question):
            status = RETRIEVAL_STATUS_NEEDS_FUNCTIONS
            answer = RETRIEVAL_NEEDS_FUNCTIONS
            confidence = 0.75
            reason = "empty direct answer and question requires logic"
        else:
            status = RETRIEVAL_STATUS_NO_ANSWER
            answer = RETRIEVAL_NO_ANSWER
            confidence = 0.0
            reason = "empty answer"

    return {
        "status": status,
        "answer": answer,
        "confidence": confidence,
        "reason": reason,
    }


# ------------------------------------------------------------
# CONFIDENCE GUARDS
# ------------------------------------------------------------

def estimate_vector_confidence(documents: List[Dict[str, Any]]) -> float:
    if not documents:
        return 0.0

    distances = [
        d.get("distance")
        for d in documents
        if isinstance(d.get("distance"), (int, float))
    ]

    if not distances:
        return 0.55

    best_distance = min(distances)

    if best_distance <= 0.25:
        return 0.95
    if best_distance <= 0.40:
        return 0.85
    if best_distance <= 0.60:
        return 0.65
    if best_distance <= 0.80:
        return 0.45

    return 0.25


def combine_retrieval_confidence(
    vector_confidence: float,
    llm_confidence: float,
) -> float:
    return round(min(vector_confidence, llm_confidence), 4)


def should_accept_retrieval_answer(
    *,
    status: str,
    confidence: float,
    min_confidence: float,
) -> bool:
    return status == RETRIEVAL_STATUS_ANSWERED and confidence >= min_confidence


# ------------------------------------------------------------
# PUBLIC STRICT RETRIEVAL PIPELINE
# ------------------------------------------------------------

def retrieval_first_answer(
    *,
    question: str,
    client,
    chroma_path: str = VECTOR_DB_PATH,
    collection_name: str = VECTOR_COLLECTION_NAME,
    top_k: int = 10,
    model: str = "gpt-4o-mini",
    min_confidence: float = 0.75,
) -> Dict[str, Any]:
    try:
        collection = get_chroma_collection(
            db_path=chroma_path,
            collection_name=collection_name,
        )

        documents = retrieve_student_documents(
            question=question,
            collection=collection,
            top_k=top_k,
        )

        if not documents:
            if question_requires_functions(question):
                decision = RetrievalDecision(
                    status=RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
                    answer=RETRIEVAL_NEEDS_FUNCTIONS,
                    confidence=0.90,
                    reason="no documents retrieved, and question requires logic",
                    documents_used=[],
                    retrieval={
                        "raw_documents": [],
                        "deep_documents": [],
                        "context": "",
                        "requires_functions_signal": True,
                    },
                )
                return asdict(decision)

            decision = RetrievalDecision(
                status=RETRIEVAL_STATUS_NO_ANSWER,
                answer=RETRIEVAL_NO_ANSWER,
                confidence=0.0,
                reason="no documents retrieved",
                documents_used=[],
                retrieval={
                    "raw_documents": [],
                    "deep_documents": [],
                    "context": "",
                    "requires_functions_signal": False,
                },
            )
            return asdict(decision)

        deep_documents = deep_search_retrieved_documents(
            question=question,
            documents=documents,
            max_documents=5,
        )

        context = build_retrieval_context(deep_documents)

        judged = judge_retrieval_answerability(
            question=question,
            context=context,
            client=client,
            model=model,
        )

        vector_confidence = estimate_vector_confidence(deep_documents)
        llm_confidence = float(judged.get("confidence", 0.0))
        final_confidence = combine_retrieval_confidence(
            vector_confidence=vector_confidence,
            llm_confidence=llm_confidence,
        )

        status = judged["status"]

        if status == RETRIEVAL_STATUS_ANSWERED:
            if not should_accept_retrieval_answer(
                status=status,
                confidence=final_confidence,
                min_confidence=min_confidence,
            ):
                if question_requires_functions(question):
                    status = RETRIEVAL_STATUS_NEEDS_FUNCTIONS
                    answer = RETRIEVAL_NEEDS_FUNCTIONS
                    confidence = max(final_confidence, 0.75)
                    reason = "retrieval answer below confidence threshold and question requires logic"
                else:
                    status = RETRIEVAL_STATUS_NO_ANSWER
                    answer = RETRIEVAL_NO_ANSWER
                    confidence = final_confidence
                    reason = "retrieval answer below confidence threshold"
            else:
                answer = judged["answer"]
                confidence = final_confidence
                reason = judged.get("reason", "direct answer found in retrieved context")

        elif status == RETRIEVAL_STATUS_NEEDS_FUNCTIONS:
            answer = RETRIEVAL_NEEDS_FUNCTIONS
            confidence = llm_confidence
            reason = judged.get("reason", "direct answer not found and question requires functions")

        else:
            if question_requires_functions(question):
                status = RETRIEVAL_STATUS_NEEDS_FUNCTIONS
                answer = RETRIEVAL_NEEDS_FUNCTIONS
                confidence = max(llm_confidence, 0.75)
                reason = "direct answer not found and question requires logic"
            else:
                answer = RETRIEVAL_NO_ANSWER
                confidence = llm_confidence
                reason = judged.get("reason", "insufficient retrieved context")

        decision = RetrievalDecision(
            status=status,
            answer=answer,
            confidence=confidence,
            reason=reason,
            documents_used=[
                {
                    "student_id": d.get("student_id"),
                    "student_name": d.get("student_name"),
                    "advisor": d.get("advisor"),
                    "doc_type": d.get("doc_type"),
                    "course_key": d.get("course_key"),
                    "distance": d.get("distance"),
                    "deep_score": d.get("deep_score"),
                }
                for d in deep_documents
            ],
            retrieval={
                "raw_documents": documents,
                "deep_documents": deep_documents,
                "context": context,
                "vector_confidence": vector_confidence,
                "llm_judgment": judged,
                "requires_functions_signal": question_requires_functions(question),
            },
        )

        return asdict(decision)

    except Exception as e:
        if question_requires_functions(question):
            decision = RetrievalDecision(
                status=RETRIEVAL_STATUS_NEEDS_FUNCTIONS,
                answer=RETRIEVAL_NEEDS_FUNCTIONS,
                confidence=0.70,
                reason=f"retrieval error, but question requires logic: {e}",
                documents_used=[],
                retrieval={
                    "raw_documents": [],
                    "deep_documents": [],
                    "context": "",
                    "error": str(e),
                    "requires_functions_signal": True,
                },
            )
            return asdict(decision)

        decision = RetrievalDecision(
            status=RETRIEVAL_STATUS_NO_ANSWER,
            answer=RETRIEVAL_NO_ANSWER,
            confidence=0.0,
            reason=f"retrieval error: {e}",
            documents_used=[],
            retrieval={
                "raw_documents": [],
                "deep_documents": [],
                "context": "",
                "error": str(e),
                "requires_functions_signal": False,
            },
        )
        return asdict(decision)


# ------------------------------------------------------------
# BACKWARD-COMPATIBLE WRAPPER
# ------------------------------------------------------------

def answer_with_retrieval(
    *,
    question: str,
    client,
    chroma_path: str = VECTOR_DB_PATH,
    collection_name: str = VECTOR_COLLECTION_NAME,
    top_k: int = 10,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    result = retrieval_first_answer(
        question=question,
        client=client,
        chroma_path=chroma_path,
        collection_name=collection_name,
        top_k=top_k,
        model=model,
    )

    return {
        "ok": result["status"] == RETRIEVAL_STATUS_ANSWERED,
        "source": "retrieval",
        "status": result["status"],
        "answer": result["answer"],
        "documents_used": result["documents_used"],
        "confidence": result["confidence"],
        "reason": result["reason"],
        "retrieval": result["retrieval"],
    }


# ------------------------------------------------------------
# DEBUG
# ------------------------------------------------------------

def debug_retrieval_first(
    *,
    question: str,
    client,
    chroma_path: str = VECTOR_DB_PATH,
    collection_name: str = VECTOR_COLLECTION_NAME,
    top_k: int = 10,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    return {
        "question": question,
        "requires_functions_signal": question_requires_functions(question),
        "direct_lookup_signal": question_is_direct_lookup(question),
        "student_id": _extract_student_id(question),
        "course_key": _extract_course_key(question),
        "result": retrieval_first_answer(
            question=question,
            client=client,
            chroma_path=chroma_path,
            collection_name=collection_name,
            top_k=top_k,
            model=model,
        ),
    }

In [ ]:
SYSTEM_NO_ANSWER = "لا أملك إجابة كافية من البيانات الحالية."
# cell 11

def sql_student_answering(question: str) -> dict:
    try:
        # =========================
        # STEP 1: RETRIEVAL FIRST
        # =========================
        retrieval_result = retrieval_first_answer(
            question=question,
            client=client,
        )

        status = retrieval_result.get("status")

        # =========================
        # CASE 1: ANSWERED
        # =========================
        if status == "answered":
            return {
                "ok": True,
                "question": question,
                "source": "retrieval",
                "answer": retrieval_result.get("answer"),
                "data": retrieval_result.get("data"),
                "decision": {"stage": "retrieval"},
            }

        # =========================
        # CASE 2: NEEDS FUNCTIONS
        # =========================
        if status == "needs_functions":
            routed = function_router(
                question,
                execute=True,
                client=client,
                model=OPENAI_MODEL,
                function_registry=build_function_registry(),
            )

            if getattr(routed, "ok", False):
                result = routed.result

                if isinstance(result, dict):
                    answer = result.get("answer") or result.get("message") or str(result)
                    return {
                        "ok": True,
                        "question": question,
                        "source": "functions",
                        "answer": answer,
                        "data": result.get("data", result),
                        "decision": routed.decision,
                    }

                return {
                    "ok": True,
                    "question": question,
                    "source": "functions",
                    "answer": str(result),
                    "data": result,
                    "decision": routed.decision,
                }

            return {
                "ok": False,
                "question": question,
                "source": "functions",
                "answer": SYSTEM_NO_ANSWER,
                "data": None,
                "decision": getattr(routed, "decision", None),
                "error": getattr(routed, "error", None),
            }

        # =========================
        # CASE 3: NO ANSWER
        # =========================
        return {
            "ok": False,
            "question": question,
            "source": "system",
            "answer": SYSTEM_NO_ANSWER,
            "data": None,
            "decision": {"stage": "retrieval_no_answer"},
        }

    except Exception as e:
        return {
            "ok": False,
            "question": question,
            "source": "system",
            "answer": SYSTEM_NO_ANSWER,
            "data": None,
            "decision": None,
            "error": str(e),
        }

In [ ]:
# cell 12

#
def run_system_test():
    results = []

    for q in test_questions1:
        print("\n" + "=" * 60)
        print("QUESTION:", q)

        try:
            res = sql_student_answering(q)

            print("OK:", res.get("ok"))
            print("SOURCE:", res.get("source"))
            print("ANSWER:", res.get("answer"))
            print("ERROR:", res.get("error"))

            print("DECISION:", res.get("decision"))

            if res.get("data") and isinstance(res.get("data"), dict):
                print("STATUS:", res["data"].get("status"))
                print("CONFIDENCE:", res["data"].get("confidence"))
                print("REASON:", res["data"].get("reason"))

            results.append({
                "question": q,
                "result": res,
            })

        except Exception as e:
            print("CRASH:", str(e))
            results.append({
                "question": q,
                "error": str(e),
            })

    return results

In [ ]:
test_questions = [
    # ===== NEEDS FIX / HARD CASES =====
    "كم الساعات المكتملة للطالبة 2216908؟",
    "اعرض بيانات الطالبة 2216908",
    "من الطالبات اللي ما اجتازوا COURSE_037؟",


    # ===== HUMAN / MESSY =====
    "المواد الباقية للطالب و اللي خلصها 2216908",
    "كم باقي لها مواد 2216908؟",
    "في مين ما خلص تدريب؟",
    "عطيني اللي متعثرين بشكل عام",
    "ابغى اعرف وضع الطالبة 2216908",
    "كم عند دكتورة نورة طالبات؟",
    "في احد باقي عليه تدريب ولا كلهم خلصوا؟"]

In [ ]:

test_questions1 = [
    # ===== NEEDS FIX / HARD CASES =====
    "ماهي المواد اللتي درسها الطالبة 2216908؟"
]

In [ ]:
run_system_test()



QUESTION: ماهي المواد اللتي درسها الطالبة 2216908؟


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

OK: True
SOURCE: functions
ANSWER: عدد المواد المجتازة/المعادلة: 17، وعدد المواد المتبقية: 18، وعدد مواد التعثر الفعلية: 15.
ERROR: None
DECISION: {'function_name': 'get_student_course_completion_summary', 'arguments': {'student_id': '2216908'}, 'needs_resolution': False, 'missing_arguments': [], 'confidence': 0.99, 'reason': 'السؤال يطلب تقسيم المواد المكتملة والمواد المتبقية للطالبة'}
STATUS: None
CONFIDENCE: None
REASON: None


[{'question': 'ماهي المواد اللتي درسها الطالبة 2216908؟',
  'result': {'ok': True,
   'question': 'ماهي المواد اللتي درسها الطالبة 2216908؟',
   'source': 'functions',
   'answer': 'عدد المواد المجتازة/المعادلة: 17، وعدد المواد المتبقية: 18، وعدد مواد التعثر الفعلية: 15.',
   'data': {'ok': True,
    'student_id': '2216908',
    'completed_courses': [{'student_id': '2216908',
      'course_key': 'CECN 282',
      'course_label': 'CECN 282 - تصميم المنطق الرقمي',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'CECS 111',
      'course_label': 'CECS 111 - مقدمة في برمجة',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'CECS 121',
      'course_label': 'CECS 121 - برمجة شيئية',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'CECS 122',
      'course_label': 'CECS 122 - رياضيات متقدمة حاسوبية',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'CECS 217',
      'course_label': 'CECS 217 - ت

In [ ]:
run_system_test()



QUESTION: ماهي المواد اللتي درسها الطالبة 2216908؟


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

OK: True
SOURCE: functions
ANSWER: عدد المواد المجتازة/المعادلة: 17، وعدد المواد المتبقية: 18.
ERROR: None
DECISION: {'function_name': 'get_student_course_completion_summary', 'arguments': {'student_id': '2216908'}, 'needs_resolution': False, 'missing_arguments': [], 'confidence': 0.9, 'reason': 'السؤال يتعلق بالمواد التي درستها الطالبة، والدالة المناسبة توفر ملخصًا عن إكمال المواد لطالبة محددة.'}
STATUS: None
CONFIDENCE: None
REASON: None


[{'question': 'ماهي المواد اللتي درسها الطالبة 2216908؟',
  'result': {'ok': True,
   'question': 'ماهي المواد اللتي درسها الطالبة 2216908؟',
   'source': 'functions',
   'answer': 'عدد المواد المجتازة/المعادلة: 17، وعدد المواد المتبقية: 18.',
   'data': {'ok': True,
    'student_id': '2216908',
    'completed_courses': [{'student_id': '2216908',
      'course_key': 'COURSE_002',
      'course_label': 'CECS 111 - مقدمة في برمجة',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'COURSE_003',
      'course_label': 'ELPR 130 - اللغة الإنجليزية-3',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'COURSE_004',
      'course_label': 'SCMT 102 - تفاضل و تكامل-1',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'COURSE_005',
      'course_label': 'SCPH 101 - فيزياء عامة -1',
      'status': 'P'},
     {'student_id': '2216908',
      'course_key': 'COURSE_006',
      'course_label': 'CECS 121 - برمجة شيئية',
      'statu

In [ ]:
# ============================================================
# CELL 13: CONVERSATION STATE & MEMORY ENGINE
# (STATEFUL + FOLLOW-UP AWARE + AGENTIC CONVERSATIONAL SUPPORT)
# ============================================================

from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any
import copy


# ============================================================
# DATA STRUCTURE
# ============================================================
@dataclass
class ConversationState:
    active_topic: Optional[str] = None
    last_route_path: Optional[str] = None
    last_sql_task: Optional[str] = None

    last_student_id: Optional[str] = None
    last_student_name: Optional[str] = None
    last_advisor_name: Optional[str] = None
    last_course_key: Optional[str] = None
    last_course_label: Optional[str] = None

    last_courses_list: List[Dict[str, Any]] = field(default_factory=list)
    last_students_list: List[Dict[str, Any]] = field(default_factory=list)
    last_rows: List[Dict[str, Any]] = field(default_factory=list)

    last_sql_data: Dict[str, Any] = field(default_factory=dict)
    last_retrieval_answer: Optional[str] = None
    last_answer_text: Optional[str] = None

    last_question: Optional[str] = None
    turn_count: int = 0


# ============================================================
# GLOBAL STATE HOLDER
# ============================================================
_CONVERSATION_STATE = ConversationState()


# ============================================================
# BASIC STATE API
# ============================================================
def reset_conversation_state():
    global _CONVERSATION_STATE
    _CONVERSATION_STATE = ConversationState()


def get_conversation_state() -> ConversationState:
    return copy.deepcopy(_CONVERSATION_STATE)


def get_conversation_state_dict() -> Dict[str, Any]:
    return asdict(get_conversation_state())


def set_conversation_state(state: ConversationState):
    global _CONVERSATION_STATE
    _CONVERSATION_STATE = copy.deepcopy(state)


# ============================================================
# LIGHT HELPERS
# ============================================================
def _safe_get_attr(obj: Any, name: str, default=None):
    if obj is None:
        return default
    if hasattr(obj, name):
        return getattr(obj, name, default)
    if isinstance(obj, dict):
        return obj.get(name, default)
    return default


def _extract_route_path(route_decision: Any) -> Optional[str]:
    return _safe_get_attr(route_decision, "path")


def _extract_primary_intent(route_decision: Any) -> Optional[str]:
    return _safe_get_attr(route_decision, "primary_intent")


def _extract_sql_task(sql_decision: Any) -> Optional[str]:
    return _safe_get_attr(sql_decision, "task_type")


def _extract_sql_entities(sql_decision: Any) -> Dict[str, Optional[str]]:
    return {
        "student_id": _safe_get_attr(sql_decision, "student_id"),
        "student_name": _safe_get_attr(sql_decision, "student_name"),
        "advisor_name": _safe_get_attr(sql_decision, "advisor_name"),
        "course_key": _safe_get_attr(sql_decision, "course_key"),
        "course_label": _safe_get_attr(sql_decision, "course_label"),
    }


# ============================================================
# FOLLOW-UP DETECTION
# ============================================================
FOLLOWUP_MARKERS = [
    "طيب",
    "طيب اذا",
    "طيب وهل",
    "وهل",
    "وفيه",
    "فيها",
    "فيهم",
    "منهم",
    "منها",
    "عليها",
    "عنده",
    "عندها",
    "له",
    "لها",
    "هذا",
    "هذه",
    "ذلك",
    "الاول",
    "الأول",
    "الثاني",
    "الثانية",
    "الثالث",
    "الثالثة",
    "نفسه",
    "نفسها",
]

SHORT_FOLLOWUP_PATTERNS = [
    "مين",
    "من",
    "كم",
    "وش",
    "ما",
    "هل",
]


def detect_followup_question(question: str, state: Optional[ConversationState] = None) -> bool:
    state = state or get_conversation_state()
    q = normalize_query_for_tools(question)

    if not q:
        return False

    # إذا لا توجد حالة سابقة فلا يوجد follow-up
    if state.turn_count <= 0:
        return False

    # سؤال قصير جدًا غالبًا follow-up
    if len(q.split()) <= 4:
        if any(x in q for x in FOLLOWUP_MARKERS):
            return True
        if any(q.startswith(x) for x in SHORT_FOLLOWUP_PATTERNS):
            return True

    # إشارات صريحة
    if any(x in q for x in FOLLOWUP_MARKERS):
        return True

    # إذا كان عندنا موضوع نشط وذكر المستخدم مرجعًا قصيرًا
    if state.active_topic and len(q.split()) <= 7:
        if any(x in q for x in ["المادة", "المقرر", "الطالب", "الطالبة", "المرشد", "الطلاب", "الطالبات"]):
            return True

    return False


# ============================================================
# ENTITY RESOLUTION FROM STATE
# ============================================================
def _match_course_from_state(question: str, state: ConversationState) -> Dict[str, Optional[str]]:
    q = normalize_query_for_tools(question)

    # match by exact/contains on previously returned courses
    for c in state.last_courses_list:
        label = str(c.get("course_label") or "").strip()
        key = str(c.get("course_key") or "").strip()

        label_norm = normalize_query_for_tools(label)
        key_norm = normalize_query_for_tools(key)

        if label_norm and label_norm in q:
            return {
                "course_label": label,
                "course_key": key
            }

        if key_norm and key_norm in q:
            return {
                "course_label": label,
                "course_key": key
            }

    # إذا المستخدم يقول "المادة الثانية" أو "أول مادة"
    if state.last_courses_list:
        if "اول ماده" in normalize_sql_text(q) or "الماده الاولى" in normalize_sql_text(q) or "أول مادة" in q:
            c = state.last_courses_list[0]
            return {
                "course_label": c.get("course_label"),
                "course_key": c.get("course_key"),
            }

        if ("ثاني مادة" in q or "المادة الثانية" in q or "الماده الثانيه" in normalize_sql_text(q)) and len(state.last_courses_list) >= 2:
            c = state.last_courses_list[1]
            return {
                "course_label": c.get("course_label"),
                "course_key": c.get("course_key"),
            }

    return {
        "course_label": None,
        "course_key": None,
    }


def _match_student_from_state(question: str, state: ConversationState) -> Dict[str, Optional[str]]:
    q = normalize_query_for_tools(question)

    if state.last_student_id and any(x in q for x in ["وضعه", "وضعها", "هل يحتاج", "هل تحتاج", "حلله", "حللها", "مرشده", "مرشدها"]):
        return {
            "student_id": state.last_student_id,
            "student_name": state.last_student_name,
        }

    # محاولة match على آخر قائمة طلاب
    for s in state.last_students_list:
        name = str(s.get("student_name") or "").strip()
        sid = str(s.get("student_id") or "").strip()

        if name and normalize_query_for_tools(name) in q:
            return {
                "student_id": sid or None,
                "student_name": name,
            }

    return {
        "student_id": None,
        "student_name": None,
    }


# ============================================================
# QUESTION ENRICHMENT
# ============================================================
def enrich_question_from_state(question: str, state: Optional[ConversationState] = None) -> str:
    state = state or get_conversation_state()
    q = normalize_query_for_tools(question)

    if not q:
        return q

    if not detect_followup_question(q, state):
        return q

    enriched = q

    # 1) course grounding
    course_match = _match_course_from_state(q, state)
    course_label = course_match.get("course_label")
    course_key = course_match.get("course_key")

    if course_label and course_key:
        if course_label not in enriched and course_key not in enriched:
            enriched = f"{enriched} والمقصود بالمقرر هو {course_label} ورمزه {course_key}"
        elif course_label in enriched and course_key not in enriched:
            enriched = f"{enriched} ورمز المقرر هو {course_key}"

    # 2) student grounding
    student_match = _match_student_from_state(q, state)
    student_id = student_match.get("student_id")
    student_name = student_match.get("student_name")

    if student_id and student_id not in enriched:
        if any(x in q for x in ["الطالب", "الطالبة", "وضعه", "وضعها", "مرشده", "مرشدها", "دعم", "خطورة"]):
            enriched = f"{enriched} والطالب المقصود رقمه {student_id}"

    # 3) advisor grounding
    if state.last_advisor_name and any(x in q for x in ["منهم", "عندها", "عنده", "تحت إشرافه", "تحت إشرافها", "طلابها", "طالباتها"]):
        if state.last_advisor_name not in enriched:
            enriched = f"{enriched} والمرشد المقصود هو {state.last_advisor_name}"

    # 4) topic grounding
    if state.active_topic and "المقصود" not in enriched:
        if state.active_topic == "exceptional_opening" and any(x in q for x in ["فيها", "فيه", "منها", "منهم", "اللي فيها", "اللي في"]):
            enriched = f"{enriched} والسياق الحالي هو المواد المقترح فتحها استثنائيًا"

    return normalize_query_for_tools(enriched)


# ============================================================
# STATE UPDATING
# ============================================================
def _infer_active_topic(route_decision: Any, sql_result: Optional[Dict[str, Any]] = None) -> Optional[str]:
    primary_intent = _extract_primary_intent(route_decision)
    sql_task = (sql_result or {}).get("task_type") if isinstance(sql_result, dict) else None

    if sql_task == "courses_to_open_for_expected_graduates":
        return "exceptional_opening"
    if sql_task == "students_by_advisor":
        return "advisor_students"
    if sql_task == "student_full_context":
        return "student_analysis"
    if primary_intent:
        return primary_intent

    return None


def update_conversation_state(
    question: str,
    route_decision: Any = None,
    sql_decision: Any = None,
    sql_result: Optional[Dict[str, Any]] = None,
    retrieval_answer: Optional[str] = None,
    final_answer: str = "",
):
    global _CONVERSATION_STATE

    state = get_conversation_state()

    state.turn_count += 1
    state.last_question = question
    state.last_answer_text = final_answer
    state.last_retrieval_answer = retrieval_answer

    state.last_route_path = _extract_route_path(route_decision)
    state.active_topic = _infer_active_topic(route_decision, sql_result=sql_result)

    if sql_decision is not None:
        state.last_sql_task = _extract_sql_task(sql_decision)

        entities = _extract_sql_entities(sql_decision)

        if entities.get("student_id"):
            state.last_student_id = entities["student_id"]
        if entities.get("student_name"):
            state.last_student_name = entities["student_name"]
        if entities.get("advisor_name"):
            state.last_advisor_name = entities["advisor_name"]
        if entities.get("course_key"):
            state.last_course_key = entities["course_key"]
        if entities.get("course_label"):
            state.last_course_label = entities["course_label"]

    if sql_result and isinstance(sql_result, dict):
        state.last_sql_task = sql_result.get("task_type") or state.last_sql_task

        data = sql_result.get("data") or {}
        state.last_sql_data = data if isinstance(data, dict) else {}

        if isinstance(data, dict):
            if "courses" in data and isinstance(data["courses"], list):
                state.last_courses_list = data["courses"]

                if data["courses"]:
                    first_course = data["courses"][0]
                    if first_course.get("course_key"):
                        state.last_course_key = first_course.get("course_key")
                    if first_course.get("course_label"):
                        state.last_course_label = first_course.get("course_label")

            if "students" in data and isinstance(data["students"], list):
                state.last_students_list = data["students"]

            if "rows" in data and isinstance(data["rows"], list):
                state.last_rows = data["rows"]

            if data.get("student_id"):
                state.last_student_id = data.get("student_id")

            if data.get("advisor"):
                state.last_advisor_name = data.get("advisor")

            if data.get("course_key"):
                state.last_course_key = data.get("course_key")

            if data.get("course_label"):
                state.last_course_label = data.get("course_label")

    set_conversation_state(state)


print("✅ Cell 13 conversation state & memory engine loaded successfully.")

✅ Cell 13 conversation state & memory engine loaded successfully.


In [ ]:
# ============================================================
# CELL 14: GENERALIZED MEMORY LAYER (CLEAN VERSION)
# ============================================================

from typing import Any

try:
    from langchain_core.chat_history import InMemoryChatMessageHistory
except ImportError:
    try:
        from langchain_community.chat_message_histories import ChatMessageHistory as InMemoryChatMessageHistory
    except ImportError:
        from langchain_community.chat_message_histories.in_memory import ChatMessageHistory as InMemoryChatMessageHistory
        
from langchain_core.messages import HumanMessage, AIMessage


# ------------------------------------------------------------
# MEMORY OBJECT
# ------------------------------------------------------------
LC_CHAT_HISTORY = InMemoryChatMessageHistory()
LC_WINDOW_TURNS = 8


# ------------------------------------------------------------
# RESET HELPERS
# ------------------------------------------------------------
def reset_langchain_chat_memory():
    global LC_CHAT_HISTORY
    LC_CHAT_HISTORY = InMemoryChatMessageHistory()


def reset_all_chat_memory():
    reset_langchain_chat_memory()


# ------------------------------------------------------------
# BASIC HELPERS
# ------------------------------------------------------------
def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()


def _serialize_recent_history(max_turns: int = 8) -> str:
    msgs = LC_CHAT_HISTORY.messages or []
    msgs = msgs[-(max_turns * 2):]

    lines = []
    for m in msgs:
        content = _safe_str(getattr(m, "content", ""))
        if not content:
            continue

        if isinstance(m, HumanMessage):
            lines.append(f"المستخدم: {content}")
        elif isinstance(m, AIMessage):
            lines.append(f"المساعد: {content}")

    return "\n".join(lines).strip()


def _add_turn_to_memory(question: str, answer_text: str):
    """
    يحفظ آخر turn بدون تكرار متتالٍ
    """
    msgs = LC_CHAT_HISTORY.messages or []

    if not (
        msgs
        and isinstance(msgs[-1], HumanMessage)
        and _safe_str(getattr(msgs[-1], "content", "")) == _safe_str(question)
    ):
        LC_CHAT_HISTORY.add_message(HumanMessage(content=_safe_str(question)))

    msgs = LC_CHAT_HISTORY.messages or []

    if not (
        msgs
        and isinstance(msgs[-1], AIMessage)
        and _safe_str(getattr(msgs[-1], "content", "")) == _safe_str(answer_text)
    ):
        LC_CHAT_HISTORY.add_message(AIMessage(content=_safe_str(answer_text)))


# ------------------------------------------------------------
# CHAT HISTORY ACCESS
# ------------------------------------------------------------
def build_chat_history_from_session() -> str:
    """
    ترجع آخر جزء من المحادثة بصيغة نصية.
    تستخدمها طبقات rewriting أو orchestration عند الحاجة.
    """
    return _serialize_recent_history(max_turns=LC_WINDOW_TURNS)


# ------------------------------------------------------------
# GENERALIZED CONTEXTUAL REWRITE
# ------------------------------------------------------------
def _rewrite_with_general_memory(question: str) -> str:
    q = _safe_str(question)
    history = build_chat_history_from_session()

    if not history:
        return q

    prompt = f"""
حوّل السؤال الحالي إلى سؤال مستقل وواضح وصالح مباشرةً للتوجيه أو الاسترجاع.

المحادثة السابقة:
{history}

السؤال الحالي:
{q}

التعليمات:
- إذا كان السؤال الحالي يعتمد على السياق، فأعد كتابته كسؤال مستقل مكتمل.
- اجعل الناتج مناسبًا للتوجيه أو البحث الدقيق.
- لا تترك كلمات مبهمة مثل: هذه، هذا، النظرية، وماذا عنها.
- إذا كان السؤال مكتملًا أصلًا، أعده كما هو.
- لا تجب على السؤال.
- لا تشرح.
- أخرج فقط السؤال النهائي.

أمثلة:
السابق: كم المكافأة للكليات العلمية؟
الحالي: والكليات النظرية؟
الناتج: كم المكافأة الشهرية لطلبة الكليات النظرية في جامعة جدة؟

السابق: ما شروط التحويل الداخلي؟
الحالي: ومتى التقديم؟
الناتج: متى التقديم على التحويل الداخلي في جامعة جدة؟
""".strip()

    try:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You rewrite follow-up questions into explicit standalone questions for routing and retrieval."
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=100,
        )

        rewritten = (resp.choices[0].message.content or "").strip()
        return rewritten if rewritten else q

    except Exception as e:
        print(f"⚠️ Memory rewrite warning: {e}")
        return q

def should_apply_memory_rewrite(question: str) -> bool:
    q = (question or "").strip()
    q_norm = normalize_query_for_tools(q)

    if not q_norm:
        return False

    followup_markers = [
        "له", "لها", "عنه", "عنها", "فيه", "فيها",
        "ومتى", "واين", "وأين", "وهل", "وهذا", "وهذه",
        "هذا", "هذه", "ذلك", "تلك", "it", "this", "that",
        "نفس الشي", "نفس الشيء", "طيب", "طيب و", "كمان", "أيضًا"
    ]

    # أسئلة قصيرة أو مرجعية فقط
    if len(q_norm.split()) <= 6:
        return True if any(x in q_norm for x in followup_markers) else False

    # أسئلة أطول: لا نعيد كتابتها بالذاكرة إلا لو فيها مرجع واضح
    return any(x in q_norm for x in followup_markers)

# ------------------------------------------------------------
# PUBLIC MEMORY API
# ------------------------------------------------------------
def rewrite_query_with_memory(question: str) -> str:
    if not should_apply_memory_rewrite(question):
        return question
    return _rewrite_with_general_memory(question)


def save_turn_to_memory(question: str, answer_text: str):
    """
    استخدم هذه الدالة داخل Cell 6 بعد تكوين الإجابة النهائية.
    """
    _add_turn_to_memory(question, answer_text)


# ------------------------------------------------------------
# DEBUG HELPERS
# ------------------------------------------------------------
def show_langchain_memory():
    print("===== RECENT CHAT MEMORY =====")
    txt = _serialize_recent_history(max_turns=LC_WINDOW_TURNS)
    print(txt if txt else "(empty)")


print("✅ Generalized memory layer loaded.")
print("✅ No monkey patching applied.")
print("✅ Use rewrite_query_with_memory(...) inside Cell 6 before routing.")
print("✅ Use save_turn_to_memory(...) after final answer.")
print("✅ Use reset_all_chat_memory() to start a fresh session.")

✅ Generalized memory layer loaded.
✅ No monkey patching applied.
✅ Use rewrite_query_with_memory(...) inside Cell 6 before routing.
✅ Use save_turn_to_memory(...) after final answer.
✅ Use reset_all_chat_memory() to start a fresh session.
